# MUSANGA V3.1 · NB06 — PlanetScope & embedding spatial information

**Question:** Does fine-scale spatial context add meaningful and transferable information for estimating *Musanga* fraction beyond the established Sentinel-2 and satellite-embedding baseline?

**Stage 1 — exploration, not model fitting:** establish native resolution, available 4-band Planet annual/monthly imagery, annual 9-band NIR texture, 64-band 10 m embeddings and the existing 100 m UAV target cells. Display matched Planet and embedding patches, inspect 30/50/100 m supervision availability and spatial alignment. Compare native PlanetScope canopy texture with spatial variability in embeddings without confusing apparent training fit with generalization.

**Stage 2 — planned ablation, only after patch QA:** model comparison under identical targets and folds:
- **B0:** S2 annual + S2 quarterly + annual satellite embeddings (NB04B baseline).
- **B1:** B0 + handcrafted Planet texture (existing NB05 features).
- **B2:** B0 + multiscale **spatial texture of embeddings** (not merely their 64 central-pixel values).
- **B3:** B0 + Planet spatial CNN features.
- **B4:** B0 + embedding spatial CNN features.
- **B5:** B0 + both Planet and embedding CNN features.

First pilot at 50 m, then 30/100 m. **Same continuous UAV target**, same spatially held-out folds for within-site testing and leave-one-forest-out for geographic transfer. Hold out complete spatial areas/sites *before* patch extraction or learned representation training. Run supervised CNN embedding extraction out-of-fold, and fit all scalers/PCA/normalization from training folds only.

**Constraints:** Annual Planet image is 4.77 m and **may not be cloud-free**. Satellite embeddings are 10 m and already summarize S1/S2 information. Yoko Planet 2021 vs UAV 2020 is a known mismatch. Existing NB04B 100 m samples are used for preview only; no CNN trained in this first stage. No full-ROI 4.77 m imagery is assumed downloaded.


## 00 · Settings — exploratory preview only


In [ ]:
from pathlib import Path
FINAL=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                          Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
if FINAL is None:
    from google.colab import drive
    drive.mount('/content/drive')
    FINAL=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                              Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert FINAL and FINAL.is_dir(),'Musanga Final workspace missing'
ROOT=FINAL/'Outputs/V3_1'
NB05=ROOT/'05_sat_predictor_exploration'
NB04=ROOT/'04_satellite_predictors/input_layers'
NB04B=ROOT/'04B_roi_quarterly_LOSO'
OUT=ROOT/'06_spatial_planet_embeddings'
FIG=OUT/'figures'
for d in (OUT,FIG):d.mkdir(parents=True,exist_ok=True)
SITES=['Baego','MOS','Yangambi','Yoko']
PLANET_YEAR={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2021}
EMBEDDING_YEAR={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
PREVIEW_PER_SITE=3
PREVIEW_PATCH_PIXELS=48      # ~229m Planet context for each selected 100m cell
EMB_PATCH_PIXELS=23          # ~230m at 10m embedding resolution
RANDOM_SEED=20261024
print('Workspace:',FINAL)


## 01 · Validate source products and review temporal/resolution limitations

Checks actual files, expected 19-band annual Planet, 9-band NIR texture and the existing 64-band annual embeddings. Annual Planet composition bands are:
1–4 screened mean B/G/R/N; 5–8 screened median; 9–12 raw mean; **13–16 raw median**; 17–19 raw valid count / screened valid count / number screened out.
The screened product is **heuristic only**; use the raw median as the stable first CNN baseline.

NB04B's original 100 m sample cache is *not recreated* or overwritten.


In [ ]:
import numpy as np,pandas as pd,rasterio,matplotlib.pyplot as plt
from rasterio.windows import Window
from rasterio.warp import transform as rio_transform
from rasterio.enums import Resampling
from pyproj import Transformer
from IPython.display import display

def planet_file(site,kind):
    root=NB05/('derived_features' if kind=='Texture' else 'input_layers')
    folder=f'Musanga_V31_NB05_UAV_ROI_{kind}'
    tag=f'NB05_UAV_ROI_{site}_{PLANET_YEAR[site]}_'
    pattern=tag+('Annual_19band*.tif' if kind=='Annual' else 'Texture_9band*.tif')
    matches=sorted((root/folder).glob(pattern))
    if len(matches)!=1:
        raise FileNotFoundError(f'{site} {kind}: expected one file, found {len(matches)}: {matches}')
    return matches[0]

def embed_file(site):
    return NB04/'EMB_ANNUAL'/f'{site}_EMB_ANNUAL_{EMBEDDING_YEAR[site]}_10m.tif'

AUDIT=[]
for site in SITES:
    for family,filepath,expected in (
        ('PLANET_ANNUAL',planet_file(site,'Annual'),19),
        ('PLANET_TEXTURE',planet_file(site,'Texture'),9),
        ('EMB_ANNUAL',embed_file(site),64)):
        assert filepath.is_file(),filepath
        with rasterio.open(filepath) as src:
            assert src.count==expected,(site,family,'expected',expected,'found',src.count)
            assert src.crs and src.width>0 and src.height>0
            AUDIT.append(dict(site=site,family=family,year=PLANET_YEAR[site] if family!='EMB_ANNUAL' else EMBEDDING_YEAR[site],
                bands=src.count,pixel_size_m=round(abs(src.res[0]),3),
                width=src.width,height=src.height,crs=str(src.crs),
                size_MB=round(filepath.stat().st_size/1e6,2),path=str(filepath)))
AUDIT=pd.DataFrame(AUDIT)
display(AUDIT.drop(columns=['path']))
AUDIT.to_csv(OUT/'01_input_audit.csv',index=False)
assert (NB05/'07_final_input_audit/07_site_summary.csv').exists(),'Complete NB05 Section 07 audit first'
print('All 12 native products found: four forests × annual Planet/texture/embeddings.')
print('Note: The 10m embeddings are not 4.77m imagery; do not upscale them to claim new spatial detail.')


## 02 · Select existing 100 m UAV reference cells (identical NB04B targets)

Only select example cells; avoid random point selection from the full Planet image, which would include areas without UAV supervision. Sampling is stratified by low/medium/high Musanga abundance **within each site**, so each forest's previews show more than zero-cover background. Cell centers are computed from NB04B's exact S2-grid transform, not from row/col as if they were Planet raster indices.


In [ ]:
from affine import Affine
SAMPLE_PATH=NB04B/'samples/roi_cells_100m.parquet'
assert SAMPLE_PATH.is_file(),SAMPLE_PATH
DF=pd.read_parquet(SAMPLE_PATH)
assert {'site','row','col','mean_probability'}.issubset(DF.columns)
DF=DF.loc[DF.site.isin(SITES)].copy()
assert len(DF)>0
RNG=np.random.default_rng(RANDOM_SEED)
CENTERS={}
SELECTED=[]
for site in SITES:
    local=DF.loc[DF.site==site].copy()
    assert not local.empty,site
    # Existing 100m NB04B samples use the 10m S2 affine aggregated exactly 10×.
    raw=NB04/'RAW_ANNUAL'/f'{site}_RAW_ANNUAL_{EMBEDDING_YEAR[site]}_10m.tif'
    assert raw.is_file(),raw
    with rasterio.open(raw) as ds:
        assert abs(ds.res[0]-10)<.05 and abs(ds.res[1]-10)<.05
        tfm=ds.transform*Affine.scale(10,10)
        crs=ds.crs
    for i in range(PREVIEW_PER_SITE):
        ordered=local.sort_values('mean_probability')
        idx=int(round((i+.5)*(len(ordered)-1)/PREVIEW_PER_SITE))
        sample=ordered.iloc[idx]
        cx,cy=tfm*(float(sample.col)+.5,float(sample.row)+.5)
        SELECTED.append(dict(site=site,group=str(sample.group),mosaic=str(sample.mosaic),
            fraction=float(sample.mean_probability),row=int(sample.row),col=int(sample.col),
            center_x=float(cx),center_y=float(cy),crs=str(crs)))
    CENTERS[site]=dict(transform=tfm,crs=crs)
PREVIEW=pd.DataFrame(SELECTED)
display(PREVIEW[['site','group','fraction','row','col']].round(3))
PREVIEW.to_csv(OUT/'02_preview_cells.csv',index=False)
print('Preview targets are UAV-model estimates at 100 m, NOT independent field labels.')


## 03 · Matched *native* Planet RGB/NIR and embedding-neighborhood previews

For the same UAV-target cell center, read a ~229 m **Planet raw annual median RGB patch** and a ~230 m **10 m satellite-embedding patch**. Embeddings contain 64 latent bands; visualize the first three PCA components as an exploratory false-colour composite (PCA fit only on the displayed patch, **not** for any prediction model). Preserve image pixel scales. This step helps spot spatial shifts, boundary masking, and structural patterns.

The common center is transformed to each raster's CRS if necessary. If a patch crosses a source footprint boundary it is padded with zeros and flagged by nonzero-band masks; no fabricated reflectance values.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
def native_patch(path,cx,cy,center_crs,size,bands):
    with rasterio.open(path) as ds:
        trans=Transformer.from_crs(center_crs,ds.crs,always_xy=True)
        x,y=trans.transform(cx,cy)
        cc,rr=~ds.transform*(x,y)
        start_col=int(np.floor(cc-size/2))
        start_row=int(np.floor(rr-size/2))
        w=Window(start_col,start_row,size,size)
        arr=ds.read(bands,window=w,boundless=True,fill_value=0).astype('float32')
        return arr,ds.crs,ds.res
def rgb_fixed(arr,valid):
    # Input arr is RGB band order; calculate one robust stretch per local patch.
    if np.count_nonzero(valid)<10:return np.ones((*valid.shape,3),dtype='float32')
    p=arr[:,valid]
    lo=np.percentile(p,2,axis=1)
    hi=np.percentile(p,98,axis=1)
    rgb=np.moveaxis(np.clip((arr-lo[:,None,None])/(hi[:,None,None]-lo[:,None,None]+1e-6),0,1),0,-1)
    rgb[~valid]=1
    return rgb
def embed_pca_rgb(cube):
    mask=np.isfinite(cube).all(axis=0)&np.any(cube!=0,axis=0)
    out=np.ones((*mask.shape,3),dtype='float32')
    if mask.sum()<10:return out,mask
    X=cube[:,mask].T
    X=StandardScaler().fit_transform(X)
    vals=PCA(n_components=3).fit_transform(X)
    pct_lo=np.percentile(vals,2,axis=0);pct_hi=np.percentile(vals,98,axis=0)
    out[mask]=np.clip((vals-pct_lo)/(pct_hi-pct_lo+1e-6),0,1)
    return out,mask

fig,axes=plt.subplots(len(PREVIEW),2,figsize=(8,3.0*len(PREVIEW)),layout='constrained')
rows=[]
for i,row in enumerate(PREVIEW.itertuples(index=False)):
    pa=planet_file(row.site,'Annual')
    eb=embed_file(row.site)
    # Planet annual 13-16 = B,G,R,N raw median; select RGB order 15,14,13
    p,_,pres=native_patch(pa,row.center_x,row.center_y,row.crs,PREVIEW_PATCH_PIXELS,[15,14,13])
    e,_,eres=native_patch(eb,row.center_x,row.center_y,row.crs,EMB_PATCH_PIXELS,list(range(1,65)))
    pmask=np.all(np.isfinite(p)&(p>0),axis=0)
    eviz,emask=embed_pca_rgb(e)
    axes[i,0].imshow(rgb_fixed(p,pmask));axes[i,1].imshow(eviz)
    axes[i,0].set_title(f'{row.site} | UAV {row.fraction*100:.1f}% | Planet RGB')
    axes[i,1].set_title('10 m embedding neighborhood (local PCA)')
    for ax in axes[i]:ax.axis('off')
    rows.append(dict(site=row.site,fraction=row.fraction,
                     planet_valid_pct=100*pmask.mean(),embedding_valid_pct=100*emask.mean(),
                     planet_pixel_size_m=abs(pres[0]),embedding_pixel_size_m=abs(eres[0])))
plt.savefig(FIG/'03_planet_vs_embedding_native_patches.png',dpi=140,bbox_inches='tight')
plt.show()
PATCH_QA=pd.DataFrame(rows)
display(PATCH_QA.round(2))
PATCH_QA.to_csv(OUT/'03_patch_coverage.csv',index=False)
print('IMPORTANT: local PCA colours are arbitrary across panels/sites and cannot be compared spectrally.')


## 04 · Spatial embeddings vs Planet: first quantitative exploratory features

To assess whether embeddings also contain spatial texture, summarize **within-patch heterogeneity** of their 64 channels. Compare each 100 m target cell's:
- Planet NIR local texture (from NB05's existing annual raster, sampled as local median);
- embedding spatial heterogeneity (median channelwise SD over a ~230 m context), plus embedding central-context mean differences.

**Descriptive only:** 12 deliberately stratified preview cells are too few for useful correlations or model fitting. This diagnostic checks that features are finite and nontrivial. Later we will extract these features across *all* labelled cells, use consistent physical windows (not just fixed pixel counts), and evaluate added value on geographic holdouts.


In [ ]:
from scipy.ndimage import median_filter
QA=[]
for row in PREVIEW.itertuples(index=False):
    e,_,_=native_patch(embed_file(row.site),row.center_x,row.center_y,row.crs,EMB_PATCH_PIXELS,list(range(1,65)))
    m=np.isfinite(e).all(axis=0)&np.any(e!=0,axis=0)
    if m.sum()<25:continue
    em=e[:,m]
    vari=np.std(em,axis=1)
    center=e[:,EMB_PATCH_PIXELS//2,EMB_PATCH_PIXELS//2]
    context=np.mean(em,axis=1)
    t,_,_=native_patch(planet_file(row.site,'Texture'),row.center_x,row.center_y,row.crs,
                       PREVIEW_PATCH_PIXELS,[1,2,3,4])
    tm=np.isfinite(t).all(axis=0)&(t>=0).all(axis=0)&np.any(t>0,axis=0)
    QA.append(dict(site=row.site,uav_fraction=row.fraction,
        embedding_channel_sd_median=float(np.median(vari)),
        embedding_context_difference=float(np.mean(abs(center-context))),
        planet_NIR_CV_r3_median=float(np.median(t[1,tm])) if tm.any() else np.nan,
        valid_embedding_pct=100*m.mean(),valid_planet_texture_pct=100*tm.mean()))
Q=pd.DataFrame(QA)
display(Q.round(4))
Q.to_csv(OUT/'04_spatial_feature_feasibility.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(10,4),layout='constrained')
for site,group in Q.groupby('site'):
    axes[0].scatter(group.uav_fraction*100,group.embedding_channel_sd_median,label=site)
    axes[1].scatter(group.uav_fraction*100,group.planet_NIR_CV_r3_median,label=site)
axes[0].set(xlabel='UAV Musanga fraction (%)',ylabel='Median channel SD in embedding patch',
            title='Embeddings: spatial heterogeneity')
axes[1].set(xlabel='UAV Musanga fraction (%)',ylabel='Planet NIR CV (radius 3)',
            title='Planet: native-scale texture')
axes[1].legend()
plt.savefig(FIG/'04_spatial_heterogeneity_feasibility.png',dpi=150,bbox_inches='tight')
plt.show()
print('This plot has only 12 stratified examples. Do NOT interpret it as prediction skill.')


## 05 · Decision gate and forthcoming modelling experiment

Before CNN model fitting, confirm:
1. Patch-validity percentages and visual alignment of Planet with its target cells; independent UAV orthomosaic/probability overlay will follow when locating the original teacher rasters.
2. Check that Planet **2021** in Yoko is an intentional temporal mismatch against UAV **2020**.
3. Use 50 m targets first and compare 30/100 m; target extraction must use the original **full-resolution UAV probability rasters**, avoiding interpolation of 100 m labels.
4. Train compact models with group/spatial folds: **100 m/50 m patches overlap heavily**, so randomly split cells is not an acceptable test.
5. Compare `B0 → B1/B2 → B3/B4 → B5` using the same site holdout and identical target cells; summarize MAE (pp), RMSE (pp), R², high-cover bias, calibration, and performance separately at MOS. Aggregate CNN output to target cells fairly.

**NB06 currently implements the exploratory entry point (Sections 00–04) and experimental specification; CNN training follows after visual QA.** Avoid building a large training architecture before verifying supervision and patch geometry.


## 06 · Verify original UAV probability rasters and Planet co-registration

The authoritative teacher rasters are the same **M2 warmstart full-mosaic U-Net probability GeoTIFFs** used by NB04 (not the 100 m aggregate). We inspect their CRS, footprint, NoData encoding, dynamic range, and cross-check the reconstructed 100 m values against NB04B sampled targets for each mosaic. The code automatically tests several plausible uint16 denominators but **refuses to accept a scale** unless the agreement is good. Do not retrain anything if this fails.

Generate site/mosaic **side-by-side Planet RGB, original UAV orthomosaic RGB, and continuous UAV probability** panels with one common geographic extent at native pixels. The images may have different sensor dates and radiometry. The purpose is to check obvious geometric shifts, not to infer identity from colours.

*Important*: UAV 2020 and Planet 2021 for Yoko; one-year mismatch is logged. A visually plausible image is not proof of subpixel registration; if the overlay looks shifted, fix georeferencing upstream before extracting labels.


In [ ]:
# 06 — Authoritative UAV teacher paths and raster metadata
import numpy as np,pandas as pd,rasterio,matplotlib.pyplot as plt, json,math
from rasterio.warp import reproject,Resampling,transform_bounds
from rasterio.windows import from_bounds,Window
from rasterio.features import rasterize
from affine import Affine
from shapely.geometry import box
from IPython.display import display

UAV_TEACHERS=FINAL/'Outputs/V3_1/03_M2_warmstart_c6a33504c37efb2ebb84_full_mosaic/probability_maps'
UAV_MOSAICS={
 'Baego':['Baego.tif'],
 'MOS':['Mos_1_2_transparent_mosaic_group1_georef.tif','Mos_5_6_transparent_mosaic_group1_georef.tif'],
 'Yangambi':['Yangambi_georef.tif'],
 'Yoko':['Yoko_kvo_georef.tif']
}
MANIFEST_FILE=FINAL/'Outputs/V3_1/00_input_audit/00_input_manifest.json'
assert MANIFEST_FILE.exists(),MANIFEST_FILE
uav_manifest=json.loads(MANIFEST_FILE.read_text())
MOSAIC_PATH={x['file']:Path(x['path']) for x in uav_manifest['mosaics']}
for site,names in UAV_MOSAICS.items():
    for name in names:
        assert name in MOSAIC_PATH and MOSAIC_PATH[name].is_file(), ('Missing orthomosaic',site,name)
def teacher_file(name):
    return UAV_TEACHERS/f'{Path(name).stem}_M2_warmstart_epoch02_probability_u16_FULL.tif'
SOURCE_TABLE=[]
for site,names in UAV_MOSAICS.items():
    for name in names:
        path=teacher_file(name)
        assert path.is_file(),f'Authoritative UAV model output missing: {path}'
        with rasterio.open(path) as ds:
            assert ds.count>=1 and ds.crs and ds.width>0 and ds.height>0
            # Distributed window samples for a rapid encoding diagnostic.
            samples=[]
            for frac_y,frac_x in ((.25,.25),(.5,.5),(.75,.75),(.3,.7),(.7,.3)):
                win=Window(max(0,int(ds.width*frac_x)-64),max(0,int(ds.height*frac_y)-64),
                           min(128,ds.width),min(128,ds.height))
                samples.append(ds.read(1,window=win))
            z=np.concatenate([a.reshape(-1) for a in samples])
            SOURCE_TABLE.append(dict(site=site,mosaic=name,crs=str(ds.crs),
                width=ds.width,height=ds.height,res_x=ds.res[0],res_y=ds.res[1],
                dtype=ds.dtypes[0],nodata=ds.nodata,
                sample_min=float(np.min(z)),sample_p99=float(np.percentile(z,99)),
                sample_max=float(np.max(z)),file=str(path)))
display(pd.DataFrame(SOURCE_TABLE).drop(columns='file'))
pd.DataFrame(SOURCE_TABLE).to_csv(OUT/'06_uav_teacher_audit.csv',index=False)
print('Five original M2 warmstart probability maps and orthomosaics located. No labels modified.')


## 07 · RAM-safe validation against NB04B

**The previous two implementations could trigger large memory use when resampling multi-gigapixel UAV probability rasters, even with GDAL VRT.** We now avoid original full-resolution rasters entirely. NB04 already produced **10, 30, 50, 75, 100 and 125 m cached UAV aggregates** on exactly the same satellite lattice. The cache stores continuous **mean Musanga probability (band 1)**, thresholded cover (band 2), and valid UAV coverage (band 3), all normalized to [0,1].

This section reads only the cached **100 m** files and compares them with known NB04B sample fractions; the existing **50 m** cache will be used downstream. No uint16 scale guessing or expensive native-raster reprojection. Rerun Sections 00–02, 06 and 07 after reconnecting to the new notebook.


In [ ]:
# 07 — FAST: reuse verified NB04 cached 10/50/100 m UAV aggregates.
# NO source UAV raster opened. NO WarpedVRT. NO full-resolution resampling.
from pathlib import Path
import rasterio,numpy as np,pandas as pd,math
from affine import Affine
from IPython.display import display

AGG_DIR=FINAL/'Outputs/V3_1/04_satellite_full_warmstart/cache/aggregates'
assert AGG_DIR.is_dir(),AGG_DIR
REFERENCE_DIAGNOSTICS=[]
UAV_SCALE={name:1.0 for names in UAV_MOSAICS.values() for name in names}

def satellite_grid(site,scale):
    path=NB04/'RAW_ANNUAL'/f'{site}_RAW_ANNUAL_{EMBEDDING_YEAR[site]}_10m.tif'
    with rasterio.open(path) as ds:
        assert ds.crs and abs(ds.res[0]-10)<.05
        return dict(crs=ds.crs,transform=ds.transform*Affine.scale(scale/10,scale/10),
                    width=int(math.ceil(ds.width*10/scale)),
                    height=int(math.ceil(ds.height*10/scale)))

def agg_file(name,scale):
    return AGG_DIR/f'{Path(name).stem}_{scale}m.tif'

def read_cached_teacher(name,site,scale):
    """Existing NB04 cache: band 1 mean probability, 2 cover>0.5, 3 UAV coverage.
    All three are ALREADY in [0,1], aligned to the canonical satellite grid.
    """
    path=agg_file(name,scale)
    assert path.is_file(),f'Cached {scale}m UAV aggregate missing: {path}'
    g=satellite_grid(site,scale)
    with rasterio.open(path) as src:
        assert src.count>=3,(name,scale,'unexpected bands',src.count)
        assert src.crs==g['crs'],(name,scale,'CRS mismatch')
        assert src.width==g['width'] and src.height==g['height'],(name,scale,'grid dimensions mismatch')
        assert src.transform.almost_equals(g['transform'],precision=1e-6),(name,scale,'grid alignment mismatch')
        a=src.read([1,3],out_dtype='float32')
    values=a[0];cov=a[1]
    return values,cov,g

for site,names in UAV_MOSAICS.items():
    for name in names:
        ref=DF.loc[(DF.site==site)&(DF.mosaic==name)]
        assert len(ref)>0,(site,name,'missing NB04B examples')
        values,cov,g=read_cached_teacher(name,site,100)
        rr=ref.row.to_numpy(dtype=int)
        cc=ref.col.to_numpy(dtype=int)
        valid_grid=(rr>=0)&(cc>=0)&(rr<g['height'])&(cc<g['width'])
        rr,cc=rr[valid_grid],cc[valid_grid]
        known=ref.mean_probability.to_numpy(dtype=float)[valid_grid]
        pred=values[rr,cc];coverage=cov[rr,cc]
        valid=(coverage>=0.90)&np.isfinite(pred)&np.isfinite(known)
        assert valid.sum()>=8,(name,'too few valid cells',int(valid.sum()))
        mae=float(np.mean(np.abs(pred[valid]-known[valid])))
        max_diff=float(np.max(np.abs(pred[valid]-known[valid])))
        REFERENCE_DIAGNOSTICS.append(dict(site=site,mosaic=name,n=int(valid.sum()),
            source='NB04_cached_100m',MAE_fraction=mae,max_abs_difference=max_diff,
            median_reference=float(np.median(known[valid])),
            median_cache=float(np.median(pred[valid]))))
        print('VERIFIED',site,Path(name).stem,'n',int(valid.sum()),'MAE',round(mae,7),
              'maximum',round(max_diff,7),flush=True)
        assert mae<0.001,(name,'NB04B cache mismatch',mae)
        del values,cov
CALIBRATION=pd.DataFrame(REFERENCE_DIAGNOSTICS)
display(CALIBRATION.round(7))
CALIBRATION.to_csv(OUT/'07_teacher_encoding_validation.csv',index=False)
print('Safe: all comparisons use small cached 100m rasters. No native UAV probability raster was read.')
print('Section 09 will use existing cached 50m probabilities; do not resample large UAV source files.')


## 08 · Low-memory visual alignment and teacher-distribution QA

The original teacher panels appeared **nearly uniformly pale yellow**, which is the `magma` colour for probability approximately 1. The previous preview did not use the aggregate raster's valid-UAV coverage band and could therefore display invalid/background values as real predictions.

This revised window-only preview reads the **10 m cached mean probability (band 1)** **and UAV coverage (band 3)** together. It masks coverage below 90%, shades invalid pixels gray, and prints within-window p05/p50/p95 and percentages near 0 or 1. It also prints the selected 100 m reference fraction for context. **Do not interpret or train on 10 m probability values if the diagnostics still show saturation at 1.** This remains display-only QA, not a full source-raster calibration.


In [ ]:
# 08 — Bounded window previews ONLY. No source-wide reproject/read.
from rasterio.windows import from_bounds as bounds_window
from rasterio.warp import transform_bounds
from rasterio.enums import Resampling

PANEL_M=350
PANEL_N=420

def preview_window(filepath,world_bounds,crs,bands,resampling=Resampling.bilinear):
    """Read one bounding-box window, decimated straight into a small array."""
    with rasterio.open(filepath) as ds:
        assert ds.crs and max(bands)<=ds.count,(filepath,bands)
        if ds.crs==crs:
            bb=world_bounds
        else:
            bb=transform_bounds(crs,ds.crs,*world_bounds,densify_pts=21)
        win=bounds_window(*bb,transform=ds.transform)
        # Protect against invalid geometry/projection or accidental huge windows.
        if win.width<=0 or win.height<=0:
            raise ValueError(f'Empty window: {filepath}')
        if win.width>1e7 or win.height>1e7:
            raise ValueError(f'Unreasonable source window: {filepath}, {win}')
        data=ds.read(bands,window=win,out_shape=(len(bands),PANEL_N,PANEL_N),
                     boundless=True,fill_value=0,resampling=resampling,
                     out_dtype='float32')
    return data

def show_rgb(cube):
    valid=np.isfinite(cube).all(axis=0)&(cube>0).all(axis=0)
    img=np.ones((*valid.shape,3),dtype='float32')
    if valid.sum()>5:
        lo=np.percentile(cube[:,valid],2,axis=1)
        hi=np.percentile(cube[:,valid],98,axis=1)
        stretch=(cube-lo[:,None,None])/(hi[:,None,None]-lo[:,None,None]+1e-6)
        img[valid]=np.clip(stretch.transpose(1,2,0)[valid],0,1)
    return img

fig,axes=plt.subplots(5,3,figsize=(13,20),layout='constrained')
prob_diagnostics=[]
idx=0
for site,names in UAV_MOSAICS.items():
    for name in names:
        ref=DF.loc[(DF.site==site)&(DF.mosaic==name)].sort_values('mean_probability')
        assert len(ref),name
        pick=ref.iloc[len(ref)//2]
        g=satellite_grid(site,100)
        cx,cy=g['transform']*(float(pick.col)+.5,float(pick.row)+.5)
        half=PANEL_M/2
        bounds=(cx-half,cy-half,cx+half,cy+half)
        # Native UAV RGB is window-read, never loaded as a complete mosaic.
        uav_rgb=preview_window(MOSAIC_PATH[name],bounds,g['crs'],[1,2,3])
        # Planet raw median RGB: R/G/B are bands 15,14,13.
        planet_rgb=preview_window(planet_file(site,'Annual'),bounds,g['crs'],[15,14,13])
        # Existing 10m NB04 UAV aggregate; band 1 is normalized mean probability.
        # NB04 aggregate bands: 1 mean probability, 2 fraction >0.5, 3 valid UAV coverage.
        # Read mask and target together: otherwise invalid/background pixels may
        # be rendered as high probability, creating a deceptively solid yellow map.
        uc=preview_window(agg_file(name,10),bounds,g['crs'],[1,3],
                          resampling=Resampling.nearest)
        prob,coverage=uc[0],uc[1]
        good=np.isfinite(prob)&np.isfinite(coverage)&(coverage>=0.90)&(prob>=0)&(prob<=1)
        if good.any():
            pv=prob[good]
            stats=dict(site=site,mosaic=name,n_valid=int(good.sum()),
                       window_valid_pct=100*float(good.mean()),
                       p05=float(np.percentile(pv,5)),p50=float(np.median(pv)),
                       p95=float(np.percentile(pv,95)),share_gt_09_pct=100*float(np.mean(pv>=.9)),
                       share_lt_01_pct=100*float(np.mean(pv<=.1)),
                       reference_100m_fraction=float(pick.mean_probability))
        else:
            stats=dict(site=site,mosaic=name,n_valid=0,window_valid_pct=0,
                       p05=np.nan,p50=np.nan,p95=np.nan,share_gt_09_pct=np.nan,
                       share_lt_01_pct=np.nan,reference_100m_fraction=float(pick.mean_probability))
        prob_diagnostics.append(stats)
        axes[idx,0].imshow(show_rgb(uav_rgb))
        axes[idx,1].imshow(show_rgb(planet_rgb))
        cmap=plt.get_cmap('magma').copy()
        cmap.set_bad('#dddddd')
        axes[idx,2].imshow(np.ma.masked_where(~good,prob),cmap=cmap,
                           vmin=0,vmax=1,interpolation='nearest')
        axes[idx,0].set_title(f'{site} {Path(name).stem}: UAV RGB')
        axes[idx,1].set_title(f'Planet {PLANET_YEAR[site]} raw median RGB')
        axes[idx,2].set_title('UAV fraction (10m, valid UAV pixels only)')
        for ax in axes[idx]: ax.axis('off')
        idx+=1
        print('PREVIEW',site,Path(name).stem,'finished',flush=True)
DIAG08=pd.DataFrame(prob_diagnostics)
display(DIAG08.round(3))
DIAG08.to_csv(OUT/'08_teacher_probability_diagnostics.csv',index=False)
print('WARNING if a 10m map has >90% of valid pixels near 1: possible masking,')
print('scaling or band-semantic issue; investigate before training a pixel U-Net.')
fig.savefig(FIG/'08_uav_planet_teacher_alignment.png',dpi=135,bbox_inches='tight')
plt.show()
plt.close(fig)
print('Window-only QA complete: examine broad landscape alignment, not individual crowns.')


## 09 · Build the 50 m training-target table from **existing verified NB04 cached aggregates**

Avoid rerunning expensive UAV raster aggregation. NB04 already stored canonical 50 m continuous Musanga fractions and valid-pixel fractions for each original mosaic. Section 07 checks the 100 m caches against NB04B samples; this section reads the corresponding 50 m caches, intersects them with `ROI.shp`, and keeps cells with at least 90% valid UAV coverage and 90% ROI membership. Source cache is not modified. This is independent of Planet availability; Section 10 checks its patches.


In [ ]:
# 09 — Build continuous 50m UAV labels (per-mosaic), enforce NB04B-style ROI/coverage rules.
import geopandas as gpd
from rasterio.features import rasterize
ROI_SOURCE=Path(uav_manifest['roi'])
assert ROI_SOURCE.is_file()
ROI_GDF=gpd.read_file(ROI_SOURCE)
assert ROI_GDF.crs and not ROI_GDF.empty
TARGET_SCALE=50
MIN_UAV_COVERAGE=.90
MIN_ROI_FRAC=.90
SUPERSAMPLE=4
TARGETS=[]
for site,names in UAV_MOSAICS.items():
    g=satellite_grid(site,TARGET_SCALE)
    polys=ROI_GDF.to_crs(g['crs'])
    fine_transform=g['transform']*Affine.scale(1/SUPERSAMPLE,1/SUPERSAMPLE)
    fine=rasterize(((geom,1) for geom in polys.geometry if geom is not None and not geom.is_empty),
        out_shape=(g['height']*SUPERSAMPLE,g['width']*SUPERSAMPLE),
        transform=fine_transform,fill=0,dtype='uint8')
    roif=fine.reshape(g['height'],SUPERSAMPLE,g['width'],SUPERSAMPLE).mean(axis=(1,3))
    del fine
    for name in names:
        probs,coverage,_=read_cached_teacher(name,site,TARGET_SCALE)
        ok=(coverage>=MIN_UAV_COVERAGE)&(roif>=MIN_ROI_FRAC)&np.isfinite(probs)&(probs>=0)&(probs<=1)
        yy,xx=np.where(ok)
        for row,col in zip(yy,xx):
            x,y=g['transform']*(int(col)+.5,int(row)+.5)
            TARGETS.append(dict(site=site,mosaic=name,group=Path(name).stem,
                year_uav=EMBEDDING_YEAR[site],year_planet=PLANET_YEAR[site],
                scale_m=TARGET_SCALE,row=int(row),col=int(col),
                x=float(x),y=float(y),crs=str(g['crs']),
                mean_probability=float(probs[row,col]),
                uav_coverage=float(coverage[row,col]),
                roi_fraction=float(roif[row,col])))
        print('50m supervised cells',site,name,len(yy),flush=True)
assert TARGETS, 'No 50m cells: inspect calibration / ROI / CRS'
T50=pd.DataFrame(TARGETS)
assert not T50.duplicated(['site','mosaic','row','col']).any()
T50.to_parquet(OUT/'09_uav_targets_50m.parquet',index=False)
display(T50.groupby(['site','group']).agg(cells=('mean_probability','size'),
    mean_fraction=('mean_probability','mean'),
    max_fraction=('mean_probability','max'),
    min_uav_coverage=('uav_coverage','min')).round(3))
print('Saved verified 50m UAV-derived labels:',OUT/'09_uav_targets_50m.parquet')


## 10 · Full 50 m spatial-feature extraction — restartable and RAM-aware

Runs on **all 14,454 target cells** from Section 09 (not just 300 per site). Reads **native 4.77 m** Planet reflectance and NIR texture neighborhoods and **10 m** embedding neighborhoods without creating coarsened satellite rasters. The target remains the existing continuous UAV-derived Musanga fraction on the 50 m grid.

**Features**: Planet B/G/R/N patch means + SD (21×21 Planet pixels, ~100 m), medians of four existing Planet NIR texture layers, and **all 64** embedding channel values at the patch centre, context means and SDs (11×11 embedding pixels, ~110 m). The wider neighborhood supplies spatial context; it does **not** change the 50 m response support.

**Restart behaviour**: deterministic 200-cell chunks, written as Parquet with an entry for **every** input cell, including rejected patches and reason. Completed chunks are checked and skipped on rerun. Each run uses an input/configuration fingerprint (target coordinates, labels, source file metadata and window settings); changes create a new cache namespace, preventing stale reuse. A consolidated feature table includes only accepted rows, and a companion QA file records rejected cells. Source imagery and previous 300-cell pilot are preserved.

**Memory/I/O**: only small raster windows are read. Opens each site's three sources once; no full mosaics and no 64-channel cubes across the whole forest. Outputs progress after each chunk. The larger MOS dataset is extracted incrementally.

You can safely interrupt after any saved batch. Reopen Colab and rerun initialization, Section 09 (or load its existing Parquet file), then Section 10. No CNN/model training occurs here.


In [ ]:
# 10 — Full, restartable native Planet + embedding spatial feature extraction.
# Requires setup cells (FINAL, OUT, NB04...), Section 01 (planet_file/embed_file),
# and Section 09 (T50) or the existing Section 09 Parquet target file.
from pathlib import Path
import hashlib,json,os,gc,time
import numpy as np,pandas as pd,rasterio
from rasterio.windows import Window
from pyproj import Transformer
from IPython.display import display

TARGET_FILE=OUT/'09_uav_targets_50m.parquet'
if 'T50' not in globals():
    assert TARGET_FILE.is_file(),f'Run Section 09 first: {TARGET_FILE}'
    T50=pd.read_parquet(TARGET_FILE)
assert len(T50)>0 and {'site','mosaic','row','col','x','y','crs','mean_probability'}.issubset(T50.columns)
BATCH_CELLS=200
PLANET_PATCH=21  # 21 x 4.77m ~100m, centred on 50m response cell
EMB_PATCH=11     # 11 x 10m =110m embedding context
MIN_PATCH_VALID=.80
MAX_NEW_BATCHES=None  # Set e.g. 3 for a short smoke test; None = all outstanding.
FORCE_NEW_EXTRACTION=False  # True adds a new cache version; normally leave False.
EXTRACTION_VERSION='v2_native_planet_embed_20261024'
COLS_KEY=['site','mosaic','row','col']
T50=T50.sort_values(COLS_KEY,kind='stable').reset_index(drop=True)
assert not T50.duplicated(COLS_KEY).any(),'Duplicated input target cell keys'

sources={}
for site in SITES:
    sources[site]=dict(annual=planet_file(site,'Annual'),
                       texture=planet_file(site,'Texture'),
                       embeddings=embed_file(site))
    for path in sources[site].values():assert path.is_file(),path

# Source identity (path + size + modification nanoseconds) and every target
# value/position must match for a chunk to be reusable.
config=dict(version=EXTRACTION_VERSION,planet_patch=PLANET_PATCH,
            embedding_patch=EMB_PATCH,min_patch_valid=MIN_PATCH_VALID,
            chunk_size=BATCH_CELLS,
            source_identity={s:{k:dict(path=str(v),size=v.stat().st_size,
                mtime_ns=v.stat().st_mtime_ns) for k,v in paths.items()}
                for s,paths in sources.items()},
            targets_sha256=hashlib.sha256(
                pd.util.hash_pandas_object(
                    T50[['site','mosaic','row','col','x','y','crs','mean_probability']],
                    index=False).values.tobytes()).hexdigest())
config_json=json.dumps(config,sort_keys=True)
run_id=hashlib.sha256(config_json.encode()).hexdigest()[:16]
if FORCE_NEW_EXTRACTION:
    # Manual new run without changing source files
    EXTRACTION_VERSION+='__manual'
    config['version']=EXTRACTION_VERSION
    run_id=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:16]
CACHE10=OUT/'10_full_spatial_features_chunks'/run_id
CACHE10.mkdir(parents=True,exist_ok=True)
manifest_path=CACHE10/'config.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text())==config,'Cached run settings changed unexpectedly'
else:
    manifest_path.write_text(json.dumps(config,indent=2,sort_keys=True))

def one_patch(ds,bands,size,x,y,source_crs,transformer):
    tx,ty=transformer.transform(x,y)
    px,py=(~ds.transform)*(tx,ty)
    # Use a deterministic centre aligned to nearest native pixel.
    c=int(np.floor(px));r=int(np.floor(py))
    h=size//2
    return ds.read(bands,window=Window(c-h,r-h,size,size),
                   boundless=True,fill_value=0,out_dtype='float32')

def extract_batch(site,block):
    out=[]
    with rasterio.open(sources[site]['annual']) as pds,\
         rasterio.open(sources[site]['texture']) as tds,\
         rasterio.open(sources[site]['embeddings']) as eds:
        trans={id(ds):Transformer.from_crs(block.crs.iloc[0],ds.crs,always_xy=True)
               for ds in (pds,tds,eds)}
        for rec in block.itertuples(index=False):
            base=dict(site=site,mosaic=rec.mosaic,group=rec.group,
                      row=int(rec.row),col=int(rec.col),
                      target_fraction=float(rec.mean_probability),
                      x=float(rec.x),y=float(rec.y),crs=str(rec.crs))
            try:
                if str(rec.crs)!=str(block.crs.iloc[0]):
                    raise ValueError('MIXED_CRS_IN_BATCH')
                p=one_patch(pds,[13,14,15,16],PLANET_PATCH,rec.x,rec.y,
                            rec.crs,trans[id(pds)])
                t=one_patch(tds,list(range(1,10)),PLANET_PATCH,rec.x,rec.y,
                            rec.crs,trans[id(tds)])
                e=one_patch(eds,list(range(1,65)),EMB_PATCH,rec.x,rec.y,
                            rec.crs,trans[id(eds)])
                pm=np.isfinite(p).all(axis=0)&(p>0).all(axis=0)
                em=np.isfinite(e).all(axis=0)&np.any(e!=0,axis=0)
                tv=np.isfinite(t).all(axis=0)&(t>=0).all(axis=0)
                base.update(planet_valid=float(pm.mean()),
                            embedding_valid=float(em.mean()),
                            planet_texture_valid=float(tv.mean()))
                if pm.mean()<MIN_PATCH_VALID:
                    base.update(accepted=False,reason='PLANET_LOW_COVERAGE')
                elif em.mean()<MIN_PATCH_VALID:
                    base.update(accepted=False,reason='EMBEDDING_LOW_COVERAGE')
                elif tv.mean()<MIN_PATCH_VALID:
                    base.update(accepted=False,reason='TEXTURE_LOW_COVERAGE')
                else:
                    px=p[:,pm]
                    ex=e[:,em]
                    center=e[:,EMB_PATCH//2,EMB_PATCH//2]
                    if not np.isfinite(center).all():
                        base.update(accepted=False,reason='EMBEDDING_CENTRE_INVALID')
                    else:
                        base.update(accepted=True,reason='OK')
                        for i,band in enumerate(['B','G','R','N']):
                            base[f'PLANET_mean_{band}']=float(px[i].mean())
                            base[f'PLANET_std_{band}']=float(px[i].std())
                        for i,band in enumerate(['NIR_CV_r1','NIR_CV_r3',
                                                  'NIR_CV_r7','NIR_highpass']):
                            v=t[i,tv]
                            base[f'PLANET_texture_{band}']=float(np.median(v))
                        means=ex.mean(axis=1);stds=ex.std(axis=1)
                        for j in range(64):
                            base[f'EMB_center_{j+1:02d}']=float(center[j])
                            base[f'EMB_mean_{j+1:02d}']=float(means[j])
                            base[f'EMB_sd_{j+1:02d}']=float(stds[j])
            except (OSError,ValueError,rasterio.errors.RasterioIOError) as exc:
                base.update(accepted=False,reason='READ_ERROR',error=str(exc)[:250])
            out.append(base)
    return pd.DataFrame(out)

# Separate every accepted/rejected target record; one output row per input cell.
# On a crash, only the currently active unsaved chunk needs recomputing.
work=[]
for site in SITES:
    site_cells=T50.loc[T50.site==site].reset_index(drop=True)
    for offset in range(0,len(site_cells),BATCH_CELLS):
        block=site_cells.iloc[offset:offset+BATCH_CELLS].copy()
        key=f'{site}_{offset:06d}_{len(block):03d}'
        work.append((site,block,CACHE10/f'{key}.parquet'))
print('FULL DATASET:',len(T50),'cells;',len(work),'restartable chunks;',
      'cache',CACHE10,flush=True)
print('Existing 300/site pilot is untouched.',flush=True)

new_batches=0
start=time.monotonic()
for k,(site,block,path) in enumerate(work,1):
    if path.exists():
        try:
            old=pd.read_parquet(path,columns=COLS_KEY+['accepted','reason'])
            expect=block[COLS_KEY].reset_index(drop=True)
            if len(old)==len(block) and old[COLS_KEY].reset_index(drop=True).equals(expect):
                print(f'[{k}/{len(work)}] CACHED {site} {path.name}: '
                      f'{int(old.accepted.sum())}/{len(old)} accepted',flush=True)
                continue
        except Exception as exc:
            print('INCOMPLETE CACHE ignored:',path.name,repr(exc),flush=True)
        # Do not trust a partial or stale cache; regenerate this block.
    if MAX_NEW_BATCHES is not None and new_batches>=MAX_NEW_BATCHES:
        print('Reached MAX_NEW_BATCHES; rerun to continue.',flush=True)
        break
    t0=time.monotonic()
    data=extract_batch(site,block)
    assert len(data)==len(block) and data[COLS_KEY].reset_index(drop=True).equals(
        block[COLS_KEY].reset_index(drop=True)),('Unexpected extraction row mismatch',site,k)
    tmp=path.with_suffix('.parquet.partial')
    data.to_parquet(tmp,index=False)
    os.replace(tmp,path)
    new_batches+=1
    print(f'[{k}/{len(work)}] SAVED {site} {path.name}: '
          f'{int(data.accepted.sum())}/{len(data)} accepted '
          f'({time.monotonic()-t0:.1f}s; overall {time.monotonic()-start:.1f}s)',
          flush=True)
    del data
    gc.collect()

# Only consolidate if every chunk is present (otherwise keep reusable batches).
missing=[str(path) for _,_,path in work if not path.exists()]
if missing:
    print('EXTRACTION PARTIAL:',len(missing),'remaining chunks. Rerun Section 10 to resume.')
else:
    # Bounded accumulated table: 14,454 rows × ~220 columns, modest in memory.
    all_rows=pd.concat([pd.read_parquet(path) for _,_,path in work],ignore_index=True)
    assert len(all_rows)==len(T50)
    assert all_rows[COLS_KEY].equals(T50[COLS_KEY]),'Joined rows out of order'
    accepted=all_rows.loc[all_rows.accepted].copy()
    rejected=all_rows.loc[~all_rows.accepted].copy()
    full_file=OUT/'10_full_spatial_features_50m.parquet'
    qa_file=OUT/'10_full_spatial_features_rejections_50m.parquet'
    report=OUT/'10_full_spatial_features_site_report.csv'
    accepted.to_parquet(full_file,index=False)
    rejected.to_parquet(qa_file,index=False)
    report_df=all_rows.groupby('site').agg(
        total=('accepted','size'),accepted=('accepted','sum'),
        mean_planet_valid=('planet_valid','mean'),
        mean_embedding_valid=('embedding_valid','mean')).reset_index()
    report_df['rejected']=report_df.total-report_df.accepted
    display(report_df.round(3))
    print('Rejection reasons:')
    display(all_rows.groupby(['site','reason']).size().rename('cells').reset_index())
    report_df.to_csv(report,index=False)
    print('COMPLETE:',len(accepted),'accepted /',len(all_rows),'targets.')
    print('Features:',full_file)
    print('QA rejected:',qa_file)
    print('Summary:',report)
    print('No model fit; preserve geographic groups for spatial holdout.')


## 11 · Validated spatial-feature ablation against S2 + embeddings (50 m)

**Objective:** quantify the *incremental* predictive value of 4.77 m Planet spectral/texture summaries and 10 m embedding spatial context beyond an identically constructed **50 m S2 annual + three-year quarterly + annual embedding** baseline.

**Feature sets (nested; same target cells and evaluation folds):**
- **B0:** 50 m S2 annual + 3-year quarterly + 50 m area-averaged 64-band annual embeddings.
- **B1:** B0 + Planet 4-band patch means/SD.
- **B2:** B0 + four Planet NIR texture statistics.
- **B3:** B0 + embedding neighborhood means/SD.
- **B4:** B0 + all Planet spectral and texture features.
- **B5:** B0 + all Planet + embedding spatial features.

**Important methodological distinction:** Unlike the earlier NB04B pilot, this block constructs predictors at **50 m** and uses **all 14,454 labelled cells** as candidate test observations. It aggregates baseline 10 m predictors onto the 50 m lattice (annual and quarterly S2, annual embeddings), while Planet statistics use native 4.77 m neighborhoods. The embedding spatial features use 10 m neighborhoods. These are different predictor supports; the CNN experiments will subsequently operate on whole image patches.

**Validation:** (a) 3 deterministic ~400 m held-out spatial-block folds, with a **150 m train–test exclusion buffer** within forests; (b) strict 4-forest leave-one-site-out (MOS 1–2 and 5–6 always held out together). Evaluation cells are fixed across models within each scheme, and training is **forest-balanced** (up to 1,500 cells per forest per fold, after excluding test/buffer). Spatially adjacent windows and MOS overlaps never cross the split boundary. Predicting on all eligible held-out cells still preserves the true imbalanced site distribution; report *equal-forest macro scores* alongside pooled metrics.

**Algorithms:** Histogram Gradient Boosting (HGB), Random Forest (RF) and Extra Trees (ET) are fit on identical folds, training-cell selections and six predictor sets. All **42 existing HGB prediction checkpoints remain unchanged and reusable**; the RF and ET variants are saved with algorithm suffixes, for 126 model/fold evaluations in total. RF/ET use 150 trees, minimum leaf size 5, 75% predictors considered at a split and 2 threads. Compare B0→B1–B5 within each algorithm, not simply the overall winning score.\n\n**Restartability and resource use:** Baseline 50 m inputs are built site-by-site from the existing 10 m GeoTIFFs and cached using source/target fingerprints. Each model/fold has a checkpoint of the **held-out predictions**, written atomically. No source UAV mosaics are opened. RF-like boosting uses one CPU thread and a fixed iteration budget to fit reasonably on ordinary Colab RAM. Use `MAX_NEW_FITS=3` for a smoke test, then `None` for completion.

**Outputs:** fold prediction Parquets, fold metrics, equal-forest and pooled summaries, per-site held-out metrics, MOS high-fraction bias, and scatterplots from held-out predictions. **R² can be negative** on strict site transfer; report it without suppressing failures. This is an assessment of UAV-derived pseudo-labels, not independently measured Musanga presence.


In [ ]:
# 11A — Assemble exact same 50m cells: existing spatial features + 50m S2/embeddings.
import os,json,hashlib,math,time,gc
from pathlib import Path
import numpy as np,pandas as pd,rasterio
from rasterio.warp import reproject,Resampling
from affine import Affine
from IPython.display import display

BLOCK11=OUT/'11_spatial_ablation_50m'
INPUT11=BLOCK11/'inputs'
FOLD11=BLOCK11/'fold_predictions'
FIG11=BLOCK11/'figures'
for folder in (BLOCK11,INPUT11,FOLD11,FIG11):
    folder.mkdir(parents=True,exist_ok=True)
DATA11=OUT/'10_full_spatial_features_50m.parquet'
assert DATA11.is_file(),'Complete Section 10 first.'
SET11={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
QWINDOW11={'Baego':(2021,2023),'MOS':(2022,2024),
           'Yangambi':(2022,2024),'Yoko':(2018,2020)}
COLUMN_KEY=['site','mosaic','row','col']
SPATIAL_FEATURES11=pd.read_parquet(DATA11)
assert SPATIAL_FEATURES11.accepted.all(),'Unexpected rejected examples in accepted-only dataset'
assert not SPATIAL_FEATURES11.duplicated(COLUMN_KEY).any()
assert len(SPATIAL_FEATURES11)==14454,('Unexpected number of training cells',len(SPATIAL_FEATURES11))
SPATIAL_FEATURES11=SPATIAL_FEATURES11.sort_values(COLUMN_KEY,kind='stable').reset_index(drop=True)
S2_SOURCE11={}
for site in SITES:
    yr=SET11[site];a,b=QWINDOW11[site]
    S2_SOURCE11[site]={
        'RAW_ANNUAL':NB04/'RAW_ANNUAL'/f'{site}_RAW_ANNUAL_{yr}_10m.tif',
        'S2_QUARTERLY_3YR':NB04/'S2_QUARTERLY_3YR'/f'{site}_S2_QUARTERLY_3YR_{a}_{b}_10m.tif',
        'EMB_ANNUAL':NB04/'EMB_ANNUAL'/f'{site}_EMB_ANNUAL_{yr}_10m.tif'}
    for family,path in S2_SOURCE11[site].items():
        assert path.is_file(),('Missing 10m predictor',site,family,path)
INPUT_FINGERPRINT11=hashlib.sha256(json.dumps({
    'version':'B11_50m_area_averaged_20261024_v1',
    'feature_file':dict(path=str(DATA11),size=DATA11.stat().st_size,
                        mtime=DATA11.stat().st_mtime_ns),
    'sources':{s:{k:(str(p),p.stat().st_size,p.stat().st_mtime_ns)
                   for k,p in fam.items()} for s,fam in S2_SOURCE11.items()},
},sort_keys=True).encode()).hexdigest()[:16]
DATA_CACHE11=INPUT11/f'combined_50m_{INPUT_FINGERPRINT11}.parquet'

def mean_baseline_site(site,local):
    # Reproject/average each satellite band from 10m directly onto native 50m
    # reference lattice. Destination is only ~1/25 original pixel count.
    with rasterio.open(S2_SOURCE11[site]['RAW_ANNUAL']) as ref:
        assert ref.crs and abs(ref.res[0]-10)<.05
        tfm=ref.transform*Affine.scale(5,5)
        width=int(math.ceil(ref.width/5));height=int(math.ceil(ref.height/5))
        crs=ref.crs
    rr=local.row.to_numpy(dtype=int);cc=local.col.to_numpy(dtype=int)
    assert ((rr>=0)&(rr<height)&(cc>=0)&(cc<width)).all(),(site,'cell outside canonical grid')
    arrs={}
    for family,filepath in S2_SOURCE11[site].items():
        with rasterio.open(filepath) as src:
            assert src.crs==crs,(site,family,'CRS differs')
            expected=44 if family=='S2_QUARTERLY_3YR' else (64 if family=='EMB_ANNUAL' else None)
            if expected is not None:assert src.count==expected,(site,family,src.count)
            for band in range(1,src.count+1):
                dest=np.full((height,width),np.nan,dtype='float32')
                # Quarterly every 11th band = clear-observation count: nearest;
                # remaining reflectance channels use area-weighted averaging.
                res=Resampling.nearest if family=='S2_QUARTERLY_3YR' and band%11==0 else Resampling.average
                reproject(rasterio.band(src,band),dest,
                          dst_transform=tfm,dst_crs=crs,
                          src_nodata=src.nodata,dst_nodata=np.nan,
                          resampling=res,num_threads=1)
                arrs[f'{family}__{band:02d}']=dest[rr,cc].copy()
                del dest
        print('BASELINE',site,family,'bands',src.count,flush=True)
    return pd.DataFrame(arrs,index=local.index)

if DATA_CACHE11.exists():
    D11=pd.read_parquet(DATA_CACHE11)
    assert len(D11)==len(SPATIAL_FEATURES11)
    assert D11[COLUMN_KEY].equals(SPATIAL_FEATURES11[COLUMN_KEY])
    print('CACHED 50m baseline + native spatial features:',DATA_CACHE11,flush=True)
else:
    assembled=[]
    for site in SITES:
        local=SPATIAL_FEATURES11.loc[SPATIAL_FEATURES11.site==site].copy()
        features=mean_baseline_site(site,local)
        joined=pd.concat([local,features],axis=1)
        part=INPUT11/f'{site}_{INPUT_FINGERPRINT11}.parquet'
        joined.to_parquet(part,index=False)
        assembled.append(joined)
        print('SAVED site input:',site,len(joined),flush=True)
        del local,features,joined
        gc.collect()
    D11=pd.concat(assembled,ignore_index=True).sort_values(COLUMN_KEY,kind='stable').reset_index(drop=True)
    D11.to_parquet(DATA_CACHE11,index=False)

BASE11=[x for x in D11.columns if x.startswith(('RAW_ANNUAL__','S2_QUARTERLY_3YR__','EMB_ANNUAL__'))]
SPEC11=[f'PLANET_{k}_{b}' for k in ('mean','std') for b in ('B','G','R','N')]
TEX11=[f'PLANET_texture_{b}' for b in ('NIR_CV_r1','NIR_CV_r3','NIR_CV_r7','NIR_highpass')]
EMBCTX11=[f'EMB_{stat}_{j:02d}' for stat in ('mean','sd') for j in range(1,65)]
FEATURE_SETS11={
    'B0_S2_EMB':BASE11,
    'B1_B0_PlanetSpectral':BASE11+SPEC11,
    'B2_B0_PlanetTexture':BASE11+TEX11,
    'B3_B0_EmbeddingSpatial':BASE11+EMBCTX11,
    'B4_B0_AllPlanet':BASE11+SPEC11+TEX11,
    'B5_B0_AllSpatial':BASE11+SPEC11+TEX11+EMBCTX11,
}
assert len(BASE11)>=108,(len(BASE11),'baseline should include 64 embeddings + 44 quarterly + annual S2')
all_columns=sorted(set(sum(FEATURE_SETS11.values(),[])))
missing=[x for x in all_columns if x not in D11.columns]
assert not missing,missing
# Exactly the SAME finite and valid target cells for all feature sets.
clearcols=[f'S2_QUARTERLY_3YR__{band:02d}' for band in (11,22,33,44)]
assert all(x in D11.columns for x in clearcols)
valid=np.isfinite(D11[all_columns].to_numpy(dtype='float32')).all(axis=1)
valid &= (D11[clearcols].to_numpy(dtype='float32')>=1).all(axis=1)
valid &= D11.target_fraction.between(0,1).to_numpy()
display(pd.DataFrame([dict(total=len(D11),common_complete=int(valid.sum()),
    rejected=int((~valid).sum()),features_B0=len(BASE11),
    features_B5=len(FEATURE_SETS11['B5_B0_AllSpatial']))]))
D11=D11.loc[valid].reset_index(drop=True)
assert D11.site.nunique()==4 and len(D11)>1000,'Too few eligible cells; inspect quarterly coverage.'
display(D11.groupby('site').target_fraction.agg(['size','mean','max']).round(3))
print('50m common-case dataset:',len(D11),'rows; source:',DATA_CACHE11)


In [ ]:
# 11B — Restartable spatial and forest-held-out benchmarking + reporting.
import os,json,hashlib,math,time
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from sklearn.ensemble import HistGradientBoostingRegressor,RandomForestRegressor,ExtraTreesRegressor
from sklearn.metrics import r2_score
from IPython.display import display

SPATIAL_BLOCK_M=400
BUFFER_M=150
SPATIAL_FOLDS=3
TRAIN_CAP_PER_FOREST=1500
MODEL_SEED=20261024
MAX_NEW_FITS=None  # Use 3 for a short smoke test; None completes remaining RF/ET jobs.
ALGORITHMS11=['HGB','RF','ET']  # HGB checkpoint names remain unchanged.
RF_ET_ARGS=dict(n_estimators=150,min_samples_leaf=5,max_features=0.75,
                max_depth=None,n_jobs=2,random_state=MODEL_SEED)
def estimator11(algorithm):
    if algorithm=='HGB':return HistGradientBoostingRegressor(**MODEL_ARGS)
    if algorithm=='RF':return RandomForestRegressor(**RF_ET_ARGS)
    if algorithm=='ET':return ExtraTreesRegressor(**RF_ET_ARGS)
    raise ValueError(algorithm)
HIGH_COVER=.40
MODEL_ARGS=dict(learning_rate=.06,max_iter=180,max_leaf_nodes=20,
                min_samples_leaf=35,l2_regularization=2,
                early_stopping=False,random_state=MODEL_SEED)
# Keep memory/time consistent on Colab; models are serial.
os.environ['OMP_NUM_THREADS']='1'
import threadpoolctl
threadpoolctl.threadpool_limits(limits=1)

D11=D11.reset_index(drop=True).copy()
# Projected x/y from Section 09, units metres, each site in EPSG:32635.
assert D11.crs.nunique()==1 and '32635' in str(D11.crs.iloc[0])
xx=D11.x.to_numpy(float);yy=D11.y.to_numpy(float)
assert np.isfinite(xx).all() and np.isfinite(yy).all()
# Stable coordinates; label nearby 400m tiles across all sites.
bx=np.floor(xx/SPATIAL_BLOCK_M).astype('int64')
by=np.floor(yy/SPATIAL_BLOCK_M).astype('int64')
# Deterministic 3-colour spatial blocks (not random 50m cell splitting).
blockid=(np.mod(bx*73856093+by*19349663,SPATIAL_FOLDS)).astype('int8')
D11['spatial_fold']=blockid
schemes11=['BLOCK_BUFFERED','LEAVE_FOREST_OUT']
folds11=[]
for scheme in schemes11:
    if scheme=='BLOCK_BUFFERED':
        folds11.extend((scheme,str(i)) for i in range(SPATIAL_FOLDS))
    else:
        folds11.extend((scheme,site) for site in SITES)

def split11(scheme,heldout):
    # Derive coordinates locally: preview/diagnostic cells may overwrite global xx/yy.
    local_x=D11.x.to_numpy(float)
    local_y=D11.y.to_numpy(float)
    test=(D11.spatial_fold.to_numpy()==int(heldout)) if scheme=='BLOCK_BUFFERED' else (
        D11.site.to_numpy()==heldout)
    train=~test
    if scheme=='BLOCK_BUFFERED':
        # Strict within-forest 150m exclusion around EVERY heldout centre.
        sites=D11.site.to_numpy()
        for site in SITES:
            tix=np.where(test&(sites==site))[0]
            ix=np.where(train&(sites==site))[0]
            if not len(tix) or not len(ix):continue
            tree=cKDTree(np.column_stack([local_x[tix],local_y[tix]]))
            near=tree.query(np.column_stack([local_x[ix],local_y[ix]]),k=1)[0] < BUFFER_M
            train[ix[near]]=False
    assert not (train & test).any()
    # Forest-balanced fit; retaining every heldout test cell for metrics.
    rng=np.random.default_rng(MODEL_SEED)
    train_ix=[]
    for site in SITES:
        eligible=np.where(train & (D11.site.to_numpy()==site))[0]
        if len(eligible)>TRAIN_CAP_PER_FOREST:
            eligible=rng.choice(eligible,TRAIN_CAP_PER_FOREST,replace=False)
        train_ix.extend(eligible)
    train_ix=np.asarray(sorted(train_ix),dtype=int)
    test_ix=np.where(test)[0]
    assert len(train_ix)>=100 and len(test_ix)>=10,(scheme,heldout,len(train_ix),len(test_ix))
    return train_ix,test_ix,int((~train&~test).sum())

def metric11(y,p):
    y=np.asarray(y,float);p=np.asarray(p,float)
    err=p-y
    high=y>=HIGH_COVER
    q25,q75=np.percentile(y,[25,75])
    return dict(n=len(y),R2=float(r2_score(y,p)) if len(y)>1 and np.var(y)>1e-12 else np.nan,
        MAE_pp=float(100*np.mean(abs(err))),RMSE_pp=float(100*np.sqrt(np.mean(err**2))),
        bias_pp=float(100*np.mean(err)),
        RPD=float(np.std(y)/np.sqrt(np.mean(err**2))) if np.mean(err**2)>0 else np.nan,
        RPIQ=float((q75-q25)/np.sqrt(np.mean(err**2))) if np.mean(err**2)>0 else np.nan,
        n_high=int(high.sum()),high_bias_pp=float(100*np.mean(err[high])) if high.any() else np.nan,
        high_recall=float(np.mean(p[high]>=HIGH_COVER)) if high.any() else np.nan)

signature=hashlib.sha256(json.dumps(dict(
    input_fingerprint=INPUT_FINGERPRINT11,feature_sets=FEATURE_SETS11,
    complete_case_keys=hashlib.sha256(pd.util.hash_pandas_object(
        D11[COLUMN_KEY],index=False).values.tobytes()).hexdigest(),
    block_m=SPATIAL_BLOCK_M,buffer_m=BUFFER_M,spatial_folds=SPATIAL_FOLDS,
    cap=TRAIN_CAP_PER_FOREST,seed=MODEL_SEED,model=MODEL_ARGS),sort_keys=True).encode()).hexdigest()[:16]
RUN11=FOLD11/signature
RUN11.mkdir(parents=True,exist_ok=True)
(RUN11/'config.json').write_text(json.dumps(dict(signature=signature,model=MODEL_ARGS,
    spatial_block_m=SPATIAL_BLOCK_M,buffer_m=BUFFER_M),indent=2))
print('MODEL RUN:',signature,'folds:',folds11,'features:',list(FEATURE_SETS11),'algorithms:',ALGORITHMS11)
print('Restartable fold prediction cache:',RUN11,flush=True)
completed=0;new_fits=0;fold_report=[]
for scheme,heldout in folds11:
    train_ix,test_ix,n_excluded=split11(scheme,heldout)
    print('FOLD',scheme,heldout,'training',len(train_ix),'testing',len(test_ix),
          'buffer excluded',n_excluded,flush=True)
    fold_report.append(dict(scheme=scheme,heldout=heldout,n_train=len(train_ix),
                            n_test=len(test_ix),n_buffer_excluded=n_excluded))
    for algorithm in ALGORITHMS11:
        for set_name,cols in FEATURE_SETS11.items():
            # HGB filenames are deliberately unchanged to reuse all 42 prior fits.
            stem=f'{scheme}_{heldout}_{set_name}'
            if algorithm!='HGB': stem+=f'_{algorithm}'
            path=RUN11/f'{stem}.parquet'
            if path.exists():
                try:
                    head=pd.read_parquet(path,columns=COLUMN_KEY+['observed','predicted'])
                    expect=D11.iloc[test_ix][COLUMN_KEY].reset_index(drop=True)
                    if len(head)==len(test_ix) and head[COLUMN_KEY].reset_index(drop=True).equals(expect):
                        completed+=1
                        print('  CACHED',algorithm,set_name,flush=True)
                        continue
                except Exception as exc:
                    print('  RECOMPUTE corrupt cache:',repr(exc)[:150],flush=True)
            if MAX_NEW_FITS is not None and new_fits>=MAX_NEW_FITS:
                print('Reached fit cap; rerun Block 11B to resume.',flush=True)
                break
            xtr=D11.iloc[train_ix][cols].to_numpy('float32')
            xte=D11.iloc[test_ix][cols].to_numpy('float32')
            ytr=D11.iloc[train_ix].target_fraction.to_numpy('float32')
            model=estimator11(algorithm)
            start=time.monotonic()
            model.fit(xtr,ytr)
            pred=np.clip(model.predict(xte),0,1)
            saved=D11.iloc[test_ix][COLUMN_KEY+['group','x','y','target_fraction']].copy().reset_index(drop=True)
            saved=saved.rename(columns={'target_fraction':'observed'})
            saved['predicted']=pred
            tmp=path.with_suffix('.parquet.partial')
            saved.to_parquet(tmp,index=False)
            os.replace(tmp,path)
            new_fits+=1;completed+=1
            print('  SAVED',algorithm,set_name,'seconds',round(time.monotonic()-start,1),flush=True)
            del model,xtr,xte,ytr,saved
        if MAX_NEW_FITS is not None and new_fits>=MAX_NEW_FITS:
            break
    if MAX_NEW_FITS is not None and new_fits>=MAX_NEW_FITS:
        break
pd.DataFrame(fold_report).to_csv(BLOCK11/'11_fold_sizes.csv',index=False)
expected=[RUN11/(f'{scheme}_{heldout}_{name}'+(f'_{algorithm}' if algorithm!='HGB' else '')+'.parquet')
          for scheme,heldout in folds11 for algorithm in ALGORITHMS11 for name in FEATURE_SETS11]
missing=[path for path in expected if not path.exists()]
print('Checkpoint status:',len(expected)-len(missing),'/',len(expected),'model/fold results.')
if missing:
    print('INCOMPLETE — rerun this cell. Missing:',len(missing),'fits.')
else:
    all_rows=[]
    for scheme,heldout in folds11:
        for algorithm in ALGORITHMS11:
            for set_name in FEATURE_SETS11:
                stem=f'{scheme}_{heldout}_{set_name}'+(f'_{algorithm}' if algorithm!='HGB' else '')
                path=RUN11/f'{stem}.parquet'
                preds=pd.read_parquet(path)
                preds['scheme']=scheme;preds['holdout']=heldout
                preds['model']=set_name;preds['algorithm']=algorithm
                all_rows.append(preds)
    ALL11=pd.concat(all_rows,ignore_index=True)
    ALL11.to_parquet(BLOCK11/'11_all_heldout_predictions.parquet',index=False)
    by_site=[];by_fold=[];summary=[]
    for (scheme,algorithm,set_name),g in ALL11.groupby(['scheme','algorithm','model'],sort=False):
        # Across spatial folds, each target is held out once.
        # Across LOSO, each forest is held out once.
        assert not g.duplicated(COLUMN_KEY).any(),(scheme,set_name,'duplicated test predictions')
        by_fold.extend(dict(scheme=scheme,algorithm=algorithm,model=set_name,holdout=fold,
                            **metric11(sub.observed,sub.predicted))
                       for fold,sub in g.groupby('holdout'))
        site_stats=[]
        for site,sg in g.groupby('site'):
            site_stats.append(dict(scheme=scheme,algorithm=algorithm,model=set_name,site=site,
                                   **metric11(sg.observed,sg.predicted)))
        by_site.extend(site_stats)
        pooled=metric11(g.observed,g.predicted)
        macro_mae=np.mean([v['MAE_pp'] for v in site_stats])
        macro_rmse=np.mean([v['RMSE_pp'] for v in site_stats])
        macro_bias=np.mean([v['bias_pp'] for v in site_stats])
        macro_r2=np.nanmean([v['R2'] for v in site_stats])
        mos=g[g.site=='MOS'];mh=mos.observed>=HIGH_COVER
        # Equal-site weighted R², calculated on a common global target mean.
        counts=g.groupby('site').size()
        weights=1/g.site.map(counts).to_numpy(float);weights/=weights.sum()
        y=g.observed.to_numpy(float);prediction=g.predicted.to_numpy(float)
        avg=np.dot(weights,y)
        ss=np.dot(weights,(y-avg)**2)
        balanced_r2=1-np.dot(weights,(y-prediction)**2)/ss if ss>0 else np.nan
        summary.append(dict(scheme=scheme,algorithm=algorithm,model=set_name,n=len(g),
            pooled_R2=pooled['R2'],pooled_MAE_pp=pooled['MAE_pp'],
            pooled_RMSE_pp=pooled['RMSE_pp'],
            equal_forest_R2=float(balanced_r2),
            mean_site_R2=float(macro_r2),
            equal_forest_MAE_pp=float(macro_mae),
            equal_forest_RMSE_pp=float(macro_rmse),
            equal_forest_bias_pp=float(macro_bias),
            MOS_high_n=int(mh.sum()),
            MOS_high_bias_pp=float(100*(mos.predicted[mh]-mos.observed[mh]).mean()) if mh.any() else np.nan))
    SUMMARY11=pd.DataFrame(summary)
    SITES11=pd.DataFrame(by_site)
    FOLDS11=pd.DataFrame(by_fold)
    SUMMARY11.to_csv(BLOCK11/'11_global_metrics.csv',index=False)
    SITES11.to_csv(BLOCK11/'11_site_metrics.csv',index=False)
    FOLDS11.to_csv(BLOCK11/'11_fold_metrics.csv',index=False)
    for scheme in schemes11:
        print('VALIDATION:',scheme,'— compare predictors within each algorithm')
        display(SUMMARY11[SUMMARY11.scheme==scheme].sort_values('equal_forest_MAE_pp').round(3))
    print('Site-level diagnostics (LOSO):')
    display(SITES11[SITES11.scheme=='LEAVE_FOREST_OUT'].round(3))
    for scheme in schemes11:
        for algorithm in ALGORITHMS11:
            f=ALL11[(ALL11.scheme==scheme)&(ALL11.algorithm==algorithm)]
            fig,axes=plt.subplots(1,4,figsize=(16,4),layout='constrained')
            for ax,site in zip(axes,SITES):
                q=f[(f.site==site)&(f.model=='B0_S2_EMB')]
                q2=f[(f.site==site)&(f.model=='B5_B0_AllSpatial')]
                for sub,label in ((q,'B0'),(q2,'B5')):
                    ax.scatter(sub.observed*100,sub.predicted*100,s=2,
                               alpha=.12,label=label,rasterized=True)
                ax.plot([0,100],[0,100],ls='--',color='grey',lw=1)
                ax.set(xlim=(0,100),ylim=(0,100),title=site,
                       xlabel='UAV Musanga (%)',ylabel='Predicted (%)')
                ax.legend()
            fig.suptitle(f'{scheme} | {algorithm} | held-out B0 vs B5')
            fig.savefig(FIG11/f'{scheme}_{algorithm}_B0_vs_B5_scatter.png',dpi=150)
            plt.show()
            plt.close(fig)
    print('COMPLETE: all benchmark metrics and held-out predictions saved:',BLOCK11)


## 11C — HGB-only recovery of NB04 training strategy

Compare **two training samples (forest-balanced random vs NB04-style abundance-stratified)** × **two prediction variants (single HGB vs five independently resampled HGB models)** on only **B0 and B1**. Use the exact existing 11B spatial and leave-one-forest-out folds, no RF or Extra Trees fitting. The original random/single HGB predictions are reused unchanged as ensemble member 0; all existing 11B outputs stay untouched.

As in NB04, stratification reserves up to 25% of the training cap for examples with Musanga fraction ≥40%, when available, then approximately balances remaining draws among the four abundance bins. Each forest is capped separately at 1,500 cells, and no example is duplicated. Model hyperparameters remain frozen. The saved member predictions make this restartable; incomplete ensembles are never reported. Run 11A and 11B first in a fresh session (11B reuses existing predictions), then 11C.

Compare within **each feature set**: effect of sampling (random vs stratified), effect of averaging (single vs ensemble), and MOS high-cover bias versus Yangambi errors. This is a focused NB04 methodological comparison at 50 m, **not** a direct replication of NB04's 100 m scores.

In [ ]:
# 11C — NB04-style stratified HGB, five-member ensembles; restartable.
import os,hashlib,json,time
import numpy as np,pandas as pd
from scipy.spatial import cKDTree
from sklearn.ensemble import HistGradientBoostingRegressor
from IPython.display import display
assert all(k in globals() for k in ['D11','split11','RUN11','MODEL_ARGS','metric11','folds11'])
F11C=['B0_S2_EMB','B1_B0_PlanetSpectral']
N11C=5
MAX_NEW_FITS11C=None  # Set 3 to smoke-test; None runs remaining members.
conf11c=dict(version='11C_v1',parent=RUN11.name,features=F11C,
             folds=folds11,n_members=N11C,cap=TRAIN_CAP_PER_FOREST,
             seed=MODEL_SEED,model=MODEL_ARGS,high_share=.25)
key11c=hashlib.sha256(json.dumps(conf11c,sort_keys=True).encode()).hexdigest()[:16]
DIR11C=BLOCK11/'11C_HGB_sampling_ensemble'/key11c
DIR11C.mkdir(parents=True,exist_ok=True)
(DIR11C/'config.json').write_text(json.dumps(conf11c,indent=2))
sites11c=D11.site.to_numpy()
xy11c=D11[['x','y']].to_numpy(float)
y11c=D11.target_fraction.to_numpy(float)
def allowed11c(scheme,hold,test):
    eligible=np.ones(len(D11),dtype=bool)
    eligible[test]=False
    if scheme=='BLOCK_BUFFERED':
        for forest in SITES:
            ti=test[sites11c[test]==forest]
            ix=np.flatnonzero(eligible & (sites11c==forest))
            if len(ti) and len(ix):
                near=cKDTree(xy11c[ti]).query(xy11c[ix],k=1)[0]<BUFFER_M
                eligible[ix[near]]=False
    return np.flatnonzero(eligible)
def sample11c(pool,mode,seed):
    rng=np.random.default_rng(seed)
    chosen=[]
    for forest in SITES:
        ids=pool[sites11c[pool]==forest]
        cap=min(len(ids),TRAIN_CAP_PER_FOREST)
        if len(ids)<=cap:
            chosen.extend(ids.tolist());continue
        if mode=='random':
            selected=rng.choice(ids,cap,replace=False).tolist()
        else:
            classes=np.digitize(y11c[ids],[.10,.20,.40])
            bins={k:ids[classes==k].copy() for k in range(4)}
            n_high=min(len(bins[3]),int(np.ceil(.25*cap)))
            selected=rng.choice(bins[3],n_high,replace=False).tolist()
            bins[3]=np.setdiff1d(bins[3],np.asarray(selected,dtype=int))
            while len(selected)<cap:
                active=[k for k in range(4) if len(bins[k])]
                assert active
                quota=max(1,(cap-len(selected))//len(active))
                for k in active:
                    n=min(quota,len(bins[k]),cap-len(selected))
                    if n:
                        new=rng.choice(bins[k],n,replace=False)
                        selected.extend(new.tolist())
                        bins[k]=np.setdiff1d(bins[k],new)
                    if len(selected)==cap:break
        assert len(selected)==cap and len(set(selected))==cap
        chosen.extend(selected)
    return np.asarray(sorted(chosen),dtype=int)
def archived11c(scheme,hold,feature):
    return RUN11/f'{scheme}_{hold}_{feature}.parquet'
def checkpoint11c(scheme,hold,feature,mode,member):
    return (archived11c(scheme,hold,feature) if mode=='random' and member==0
            else DIR11C/f'{scheme}_{hold}_{feature}_{mode}_m{member}.parquet')
def valid11c(path,expected):
    if not path.exists():return False
    try:
        d=pd.read_parquet(path)
        return (len(d)==len(expected) and d[COLUMN_KEY].reset_index(drop=True).equals(expected)
                and np.isfinite(d.predicted).all())
    except Exception:return False
new11c=0;stop11c=False
for scheme,hold in folds11:
    train_original,test,_=split11(scheme,hold)
    pool=allowed11c(scheme,hold,test)
    # Reconstruct original 11B sampling exactly. Its random member 0 is cached.
    assert np.array_equal(sample11c(pool,'random',MODEL_SEED),train_original)
    expected=D11.iloc[test][COLUMN_KEY].reset_index(drop=True)
    for feature in F11C:
        assert valid11c(archived11c(scheme,hold,feature),expected)
        for mode in ('random','stratified'):
            for member in range(N11C):
                path=checkpoint11c(scheme,hold,feature,mode,member)
                if valid11c(path,expected):continue
                if MAX_NEW_FITS11C is not None and new11c>=MAX_NEW_FITS11C:
                    stop11c=True;break
                seed=MODEL_SEED+1009*member+sum(map(ord,str(hold)))+(100000 if mode=='stratified' else 0)
                ids=sample11c(pool,mode,seed)
                kwargs=dict(MODEL_ARGS,random_state=seed)
                start=time.monotonic()
                model=HistGradientBoostingRegressor(**kwargs)
                model.fit(D11.iloc[ids][FEATURE_SETS11[feature]].to_numpy('float32'),
                          y11c[ids].astype('float32'))
                pred=np.clip(model.predict(D11.iloc[test][FEATURE_SETS11[feature]].to_numpy('float32')),0,1)
                output=D11.iloc[test][COLUMN_KEY+['target_fraction']].copy().reset_index(drop=True)
                output=output.rename(columns={'target_fraction':'observed'})
                output['predicted']=pred
                tmp=path.with_suffix('.parquet.partial')
                output.to_parquet(tmp,index=False);os.replace(tmp,path)
                new11c+=1
                print('SAVED',scheme,hold,feature,mode,'member',member,
                      round(time.monotonic()-start,1),'s',flush=True)
            if stop11c:break
        if stop11c:break
    if stop11c:break
tasks11c=[(sc,ho,fe,mo,me) for sc,ho in folds11 for fe in F11C
          for mo in ('random','stratified') for me in range(N11C)]
missing11c=[key for key in tasks11c
            if not valid11c(checkpoint11c(*key),
                D11.iloc[split11(key[0],key[1])[1]][COLUMN_KEY].reset_index(drop=True))]
print('11C MEMBER CACHE:',len(tasks11c)-len(missing11c),'/',len(tasks11c),flush=True)
if missing11c:
    print('PARTIAL — rerun 11C to resume; outstanding',len(missing11c),flush=True)
else:
    reports=[];site_reports=[];prediction_frames=[]
    for scheme,hold in folds11:
        for feature in F11C:
            for mode in ('random','stratified'):
                members=[pd.read_parquet(checkpoint11c(scheme,hold,feature,member_mode,idx))
                         for member_mode,idx in [(mode,i) for i in range(N11C)]]
                assert all(x[COLUMN_KEY].equals(members[0][COLUMN_KEY]) for x in members)
                mat=np.stack([x.predicted.to_numpy(float) for x in members])
                for variant,pred in [('single',mat[0]),('ensemble5',mat.mean(axis=0))]:
                    q=members[0][COLUMN_KEY+['observed']].copy()
                    q['predicted']=np.clip(pred,0,1)
                    q['scheme']=scheme;q['holdout']=hold
                    q['feature']=feature;q['sampling']=mode;q['variant']=variant
                    prediction_frames.append(q)
    P11C=pd.concat(prediction_frames,ignore_index=True)
    P11C.to_parquet(DIR11C/'heldout_predictions.parquet',index=False)
    for (scheme,feature,sampling,variant),g in P11C.groupby(['scheme','feature','sampling','variant']):
        assert not g.duplicated(COLUMN_KEY).any()
        site_metrics=[]
        for site,t in g.groupby('site'):
            x=dict(scheme=scheme,feature=feature,sampling=sampling,
                   variant=variant,site=site,**metric11(t.observed,t.predicted))
            site_metrics.append(x);site_reports.append(x)
        counts=g.groupby('site').size()
        weights=1/g.site.map(counts).to_numpy(float);weights/=weights.sum()
        y=g.observed.to_numpy(float);p=g.predicted.to_numpy(float)
        mu=np.dot(weights,y);denom=np.dot(weights,(y-mu)**2)
        mos=g[g.site=='MOS'];hi=mos.observed>=.40
        reports.append(dict(scheme=scheme,feature=feature,sampling=sampling,
            variant=variant,n=len(g),pooled_R2=metric11(y,p)['R2'],
            equal_forest_R2=1-np.dot(weights,(y-p)**2)/denom,
            equal_forest_MAE_pp=np.mean([x['MAE_pp'] for x in site_metrics]),
            mean_site_R2=np.nanmean([x['R2'] for x in site_metrics]),
            MOS_high_n=int(hi.sum()),MOS_high_bias_pp=float(
                100*(mos.predicted[hi]-mos.observed[hi]).mean()) if hi.any() else np.nan))
    results11c=pd.DataFrame(reports);by_site11c=pd.DataFrame(site_reports)
    results11c.to_csv(DIR11C/'global_metrics.csv',index=False)
    by_site11c.to_csv(DIR11C/'site_metrics.csv',index=False)
    for scheme in ('BLOCK_BUFFERED','LEAVE_FOREST_OUT'):
        print('11C',scheme)
        display(results11c[results11c.scheme==scheme].sort_values(
            ['feature','equal_forest_MAE_pp']).round(3))
    print('COMPLETE:',DIR11C,flush=True)


## 11D — MOS prediction audit (no model fitting)

Investigate why the **leave-one-forest-out MOS high-abundance bias** in 11C appears exactly identical for random, stratified, single and ensemble HGB models **within each feature set**. Read **actual saved held-out predictions**, and independently recompute high-cover bias. Do **not** assume a reporting bug before inspecting the prediction vectors.

The audit checks (a) whether predictions vary at all, (b) whether predictions vary specifically for MOS observations ≥40% Musanga, (c) where predictions saturate/flatten, (d) potential leakage or stale results through unchanged predictions, and (e) changes within abundance bins and MOS mosaics. The diagnostic also verifies random/single predictions against archived 11B files and the arithmetic mean of the five member predictions. Includes high-resolution numeric tables and held-out scatter/calibration plots.

**Critical distinction:** identical *bias* can arise from different predictions that have the same mean. Conversely, precisely identical bias alongside nearly constant predictions may indicate saturation. Neither should be inferred from aggregate metrics alone.

Only persisted 11C predictions are necessary. Run initialization (Section 00), then this block; it auto-detects the completed 11C run under the usual NB06 output path. No source UAV/Planet mosaics are read or recomputed.

In [ ]:
# 11D — Independent audit of cached 11C predictions, focused on held-out MOS.
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
from itertools import combinations
import hashlib

ROOT11D=(OUT/'11_spatial_ablation_50m'/'11C_HGB_sampling_ensemble')
folders11d=sorted((p for p in ROOT11D.glob('*') if (p/'heldout_predictions.parquet').exists()),
                  key=lambda p:(p/'heldout_predictions.parquet').stat().st_mtime,reverse=True)
assert folders11d,'No completed 11C heldout_predictions.parquet. Run 11C first.'
DIR11D=folders11d[0]
AUDIT11D=DIR11D/'11D_diagnostics'
AUDIT11D.mkdir(exist_ok=True)
print('Auditing saved 11C results:',DIR11D,flush=True)
P=pd.read_parquet(DIR11D/'heldout_predictions.parquet')
need={'site','mosaic','row','col','observed','predicted',
      'scheme','holdout','feature','sampling','variant'}
assert need.issubset(P.columns),sorted(need-set(P.columns))
assert np.isfinite(P[['observed','predicted']].to_numpy(float)).all()
KEYS11D=['site','mosaic','row','col']
P['configuration']=P['feature']+' | '+P['sampling']+' | '+P['variant']
F11D=['B0_S2_EMB','B1_B0_PlanetSpectral']
MODES11D=[('random','single'),('random','ensemble5'),
          ('stratified','single'),('stratified','ensemble5')]
MOS11D=P[(P.scheme=='LEAVE_FOREST_OUT')&(P.site=='MOS')].copy()
assert MOS11D.holdout.eq('MOS').all(),'MOS data appears outside MOS holdout!'
expected11d=len(MOS11D[MOS11D.configuration==MOS11D.configuration.iloc[0]])
print('MOS test cells per configuration:',expected11d,
      '| high >=40%:',int((MOS11D[MOS11D.configuration==MOS11D.configuration.iloc[0]].observed>=.4).sum()))
assert expected11d>1000
assert len(MOS11D)==expected11d*len(F11D)*len(MODES11D)
# Every model must predict the exact same MOS cells and UAV targets.
mosaics11d={}
for feature in F11D:
    for mode,variant in MODES11D:
        q=MOS11D[(MOS11D.feature==feature)&(MOS11D.sampling==mode)&
                 (MOS11D.variant==variant)].sort_values(KEYS11D).reset_index(drop=True)
        assert len(q)==expected11d and not q.duplicated(KEYS11D).any()
        mosaics11d[(feature,mode,variant)]=q
base11d=mosaics11d[(F11D[0],'random','single')]
for q in mosaics11d.values():
    assert q[KEYS11D].equals(base11d[KEYS11D]),'Mismatched held-out MOS cell coordinates'
    assert np.allclose(q.observed,base11d.observed,atol=1e-8),'Different UAV targets!'
    
# Independently recompute MOS high-cover statistics (unrounded; by mosaic too).
high11d=base11d.observed.to_numpy(float)>=.40
y11d=base11d.observed.to_numpy(float)
stats11d=[]
for (feature,mode,variant),q in mosaics11d.items():
    pred=q.predicted.to_numpy(float)
    err=(pred-y11d)*100
    hi=pred[high11d]
    stats11d.append(dict(feature=feature,sampling=mode,variant=variant,
        n=len(q),n_high=int(high11d.sum()),
        high_actual_mean_pp=100*float(y11d[high11d].mean()),
        high_pred_mean_pp=100*float(hi.mean()),
        high_bias_pp=float(err[high11d].mean()),
        high_MAE_pp=float(abs(err[high11d]).mean()),
        high_RMSE_pp=float(np.sqrt(np.mean(err[high11d]**2))),
        high_recall_40=float((hi>=.40).mean()),
        high_pred_SD_pp=100*float(hi.std()),
        high_pred_min_pp=100*float(hi.min()),
        high_pred_p50_pp=100*float(np.median(hi)),
        high_pred_max_pp=100*float(hi.max()),
        high_unique_6dp=int(len(np.unique(np.round(hi,6)))),
        high_pred_sha256=hashlib.sha256(hi.tobytes()).hexdigest()[:12]))
S11D=pd.DataFrame(stats11d)
S11D.to_csv(AUDIT11D/'11D_MOS_high_detailed.csv',index=False,float_format='%.12g')
print('MOS >=40%: independent, UNROUNDED metrics and prediction-vector fingerprints')
display(S11D)

# Pairwise matched predictions: compare both whole MOS and its high-cover tail.
changes11d=[]
for feature in F11D:
    candidates=[(mode,variant) for mode,variant in MODES11D]
    for (m1,v1),(m2,v2) in combinations(candidates,2):
        a=mosaics11d[(feature,m1,v1)].predicted.to_numpy(float)
        b=mosaics11d[(feature,m2,v2)].predicted.to_numpy(float)
        dif=100*(b-a)
        for label,mask in [('ALL',np.ones(len(a),dtype=bool)),('HIGH_GE40',high11d)]:
            d=dif[mask]
            changes11d.append(dict(feature=feature,first=m1+'_'+v1,second=m2+'_'+v2,
                subset=label,n=int(mask.sum()),mean_delta_pp=float(d.mean()),
                mean_abs_delta_pp=float(abs(d).mean()),
                max_abs_delta_pp=float(abs(d).max()),
                pct_bitwise_identical=float(100*np.mean(a[mask]==b[mask])),
                pct_within_0_01pp=float(100*np.mean(abs(d)<.01))))
CHANGES11D=pd.DataFrame(changes11d)
CHANGES11D.to_csv(AUDIT11D/'11D_pairwise_prediction_differences.csv',index=False)
print('Paired prediction changes, including all high-Musanga pixels:')
display(CHANGES11D.round(5))

# All predicted/observed abundance classes; per mosaic confirms MOS subdomains.
bins11d=[-.000001,.10,.20,.40,.60,.80,1.000001]
labels11d=['0–10','10–20','20–40','40–60','60–80','80–100']
binned11d=[]
for (feature,mode,variant),q in mosaics11d.items():
    tmp=q.copy()
    tmp['abundance_bin']=pd.cut(tmp.observed,bins11d,labels=labels11d)
    for (mosaic,bname),chunk in tmp.groupby(['mosaic','abundance_bin'],observed=True):
        err=100*(chunk.predicted-chunk.observed)
        binned11d.append(dict(feature=feature,sampling=mode,variant=variant,
            mosaic=mosaic,observed_bin=str(bname),n=len(chunk),
            observed_mean_pp=float(100*chunk.observed.mean()),
            predicted_mean_pp=float(100*chunk.predicted.mean()),
            bias_pp=float(err.mean()),MAE_pp=float(abs(err).mean()),
            predicted_sd_pp=float(100*chunk.predicted.std(ddof=0)),
            fraction_pred_ge40=float((chunk.predicted>=.40).mean())))
BINS11D=pd.DataFrame(binned11d)
BINS11D.to_csv(AUDIT11D/'11D_MOS_by_mosaic_abundance_bin.csv',index=False)
print('MOS: by mosaic and observed Musanga abundance bin')
display(BINS11D.round(3))

# Verify saved ensemble averages and original random/single checkpoint for MOS.
# Only read 11B predictions if available, no model refitting.
FOLD_ROOT11D=OUT/'11_spatial_ablation_50m'/'fold_predictions'
parent11d=(DIR11D/'config.json')
if parent11d.exists():
    import json
    config11d=json.loads(parent11d.read_text())
    run_name11d=config11d.get('parent')
    ORIGINAL11D=FOLD_ROOT11D/run_name11d if run_name11d else None
else:
    ORIGINAL11D=None
checks11d=[]
for feature in F11D:
    first=mosaics11d[(feature,'random','single')]
    path=(ORIGINAL11D/f'LEAVE_FOREST_OUT_MOS_{feature}.parquet') if ORIGINAL11D else None
    if path is not None and path.exists():
        ref=pd.read_parquet(path).sort_values(KEYS11D).reset_index(drop=True)
        assert ref[KEYS11D].equals(first[KEYS11D])
        checks11d.append(dict(feature=feature,test='11B_vs_11C_random_single',
                              max_abs_diff_pp=float(100*abs(
                                  ref.predicted.to_numpy()-first.predicted.to_numpy()).max())))
    for mode in ('random','stratified'):
        member_files=[]
        for member in range(5):
            fn=(ORIGINAL11D/f'LEAVE_FOREST_OUT_MOS_{feature}.parquet'
                if mode=='random' and member==0 else
                DIR11D/f'LEAVE_FOREST_OUT_MOS_{feature}_{mode}_m{member}.parquet')
            assert fn is not None and fn.exists(),f'Missing ensemble member: {fn}'
            member_files.append(pd.read_parquet(fn).sort_values(KEYS11D).reset_index(drop=True))
        assert all(f[KEYS11D].equals(first[KEYS11D]) for f in member_files)
        mean_pred=np.stack([f.predicted.to_numpy(float) for f in member_files]).mean(axis=0)
        ensemble=mosaics11d[(feature,mode,'ensemble5')].predicted.to_numpy(float)
        checks11d.append(dict(feature=feature,test=mode+'_five_member_average',
                              max_abs_diff_pp=float(100*np.max(abs(mean_pred-ensemble)))))
CHECKS11D=pd.DataFrame(checks11d)
CHECKS11D.to_csv(AUDIT11D/'11D_cache_integrity_checks.csv',index=False)
display(CHECKS11D)
assert (CHECKS11D.max_abs_diff_pp<1e-5).all(),'11C aggregation does not match its constituent predictions!'
print('Checkpoint and ensemble-integrity checks passed.')

# Side-by-side held-out MOS plots; low-alpha thousands of points remain readable.
fig,axs=plt.subplots(2,3,figsize=(15,9),constrained_layout=True)
for ix,feature in enumerate(F11D):
    ax=axs[ix,0]
    for mode,variant in MODES11D:
        q=mosaics11d[(feature,mode,variant)]
        name=mode[:5]+'_'+variant
        ax.scatter(q.observed*100,q.predicted*100,s=2,alpha=.08,rasterized=True,label=name)
    ax.plot([0,100],[0,100],'--',color='grey',lw=1)
    ax.axvline(40,ls=':',color='grey',lw=1)
    ax.set(xlim=(0,100),ylim=(0,100),title=feature+' | held-out MOS',
           xlabel='Observed UAV Musanga (%)',ylabel='Predicted satellite (%)')
    ax.legend(fontsize=7,markerscale=3)
    ax=axs[ix,1]
    for mode,variant in MODES11D:
        q=mosaics11d[(feature,mode,variant)]
        b=pd.cut(q.observed,bins11d,labels=labels11d)
        means=q.groupby(b,observed=True).predicted.mean()
        ax.plot([5,15,30,50,70,90][:len(means)],means.values*100,
                marker='o',label=mode[:5]+'_'+variant)
    ax.plot([0,100],[0,100],'--',color='grey',lw=1)
    ax.set(xlim=(0,100),ylim=(0,100),title='Mean predicted by observed bin',
           xlabel='Observed Musanga class midpoint (%)',ylabel='Mean predicted (%)')
    ax.legend(fontsize=7)
    ax=axs[ix,2]
    for mode,variant in MODES11D:
        q=mosaics11d[(feature,mode,variant)]
        hi=q.loc[q.observed>=.4,'predicted']*100
        ax.hist(hi,bins=np.linspace(0,100,26),histtype='step',density=True,
                lw=1.3,label=mode[:5]+'_'+variant)
    ax.axvline(40,ls=':',color='grey',lw=1)
    ax.set(title='Predictions for observed >=40%',
           xlabel='Predicted Musanga (%)',ylabel='Probability density')
    ax.legend(fontsize=7)
fig.savefig(AUDIT11D/'11D_MOS_high_abundance_diagnostics.png',dpi=160)
plt.show();plt.close(fig)

# Make audit conclusion explicit without assuming explanation in advance.
print('INDEPENDENT AUDIT INTERPRETATION:')
for feature,part in S11D.groupby('feature'):
    spread=float(part.high_bias_pp.max()-part.high_bias_pp.min())
    pair=CHANGES11D[(CHANGES11D.feature==feature)&(CHANGES11D.subset=='HIGH_GE40')]
    biggest=float(pair.max_abs_delta_pp.max())
    print(feature,'high-cover bias spread (pp):',f'{spread:.12g}',
          '| largest per-cell prediction difference (pp):',f'{biggest:.12g}')
    if spread<1e-9 and biggest>1e-5:
        print('  Same MEAN high-cover bias but different predictions: inspect saturation / cancellation.')
    elif spread<1e-9 and biggest<=1e-5:
        print('  Nearly identical high-cover predictions: inspect saturation or shared predictor limits.')
    else:
        print('  High-cover bias genuinely differs; the previous displayed equality requires a reporting check.')
print('11D saved CSVs and figure to:',AUDIT11D)


## 12 · Native PlanetScope patch CNN — matched geographical validation

**12A (one-time, restartable):** extract *actual* 4.77 m Planet RGB+NIR patches (four native reflectance bands B,G,R,N, 32 × 32 pixels ≈153 m × 153 m) centered on all 14,454 accepted 50 m UAV reference cells. No 50 m Planet raster is produced and no UAV full mosaic is reopened. Patches have ~153 m spatial context around a **50 m fractional abundance target**; a patch is *not* a pixel-level Musanga map. Cache 200-patch batches per site on Drive with SHA-based source/target settings, and make subsequent runs skip verified complete chunks. Input output stores native 4-band image tensors as float16 and pixel-validity QA separately. Native 32×32 images are preserved, rather than reduced to means/SDs.

**12B (restartable GPU CNN):** supervised 4-channel CNN → one *50 m Musanga fraction*, with the **same exact 7 test folds and forest-balanced training-cell selection** as Block 11. Each of the three 400 m block tests has a 150 m label-center exclusion buffer; LOFO holds out each entire forest (including both MOS mosaics). Training-only per-band robust normalization; training-only flip/rotation augmentation. Fixed training epochs and architecture (no model selection based on the test forest). Separate checkpoint per fold; saved predictions are matched one-to-one to 11B HGB B0 and B1 held-out predictions. Report equal-forest R²/MAE, pooled R² and **MOS ≥40% high-abundance bias/recall**, plus per-site results and scatterplots.

**What this isolates:** does learning *native Planet spatial patterns* carry predictive signal without handcrafted texture features? The CNN uses Planet-only inputs, whereas HGB B1 uses S2, embeddings, and Planet means/SDs, so a direct superiority claim would conflate the data sources. Later test a **hybrid** model (CNN representation + S2/embeddings), trained and assessed on the same folds, if the Planet patch CNN looks promising.

**Efficiency:** 12A uses small raster windows and caches all native patches; 12B loads ~120 MB of patches, fits serially on GPU if available, prints per-epoch loss and writes each fold's predictions atomically. Use `MAX_NEW_BATCHES12A=2` for extraction smoke test, or `MAX_NEW_FOLDS12B=1` for training smoke test. No prior sections/models are retrained.

**Caveats:** patch neighborhoods can include satellite pixels across nearby fold boundaries (only imagery, not labels); the 150 m exclusion buffer is applied to labeled cell centers. Some Planet epochs and UAV acquisition years differ (particularly Yoko). Success is judged on *held-out UAV pseudo-label fractions*, not field-measured Musanga or individual-crown resolution.


In [ ]:
# 12A — Native 4-band Planet 32x32 patch extraction; per-site restartable.
from pathlib import Path
import os,json,hashlib,time,gc
import numpy as np,pandas as pd,rasterio
from rasterio.windows import Window
from pyproj import Transformer
from IPython.display import display
assert 'OUT' in globals() and 'SITES' in globals() and 'planet_file' in globals()
P12=32
CHUNK12=200
MAX_NEW_BATCHES12A=None  # Try 2 to check extraction; None finishes all.
OUT12=OUT/'12_native_planet_cnn'
CACHE12=OUT12/'native_patches'
CACHE12.mkdir(parents=True,exist_ok=True)
FILE12=OUT/'10_full_spatial_features_50m.parquet'
assert FILE12.exists(),FILE12
K12=['site','mosaic','row','col']
T12=pd.read_parquet(FILE12,columns=K12+['x','y','crs','target_fraction'])
T12=T12.sort_values(K12,kind='stable').reset_index(drop=True)
assert not T12.duplicated(K12).any()
assert len(T12)==14454
T12['global_index']=np.arange(len(T12),dtype=np.int32)
PLANET_PATH12={site:planet_file(site,'Annual') for site in SITES}
manifest12=dict(version='P12_32x32_BGRN_v1',size=P12,chunk=CHUNK12,bands=[13,14,15,16],
                target_digest=hashlib.sha256(pd.util.hash_pandas_object(
                    T12[K12+['x','y','crs','target_fraction']],index=False
                ).values.tobytes()).hexdigest(),
                sources={site:dict(path=str(path),size=path.stat().st_size,
                    mtime_ns=path.stat().st_mtime_ns) for site,path in PLANET_PATH12.items()})
id12=hashlib.sha256(json.dumps(manifest12,sort_keys=True).encode()).hexdigest()[:16]
RUN12=CACHE12/id12
RUN12.mkdir(parents=True,exist_ok=True)
(RUN12/'config.json').write_text(json.dumps(manifest12,indent=2,sort_keys=True))
print('Native Planet patches:',len(T12),'| cache:',RUN12,flush=True)
new12=0
for site in SITES:
    sub=T12[T12.site==site].reset_index(drop=True)
    site_dir=RUN12/site;site_dir.mkdir(exist_ok=True)
    # Each chunk is independent; a completed chunk is never overwritten.
    with rasterio.open(PLANET_PATH12[site]) as ds:
        assert ds.count>=16 and ds.crs and abs(ds.res[0]-4.77)<.5,(site,ds.res)
        transformer=Transformer.from_crs(sub.crs.iloc[0],ds.crs,always_xy=True)
        assert sub.crs.nunique()==1,(site,'mixed CRS')
        for off in range(0,len(sub),CHUNK12):
            part=sub.iloc[off:off+CHUNK12]
            dest=site_dir/f'patches_{off:06d}_{len(part):03d}.npz'
            if dest.exists():
                try:
                    with np.load(dest) as pack:
                        assert pack['patches'].shape==(len(part),4,P12,P12)
                        assert np.array_equal(pack['index'],part.global_index.to_numpy())
                        assert pack['valid'].shape==(len(part),)
                    print('CACHED',site,dest.name,flush=True)
                    continue
                except Exception as exc:
                    print('REBUILD incomplete patch chunk',dest.name,repr(exc)[:120],flush=True)
            if MAX_NEW_BATCHES12A is not None and new12>=MAX_NEW_BATCHES12A:
                print('Reached extraction batch limit; rerun 12A to continue.',flush=True)
                break
            started=time.monotonic()
            patches=np.zeros((len(part),4,P12,P12),dtype=np.float16)
            valid=np.zeros(len(part),dtype=np.float32)
            for j,rec in enumerate(part.itertuples(index=False)):
                xx,yy=transformer.transform(rec.x,rec.y)
                cc,rr=(~ds.transform)*(xx,yy)
                c=int(np.floor(cc));r=int(np.floor(rr))
                data=ds.read([13,14,15,16],window=Window(
                    c-P12//2,r-P12//2,P12,P12),boundless=True,fill_value=0,
                    out_dtype='float32')
                good=np.isfinite(data).all(axis=0)&(data>0).all(axis=0)
                valid[j]=float(good.mean())
                data[:,~good]=0
                patches[j]=np.clip(data,-65000,65000).astype(np.float16)
            tmp=dest.with_suffix('.npz.partial')
            # Write file object explicitly to avoid np.savez adding '.npz' suffix.
            with open(tmp,'wb') as stream:
                np.savez_compressed(stream,patches=patches,
                                    index=part.global_index.to_numpy(dtype='int32'),
                                    valid=valid)
            os.replace(tmp,dest)
            new12+=1
            print('SAVED',site,dest.name,'valid>=80%',int((valid>=.80).sum()),
                  '/',len(valid),'seconds',round(time.monotonic()-started,1),flush=True)
        if MAX_NEW_BATCHES12A is not None and new12>=MAX_NEW_BATCHES12A:
            break
# Consolidation only after ALL expected site chunks are present.
missing12=[]
for site in SITES:
    sub=T12[T12.site==site]
    for off in range(0,len(sub),CHUNK12):
        part=sub.iloc[off:off+CHUNK12]
        path=RUN12/site/f'patches_{off:06d}_{len(part):03d}.npz'
        if not path.exists():missing12.append(str(path))
print('12A chunks remaining:',len(missing12),flush=True)
if not missing12:
    # One compact, sequentially ordered tensor for fast training.
    # float16 array is 118 MB; store as .npy for memory mapping.
    tensor_path=RUN12/'planet_native_32px_float16.npy'
    validity_path=RUN12/'patch_validity.parquet'
    from numpy.lib.format import open_memmap
    X12disk=open_memmap(tensor_path,mode='w+',dtype='float16',
                        shape=(len(T12),4,P12,P12))
    valid_all=np.full(len(T12),np.nan,dtype='float32')
    for site in SITES:
        sub=T12[T12.site==site].reset_index(drop=True)
        for off in range(0,len(sub),CHUNK12):
            part=sub.iloc[off:off+CHUNK12]
            path=RUN12/site/f'patches_{off:06d}_{len(part):03d}.npz'
            with np.load(path) as pack:
                indices=pack['index']
                assert np.array_equal(indices,part.global_index.to_numpy())
                X12disk[indices]=pack['patches']
                valid_all[indices]=pack['valid']
    X12disk.flush();del X12disk
    assert np.isfinite(valid_all).all()
    QA12=T12[K12+['global_index','target_fraction']].copy()
    QA12['planet_native_valid']=valid_all
    QA12.to_parquet(validity_path,index=False)
    display(QA12.assign(good=QA12.planet_native_valid>=.8).groupby('site').agg(
        total=('good','size'),valid=('good','sum'),
        mean_valid=('planet_native_valid','mean')).round(3))
    print('12A COMPLETE:',tensor_path,'| QA:',validity_path,flush=True)
else:
    print('12A incomplete; rerun this block (saved chunks are reused).',flush=True)


In [ ]:
# 12B — Native Planet RGB+NIR CNN, matched seven-fold geographic validation.
# Requires 11A/11B initialized in this runtime; 12A patches must be complete.
import os,json,hashlib,time,random,shutil
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
import torch
from torch import nn
from torch.utils.data import DataLoader,Dataset

SEED12B=20261024
EPOCHS12B=22
BATCH12B=96
LR12B=0.001
MAX_NEW_FOLDS12B=None # Set 1 to smoke-test, None to finish all seven.
NUM_WORKERS12B=0
DEVICE12B=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert 'D11' in globals() and 'split11' in globals() and 'RUN11' in globals()
# Standalone protection for Colab cells: 11B's old split11 referred to a mutable
# global "xx"; notebook plotting code can replace it with a scalar.
# Define the exact same split logic using coordinate arrays local to this function.
from scipy.spatial import cKDTree
def split12b(scheme,heldout):
    sites=D11.site.to_numpy()
    xcoords=D11.x.to_numpy(dtype=float)
    ycoords=D11.y.to_numpy(dtype=float)
    test=(D11.spatial_fold.to_numpy()==int(heldout)) if scheme=='BLOCK_BUFFERED' else (sites==heldout)
    train=~test
    if scheme=='BLOCK_BUFFERED':
        for site in SITES:
            ti=np.where(test & (sites==site))[0]
            ix=np.where(train & (sites==site))[0]
            if len(ti)==0 or len(ix)==0:continue
            tree=cKDTree(np.column_stack((xcoords[ti],ycoords[ti])))
            nearby=tree.query(np.column_stack((xcoords[ix],ycoords[ix])),k=1)[0]<BUFFER_M
            train[ix[nearby]]=False
    rng=np.random.default_rng(MODEL_SEED)
    train_ids=[]
    for site in SITES:
        eligible=np.where(train & (sites==site))[0]
        if len(eligible)>TRAIN_CAP_PER_FOREST:
            eligible=rng.choice(eligible,TRAIN_CAP_PER_FOREST,replace=False)
        train_ids.extend(eligible.tolist())
    train_ids=np.asarray(sorted(train_ids),dtype=int)
    test_ids=np.where(test)[0]
    assert len(train_ids)>=100 and len(test_ids)>=10
    assert not np.intersect1d(train_ids,test_ids).size
    return train_ids,test_ids,int((~train & ~test).sum())
# Check exact held-out rows against archived 11B HGB files before any CNN fitting.
for scheme,hold in folds11:
    train_ids,hold_ids,excluded=split12b(scheme,hold)
    ref_file=RUN11/f'{scheme}_{hold}_B0_S2_EMB.parquet'
    assert ref_file.is_file(),('Missing 11B original HGB predictions',ref_file)
    saved=pd.read_parquet(ref_file,columns=['site','mosaic','row','col'])
    assert saved.reset_index(drop=True).equals(
        D11.iloc[hold_ids][['site','mosaic','row','col']].reset_index(drop=True)),(
        '12B holdout does not match original Block 11',scheme,hold)
print('12B fold safeguard: all seven test sets match archived 11B exactly.',flush=True)
assert 'RUN12' in globals() and 'T12' in globals(),'Run 12A in this runtime first.'
PATCH_FILE12=RUN12/'planet_native_32px_float16.npy'
QA_FILE12=RUN12/'patch_validity.parquet'
assert PATCH_FILE12.exists() and QA_FILE12.exists(),'Finish all 12A extraction chunks first.'
Q12=pd.read_parquet(QA_FILE12)
K12=['site','mosaic','row','col']
assert len(Q12)==len(D11) and Q12[K12].equals(D11[K12]),'12A and 11B target rows differ'
assert Q12.global_index.to_numpy().tolist()==list(range(len(Q12)))
LOCAL12=Path('/content/musanga_nb06_native_cache')
LOCAL12.mkdir(parents=True,exist_ok=True)
LOCAL_PATCH12=LOCAL12/(RUN12.name+'_'+PATCH_FILE12.name)
if not LOCAL_PATCH12.exists() or LOCAL_PATCH12.stat().st_size!=PATCH_FILE12.stat().st_size:
    print('Copying 12A compact native patches from Drive to Colab SSD (once/session)...',flush=True)
    shutil.copy2(PATCH_FILE12,LOCAL_PATCH12)
X12=np.load(LOCAL_PATCH12,mmap_mode='r')
assert X12.shape==(len(D11),4,32,32),X12.shape
good12=(Q12.planet_native_valid.to_numpy(float)>=.80)
print('CNN patch coverage:',int(good12.sum()),'/',len(good12),
      '| computing device:',DEVICE12B,flush=True)
assert good12.mean()>.90,'Unexpected number of invalid native patches; inspect QA.'
# For rigor against 11B, require same prediction cells and fail rather than
# silently dropping any test examples. If failed, diagnose/rebuild patch validity.
assert good12.all(),'At least one Planet patch below 80%; cannot compare to full 11B folds as-is.'

torch.set_num_threads(2)
def set_seed12b(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

class PlanetPatches12(Dataset):
    def __init__(self,indices,means,scales,augment=False):
        self.ids=np.asarray(indices,dtype=np.int64)
        self.mu=np.asarray(means,dtype=np.float32).reshape(4,1,1)
        self.scale=np.asarray(scales,dtype=np.float32).reshape(4,1,1)
        self.augment=augment
    def __len__(self):return len(self.ids)
    def __getitem__(self,i):
        k=int(self.ids[i])
        patch=np.asarray(X12[k],dtype=np.float32)
        valid=np.all(patch>0,axis=0)
        patch=np.clip((patch-self.mu)/self.scale,-3,3)
        patch[:,~valid]=0.
        if self.augment:
            if np.random.rand()<.5:patch=patch[:,:,::-1]
            if np.random.rand()<.5:patch=patch[:,::-1,:]
            patch=np.rot90(patch,np.random.randint(0,4),axes=(1,2))
        return torch.from_numpy(np.ascontiguousarray(patch)),torch.tensor(
            float(D11.target_fraction.iloc[k]),dtype=torch.float32)

class PlanetCNN12(nn.Module):
    def __init__(self):
        super().__init__()
        def stack(a,b):
            return nn.Sequential(nn.Conv2d(a,b,3,padding=1),
                nn.BatchNorm2d(b),nn.ReLU(inplace=True),
                nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),
                nn.ReLU(inplace=True),nn.MaxPool2d(2))
        self.encoder=nn.Sequential(stack(4,24),stack(24,48),stack(48,64),
                                    stack(64,96),nn.AdaptiveAvgPool2d(1))
        self.head=nn.Sequential(nn.Flatten(),nn.Linear(96,48),nn.ReLU(),
                                 nn.Dropout(.15),nn.Linear(48,1),nn.Sigmoid())
    def forward(self,x):return self.head(self.encoder(x)).flatten()

def train_normalization12(train_ids):
    # Per-band robust statistics ONLY from allowed training observations.
    # Take a reproducible pixel subsample; exclude padding or nodata.
    sample=np.asarray(X12[train_ids[::max(1,len(train_ids)//1000)]],dtype='float32')
    mu=[];scale=[]
    for b in range(4):
        z=sample[:,b].reshape(-1)
        z=z[np.isfinite(z)&(z>0)]
        assert len(z)>1000,('Empty Planet band',b)
        p10,p50,p90=np.percentile(z,[10,50,90])
        mu.append(float(p50));scale.append(float(max((p90-p10)/2,1e-3)))
    return mu,scale

conf12b=dict(version='P12_CNN_v1',parent=RUN11.name,patch_run=RUN12.name,
             epochs=EPOCHS12B,batch=BATCH12B,lr=LR12B,seed=SEED12B,
             architecture='4ch_CNN_24_48_64_96',train_selection='exact_11B',
             normalization='per_fold_train_only_10_50_90')
runid12b=hashlib.sha256(json.dumps(conf12b,sort_keys=True).encode()).hexdigest()[:16]
RUN12B=OUT12/'cnn_fold_predictions'/runid12b
RUN12B.mkdir(parents=True,exist_ok=True)
(RUN12B/'config.json').write_text(json.dumps(conf12b,indent=2))
print('CNN fold results:',RUN12B,'| source=',RUN12,flush=True)

def fit_one12(scheme,hold):
    train_ids,test_ids,_=split12b(scheme,hold)
    assert good12[train_ids].all() and good12[test_ids].all()
    means,scales=train_normalization12(train_ids)
    set_seed12b(SEED12B+sum(ord(x) for x in str(hold))+
                (0 if scheme=='BLOCK_BUFFERED' else 100))
    tr=DataLoader(PlanetPatches12(train_ids,means,scales,True),
                  batch_size=BATCH12B,shuffle=True,num_workers=NUM_WORKERS12B,
                  pin_memory=DEVICE12B.type=='cuda')
    te=DataLoader(PlanetPatches12(test_ids,means,scales,False),
                  batch_size=BATCH12B,shuffle=False,num_workers=NUM_WORKERS12B,
                  pin_memory=DEVICE12B.type=='cuda')
    model=PlanetCNN12().to(DEVICE12B)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR12B,weight_decay=1e-3)
    lossfn=nn.MSELoss()
    progress=[]
    for ep in range(EPOCHS12B):
        model.train();losses=[];count=0
        for x,y in tr:
            x=x.to(DEVICE12B,non_blocking=True)
            y=y.to(DEVICE12B,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            out=model(x)
            loss=lossfn(out,y)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.item())*len(y));count+=len(y)
        avg=sum(losses)/count
        progress.append(dict(epoch=ep+1,train_mse=avg))
        print(f'  {scheme} {hold} epoch {ep+1:02d}/{EPOCHS12B} '
              f'train MSE={avg:.5f}',flush=True)
    model.eval();preds=[]
    with torch.inference_mode():
        for x,_ in te:
            pred=model(x.to(DEVICE12B,non_blocking=True)).cpu().numpy()
            preds.append(pred)
    preds=np.clip(np.concatenate(preds),0,1)
    assert len(preds)==len(test_ids)
    return test_ids,preds,progress,means,scales

new12b=0
for scheme,hold in folds11:
    name=f'{scheme}_{hold}_PlanetCNN.parquet'
    dest=RUN12B/name
    _,test_ids,_=split12b(scheme,hold)
    expected=D11.iloc[test_ids][K12].reset_index(drop=True)
    if dest.exists():
        try:
            old=pd.read_parquet(dest)
            if (len(old)==len(test_ids) and old[K12].equals(expected) and
                np.isfinite(old.predicted).all()):
                print('CACHED CNN',name,flush=True)
                continue
        except Exception as ex:
            print('Retry partial fold:',name,repr(ex)[:100],flush=True)
    if MAX_NEW_FOLDS12B is not None and new12b>=MAX_NEW_FOLDS12B:
        print('Reached fold limit; rerun 12B to continue.',flush=True)
        break
    started=time.monotonic()
    ids,pred,losslog,mu,scale=fit_one12(scheme,hold)
    saved=D11.iloc[ids][K12+['target_fraction']].reset_index(drop=True)
    saved=saved.rename(columns={'target_fraction':'observed'})
    saved['predicted']=pred
    tmp=dest.with_suffix('.parquet.partial')
    saved.to_parquet(tmp,index=False);os.replace(tmp,dest)
    pd.DataFrame(losslog).to_csv(RUN12B/f'{scheme}_{hold}_training_log.csv',index=False)
    (RUN12B/f'{scheme}_{hold}_normalization.json').write_text(
        json.dumps(dict(median=mu,robust_scale=scale),indent=2))
    new12b+=1
    print('SAVED CNN held-out fold:',name,'seconds',round(time.monotonic()-started,1),flush=True)

missing12b=[(scheme,hold) for scheme,hold in folds11
    if not (RUN12B/f'{scheme}_{hold}_PlanetCNN.parquet').exists()]
if missing12b:
    print('CNN evaluation partial:',len(missing12b),'folds remaining.',flush=True)
else:
    # Match the exact same held-out UAV observations across three predictors.
    predictions=[]
    for scheme,hold in folds11:
        c=pd.read_parquet(RUN12B/f'{scheme}_{hold}_PlanetCNN.parquet')
        for model,label in [('PlanetCNN','CNN'),
                            ('B0_S2_EMB','HGB_B0'),('B1_B0_PlanetSpectral','HGB_B1')]:
            q=(c.copy() if model=='PlanetCNN' else pd.read_parquet(
                RUN11/f'{scheme}_{hold}_{model}.parquet'))
            assert q[K12].equals(c[K12]) and np.allclose(q.observed,c.observed)
            q['scheme']=scheme;q['holdout']=hold;q['model']=label
            predictions.append(q)
    P12B=pd.concat(predictions,ignore_index=True)
    P12B.to_parquet(RUN12B/'all_heldout_predictions.parquet',index=False)
    reports=[];site_reports=[]
    for (scheme,model),g in P12B.groupby(['scheme','model']):
        assert not g.duplicated(K12).any()
        per_site=[]
        for site,sub in g.groupby('site'):
            rec=dict(scheme=scheme,model=model,site=site,
                     **metric11(sub.observed,sub.predicted))
            per_site.append(rec);site_reports.append(rec)
        counts=g.groupby('site').size()
        weights=1/g.site.map(counts).to_numpy(float);weights/=weights.sum()
        y=g.observed.to_numpy(float);p=g.predicted.to_numpy(float)
        centre=np.dot(weights,y);denom=np.dot(weights,(y-centre)**2)
        mos=g[g.site=='MOS'];hi=mos.observed>=.4
        reports.append(dict(scheme=scheme,model=model,n=len(g),
            pooled_R2=metric11(y,p)['R2'],
            equal_forest_R2=1-np.dot(weights,(y-p)**2)/denom,
            equal_forest_MAE_pp=np.mean([row['MAE_pp'] for row in per_site]),
            mean_site_R2=np.nanmean([row['R2'] for row in per_site]),
            MOS_high_n=int(hi.sum()),MOS_high_bias_pp=float(
                100*(mos.predicted[hi]-mos.observed[hi]).mean()) if hi.any() else np.nan,
            MOS_high_recall=float((mos.predicted[hi]>=.4).mean()) if hi.any() else np.nan))
    summary12b=pd.DataFrame(reports)
    sites12b=pd.DataFrame(site_reports)
    summary12b.to_csv(RUN12B/'global_metrics.csv',index=False)
    sites12b.to_csv(RUN12B/'site_metrics.csv',index=False)
    for scheme in ('BLOCK_BUFFERED','LEAVE_FOREST_OUT'):
        print('12B:',scheme)
        display(summary12b[summary12b.scheme==scheme].sort_values(
            'equal_forest_MAE_pp').round(3))
    fig,axes=plt.subplots(1,4,figsize=(16,4),constrained_layout=True)
    for ax,site in zip(axes,SITES):
        subset=P12B[(P12B.scheme=='LEAVE_FOREST_OUT')&(P12B.site==site)]
        for model,style in [('CNN','native Planet CNN'),('HGB_B1','HGB B1')]:
            d=subset[subset.model==model]
            ax.scatter(100*d.observed,100*d.predicted,s=3,alpha=.14,
                       label=style,rasterized=True)
        ax.plot([0,100],[0,100],'--',color='grey',lw=1)
        ax.set(xlim=(0,100),ylim=(0,100),title=site,
               xlabel='UAV Musanga (%)',ylabel='Held-out prediction (%)')
        ax.legend(fontsize=8)
    fig.suptitle('Strict leave-one-forest-out | Planet CNN versus HGB B1')
    fig.savefig(RUN12B/'LOFO_CNN_vs_HGB_B1_scatter.png',dpi=160)
    plt.show();plt.close(fig)
    print('12B COMPLETE:',RUN12B,flush=True)


## 12C · Hybrid Planet patch CNN + Sentinel-2 seasonal + embeddings

**Purpose:** the Planet-only CNN in 12B transferred poorly (equal-forest R² −0.068); test whether *learning fine-scale Planet spatial structure **together with*** the strong B0 satellite predictors improves held-out 50 m Musanga fractions. The **same 32×32 native Planet RGB-NIR patches** and **all 123 B0 features (annual S2, 3-year quarterly S2, 64 annual embeddings)** are used.

A **two-branch fusion network** learns Planet spatial patterns from the 4-band CNN encoder and learns baseline predictors from a compact MLP; concatenated representations predict a **single 50 m fraction**. This is *not* a Planet U-Net and not pixel-level segmentation. The architecture is trained from scratch because 12B stored predictions, not transferable encoder weights.

**Validation:** use exactly the seven frozen 11B/12B geographic test folds; forest-balanced training cells, 400 m spatial blocks with 150 m buffer, and strict LOFO holding out both MOS mosaics as a forest. All patch and tabular scaling parameters are derived exclusively from each fold's training observations; training-only flips/rotations are used. No test-dependent model tuning. Report equal-forest MAE/R², pooled R², site-level R², MOS high-cover bias and recall, paired to **HGB B0, HGB B1 and Planet-only CNN** on identical test cells.

**Speed and restartability:** GPU recommended; one fold is ~minutes or less, depending on hardware. Predictions and training curves are saved independently per fold with a fingerprinted run directory. No new patch extraction or HGB fits. Set `MAX_NEW_FOLDS12C=1` for a smoke test, `None` for all seven. **Prerequisites:** setup (00), input-function definition (01), 11A, 11B (cached), 12A (cached), 12B (cached) in that order; then 12C. This is an exploratory architecture comparison; absence of an independent tuning forest limits interpretation of architecture choice.

In [ ]:
# 12C — Hybrid 4-band native Planet CNN + 123 S2/embedding predictors.
# No feature extraction. Reuses 11B/12B held-out keys and 12A patch tensor.
import os,json,hashlib,time,random
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
import torch
from torch import nn
from torch.utils.data import DataLoader,Dataset
assert all(k in globals() for k in ('D11','RUN11','RUN12','RUN12B','X12',
                                    'split12b','FEATURE_SETS11','metric11','folds11'))
SEED12C=20261024
EPOCHS12C=30
BATCH12C=96
LR12C=0.001
MAX_NEW_FOLDS12C=1   # First run: 1; set None after inspection.
DEVICE12C=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
C12=FEATURE_SETS11['B0_S2_EMB']
assert len(C12)==123,len(C12)
K12C=['site','mosaic','row','col']
assert D11[K12C].equals(T12[K12C])
TAB12C=D11[C12].to_numpy(dtype=np.float32)
assert np.isfinite(TAB12C).all()
Y12C=D11.target_fraction.to_numpy(dtype=np.float32)
assert X12.shape==(len(D11),4,32,32)
assert good12.all(),'CNN comparison requires all original 50 m cells.'
torch.set_num_threads(2)

def seed12c(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

class HybridCells12C(Dataset):
    def __init__(self,indices,p_mu,p_sc,t_mu,t_sc,augment=False):
        self.indices=np.asarray(indices,dtype=np.int64)
        self.pmu=np.asarray(p_mu,np.float32).reshape(4,1,1)
        self.psc=np.asarray(p_sc,np.float32).reshape(4,1,1)
        self.tmu=np.asarray(t_mu,np.float32)
        self.tsc=np.asarray(t_sc,np.float32)
        self.augment=augment
    def __len__(self):return len(self.indices)
    def __getitem__(self,i):
        idx=int(self.indices[i])
        raw=np.asarray(X12[idx],dtype=np.float32)
        valid=np.all(np.isfinite(raw)&(raw>0),axis=0)
        patch=np.clip((raw-self.pmu)/self.psc,-3,3)
        patch[:,~valid]=0
        if self.augment:
            if np.random.random()<.5:patch=patch[:,:,::-1]
            if np.random.random()<.5:patch=patch[:,::-1,:]
            patch=np.rot90(patch,np.random.randint(4),axes=(1,2))
        tab=np.clip((TAB12C[idx]-self.tmu)/self.tsc,-5,5)
        return (torch.from_numpy(np.ascontiguousarray(patch)),
                torch.from_numpy(np.ascontiguousarray(tab,dtype=np.float32)),
                torch.tensor(Y12C[idx],dtype=torch.float32))

class HybridPlanet12C(nn.Module):
    def __init__(self,n_tab):
        super().__init__()
        def conv(a,b):
            return nn.Sequential(nn.Conv2d(a,b,3,padding=1),
                nn.BatchNorm2d(b),nn.ReLU(inplace=True),
                nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),
                nn.ReLU(inplace=True),nn.MaxPool2d(2))
        self.planet=nn.Sequential(conv(4,24),conv(24,48),conv(48,64),
                                  conv(64,96),nn.AdaptiveAvgPool2d(1),
                                  nn.Flatten(),nn.Linear(96,48),nn.ReLU())
        self.satellite=nn.Sequential(nn.Linear(n_tab,96),nn.LayerNorm(96),
                                     nn.ReLU(),nn.Dropout(.10),
                                     nn.Linear(96,48),nn.ReLU())
        self.head=nn.Sequential(nn.Linear(96,48),nn.ReLU(),nn.Dropout(.15),
                                nn.Linear(48,1),nn.Sigmoid())
    def forward(self,planet,satellite):
        return self.head(torch.cat((self.planet(planet),
                                    self.satellite(satellite)),dim=1)).flatten()

def norm12c(train):
    # Training-only robust Planet band normalization; valid nonzero pixels.
    sample=np.asarray(X12[train[::max(1,len(train)//1000)]],dtype=np.float32)
    med=[];scale=[]
    for band in range(4):
        values=sample[:,band].reshape(-1)
        values=values[np.isfinite(values)&(values>0)]
        assert len(values)>1000
        q10,q50,q90=np.percentile(values,[10,50,90])
        med.append(float(q50));scale.append(float(max((q90-q10)/2,1e-3)))
    # Robust tabular scaling; zero-spread bands safely use fallback unit scale.
    t=TAB12C[train].astype(np.float64)
    q10=np.percentile(t,10,axis=0)
    q50=np.percentile(t,50,axis=0)
    q90=np.percentile(t,90,axis=0)
    spread=(q90-q10)/2
    spread=np.where(spread>1e-6,spread,1.)
    return (np.asarray(med,np.float32),np.asarray(scale,np.float32),
            q50.astype(np.float32),spread.astype(np.float32))

CONF12C=dict(version='hybrid_12C_v1',parent_hgb=RUN11.name,
             parent_planet_cnn=RUN12B.name,patches=RUN12.name,
             n_tab=len(C12),feature_names=C12,epochs=EPOCHS12C,
             batch=BATCH12C,learning_rate=LR12C,seed=SEED12C,
             architecture='cnn_24_48_64_96_plus_mlp96_48_fuse96',
             validation='same_seven_folds_11B',train_cap=TRAIN_CAP_PER_FOREST,
             normalization='training_only_robust',augmentation='flips_rotate90')
HASH12C=hashlib.sha256(json.dumps(CONF12C,sort_keys=True).encode()).hexdigest()[:16]
DIR12C=OUT12/'hybrid_fold_predictions'/HASH12C
DIR12C.mkdir(parents=True,exist_ok=True)
(DIR12C/'config.json').write_text(json.dumps(CONF12C,indent=2))
print('12C hybrid output:',DIR12C,'| device:',DEVICE12C,flush=True)

# Verify all seven folds against both original HGB and Planet-only CNN.
for scheme,hold in folds11:
    tr,te,_=split12b(scheme,hold)
    ref=D11.iloc[te][K12C].reset_index(drop=True)
    for source in [RUN11/f'{scheme}_{hold}_B0_S2_EMB.parquet',
                   RUN12B/f'{scheme}_{hold}_PlanetCNN.parquet']:
        assert source.exists(),('Missing benchmark',source)
        old=pd.read_parquet(source,columns=K12C)
        assert old.reset_index(drop=True).equals(ref),('Mismatch',source)
print('12C fold safeguard: all seven training/test partitions and test keys verified.',flush=True)

def fit12c(scheme,hold):
    train,test,_=split12b(scheme,hold)
    p_mu,p_sc,t_mu,t_sc=norm12c(train)
    seed12c(SEED12C+sum(map(ord,str(hold)))+
            (0 if scheme=='BLOCK_BUFFERED' else 100))
    train_loader=DataLoader(HybridCells12C(train,p_mu,p_sc,t_mu,t_sc,True),
        batch_size=BATCH12C,shuffle=True,num_workers=0,pin_memory=DEVICE12C.type=='cuda')
    test_loader=DataLoader(HybridCells12C(test,p_mu,p_sc,t_mu,t_sc,False),
        batch_size=BATCH12C,shuffle=False,num_workers=0,pin_memory=DEVICE12C.type=='cuda')
    model=HybridPlanet12C(len(C12)).to(DEVICE12C)
    opt=torch.optim.AdamW(model.parameters(),lr=LR12C,weight_decay=1e-3)
    lossfunc=nn.MSELoss()
    curves=[]
    for epoch in range(EPOCHS12C):
        model.train();total=0.;count=0
        for patch,tab,target in train_loader:
            patch=patch.to(DEVICE12C,non_blocking=True)
            tab=tab.to(DEVICE12C,non_blocking=True)
            target=target.to(DEVICE12C,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            pred=model(patch,tab)
            loss=lossfunc(pred,target)
            loss.backward()
            opt.step()
            total+=float(loss.item())*len(target);count+=len(target)
        mse=total/count
        curves.append(dict(epoch=epoch+1,train_mse=mse))
        print(f'  HYBRID {scheme} {hold} epoch {epoch+1:02d}/{EPOCHS12C}'
              f' train MSE={mse:.5f}',flush=True)
    model.eval();preds=[]
    with torch.inference_mode():
        for patch,tab,_ in test_loader:
            pred=model(patch.to(DEVICE12C),tab.to(DEVICE12C))
            preds.append(pred.cpu().numpy())
    result=np.clip(np.concatenate(preds),0,1)
    assert len(result)==len(test)
    return test,result,curves

new12c=0
for scheme,hold in folds11:
    output=DIR12C/f'{scheme}_{hold}_HybridCNN.parquet'
    train,test,_=split12b(scheme,hold)
    ref=D11.iloc[test][K12C].reset_index(drop=True)
    if output.exists():
        try:
            existing=pd.read_parquet(output)
            if (len(existing)==len(test) and existing[K12C].equals(ref)
                and np.isfinite(existing.predicted).all()):
                print('CACHED HYBRID',scheme,hold,flush=True)
                continue
        except Exception as err:
            print('Recomputing incomplete fold:',repr(err)[:100],flush=True)
    if MAX_NEW_FOLDS12C is not None and new12c>=MAX_NEW_FOLDS12C:
        print('Reached pilot fold limit; rerun 12C to resume.',flush=True)
        break
    tick=time.monotonic()
    idx,pred,curve=fit12c(scheme,hold)
    frame=D11.iloc[idx][K12C+['target_fraction']].reset_index(drop=True)
    frame=frame.rename(columns={'target_fraction':'observed'})
    frame['predicted']=pred
    tmp=output.with_suffix('.parquet.partial')
    frame.to_parquet(tmp,index=False);os.replace(tmp,output)
    pd.DataFrame(curve).to_csv(DIR12C/f'{scheme}_{hold}_train_curve.csv',index=False)
    new12c+=1
    print('SAVED hybrid',scheme,hold,'seconds',round(time.monotonic()-tick,1),flush=True)

remaining12c=[(scheme,hold) for scheme,hold in folds11
    if not (DIR12C/f'{scheme}_{hold}_HybridCNN.parquet').exists()]
print('12C fold completion:',len(folds11)-len(remaining12c),'/',len(folds11),flush=True)
if remaining12c:
    print('PARTIAL: change MAX_NEW_FOLDS12C=None and rerun 12C.',flush=True)
else:
    records=[];details=[];allpred=[]
    for scheme,hold in folds11:
        fn=f'{scheme}_{hold}'
        hybrid=pd.read_parquet(DIR12C/f'{fn}_HybridCNN.parquet')
        comparison=[('HybridCNN',hybrid),
            ('HGB_B0',pd.read_parquet(RUN11/f'{fn}_B0_S2_EMB.parquet')),
            ('HGB_B1',pd.read_parquet(RUN11/f'{fn}_B1_B0_PlanetSpectral.parquet')),
            ('PlanetCNN',pd.read_parquet(RUN12B/f'{fn}_PlanetCNN.parquet'))]
        for label,part in comparison:
            assert part[K12C].equals(hybrid[K12C])
            assert np.allclose(part.observed,hybrid.observed)
            q=part[K12C+['observed','predicted']].copy()
            q['scheme']=scheme;q['holdout']=hold;q['model']=label
            allpred.append(q)
    PP12C=pd.concat(allpred,ignore_index=True)
    PP12C.to_parquet(DIR12C/'matched_heldout_predictions.parquet',index=False)
    for (scheme,label),g in PP12C.groupby(['scheme','model']):
        assert not g.duplicated(K12C).any()
        site_metrics=[]
        for site,sub in g.groupby('site'):
            rec=dict(scheme=scheme,model=label,site=site,
                     **metric11(sub.observed,sub.predicted))
            details.append(rec);site_metrics.append(rec)
        counts=g.groupby('site').size()
        w=1/g.site.map(counts).to_numpy(float);w/=w.sum()
        y=g.observed.to_numpy(float);pred=g.predicted.to_numpy(float)
        centre=np.dot(w,y);denom=np.dot(w,(y-centre)**2)
        mos=g[g.site=='MOS'];high=mos.observed>=.4
        records.append(dict(scheme=scheme,model=label,n=len(g),
            pooled_R2=metric11(y,pred)['R2'],
            equal_forest_R2=1-np.dot(w,(y-pred)**2)/denom,
            mean_site_R2=np.nanmean([x['R2'] for x in site_metrics]),
            equal_forest_MAE_pp=np.mean([x['MAE_pp'] for x in site_metrics]),
            MOS_high_n=int(high.sum()),
            MOS_high_bias_pp=float(100*(mos.predicted[high]-mos.observed[high]).mean()),
            MOS_high_recall=float((mos.predicted[high]>=.40).mean())))
    metrics12c=pd.DataFrame(records)
    site_metrics12c=pd.DataFrame(details)
    metrics12c.to_csv(DIR12C/'global_metrics.csv',index=False)
    site_metrics12c.to_csv(DIR12C/'site_metrics.csv',index=False)
    for scheme in ('BLOCK_BUFFERED','LEAVE_FOREST_OUT'):
        print('12C RESULT:',scheme,flush=True)
        display(metrics12c[metrics12c.scheme==scheme].sort_values('equal_forest_MAE_pp').round(3))
    print('12C strict LOSO individual forests:')
    display(site_metrics12c[site_metrics12c.scheme=='LEAVE_FOREST_OUT'].round(3))
    fig,axes=plt.subplots(1,4,figsize=(16,4),layout='constrained')
    for ax,site in zip(axes,SITES):
        q=PP12C[(PP12C.scheme=='LEAVE_FOREST_OUT')&(PP12C.site==site)]
        for label in ('HybridCNN','HGB_B1'):
            part=q[q.model==label]
            ax.scatter(part.observed*100,part.predicted*100,s=2,alpha=.12,
                       rasterized=True,label=label)
        ax.plot([0,100],[0,100],ls='--',color='grey',lw=1)
        ax.set(xlim=(0,100),ylim=(0,100),title=site,
               xlabel='UAV probability fraction (%)',ylabel='Prediction (%)')
        ax.legend(fontsize=8)
    fig.suptitle('Strict forest holdout: Hybrid CNN vs HGB B1')
    fig.savefig(DIR12C/'LOFO_hybrid_vs_HGB_B1_scatter.png',dpi=160)
    plt.show();plt.close(fig)
    print('12C COMPLETE:',DIR12C,flush=True)


## 13 · Dense PlanetScope U-Net — stage A: pixel-level teacher alignment and safety audit

**Scientific objective:** native PlanetScope RGB+NIR → dense Musanga fractional *probability per ~4.77 m satellite pixel*, supervised by the existing UAV U-Net teacher probability maps. This differs fundamentally from Blocks 11–12 (one prediction per 50 m cell). The 50 m dataset stays an independent comparison benchmark, not the training target for this stage.

**Stage 13A (this block):** before expensive dataset generation or model training, inspect the five authoritative full-resolution UAV probability rasters using **small, geographically aligned PlanetScope windows**. Reproject only each window to the Planet pixel lattice, audit the original UAV uint16 probability encoding against independently cached 10 m teacher probabilities, check valid-data coverage, and show RGB/teacher comparison panels. Use validated U-Net teacher information rather than assuming that raw uint16 values represent probabilities. Keep the MOS mosaics separate and preserve their footprints. A failed encoding or alignment audit **stops the pipeline**; no automatic silent clipping or normalization.

**Next stages, only after we inspect 13A outputs:** (13B) construct spatially disjoint 4.77 m Planet/UAV-supervised tiles, recording valid masks, original geography, mosaic boundaries, and train/test blocks, with streamed checkpointing; (13C) train and evaluate a **Planet-only dense U-Net** under spatial and leave-forest-out validation. Then introduce Sentinel-2 temporal/spectral, 10 m embedding **spatial patches**, and GEE-derived predictors **one branch at a time**, with resolution-aware resampling and aligned supervision. This isolates each family's incremental contribution and avoids creating hundreds of untested channels at once.

**Important:** the UAV maps are pseudolabels, not independent pixel-level ground truth. Coarser predictors must never be presented as having true 4.77 m information. Geographically disjoint dense holdouts require buffer zones around the *entire tiles*, not just their centers. Evaluation at 10/30/50/100 m will aggregate dense predictions on matched footprints and independent held-out UAV teacher observations.

Run only setup (00), Planet path function cell (01), and **13A** in a fresh session. 13A deliberately produces diagnostic samples only; it does not write training data or train a neural network.

In [ ]:
# 13A — Five-mosaic native Planet vs dense UAV teacher registration/encoding pilot.
# Only small windows; NO full-resolution UAV raster read or global resampling.
from pathlib import Path
import os,json,hashlib,math,time,contextlib
import numpy as np,pandas as pd,rasterio,matplotlib.pyplot as plt
from rasterio.windows import Window
from rasterio.warp import reproject,Resampling,transform_bounds
from rasterio.vrt import WarpedVRT
from rasterio.enums import MaskFlags
from pyproj import Transformer
from IPython.display import display
assert 'FINAL' in globals() and 'OUT' in globals() and 'planet_file' in globals()
PLANET_SITES13=['Baego','MOS','Yangambi','Yoko']
MOS13={
    'Baego':['Baego.tif'],
    'MOS':['Mos_1_2_transparent_mosaic_group1_georef.tif',
           'Mos_5_6_transparent_mosaic_group1_georef.tif'],
    'Yangambi':['Yangambi_georef.tif'],
    'Yoko':['Yoko_kvo_georef.tif']}
TEACHER_ROOT13=FINAL/'Outputs/V3_1/03_M2_warmstart_c6a33504c37efb2ebb84_full_mosaic/probability_maps'
CACHED_AGG13=FINAL/'Outputs/V3_1/04_satellite_full_warmstart/cache/aggregates'
OUT13=OUT/'13_dense_planet_unet'
AUDIT13=OUT13/'13A_registration_audit'
AUDIT13.mkdir(parents=True,exist_ok=True)
TARGETS13=OUT/'09_uav_targets_50m.parquet'
assert TARGETS13.exists(),'Existing 50m anchor table missing; Section 09 cache needed.'
ANCHOR13=pd.read_parquet(TARGETS13)
assert {'site','mosaic','x','y','crs','mean_probability'}.issubset(ANCHOR13.columns)
PILOT_SIZE13=64     # Native 4.77 m Planet pixels; ~305m x 305m window.
PILOTS_PER_MOSAIC13=2
MIN_TEACHER_VALID13=.50   # Report validity; don't silently reject samples.
ENCODING_TOL13=.12        # Validation against independent 10m teacher aggregate.
GTIFF_CACHE13=128         # MB per GDAL process, not raster-wide.
MAX_PILOTS13=None         # 2 for a short first look; None = 10 mosaic pilots.
os.environ['GDAL_CACHEMAX']=str(GTIFF_CACHE13)

def teacher_path13(name):
    return TEACHER_ROOT13/f'{Path(name).stem}_M2_warmstart_epoch02_probability_u16_FULL.tif'
def cached10_path13(name):
    return CACHED_AGG13/f'{Path(name).stem}_10m.tif'
def ref10_path13(site):
    year={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}[site]
    return FINAL/'Outputs/V3_1/04_satellite_predictors/input_layers/RAW_ANNUAL'/f'{site}_RAW_ANNUAL_{year}_10m.tif'

def read_10m_teacher13(name,bounds,crs):
    # Existing cached teacher is already normalized [0,1] and has UAV coverage.
    path=cached10_path13(name)
    assert path.exists(),path
    with rasterio.open(path) as src:
        coords=(bounds if src.crs==crs else transform_bounds(crs,src.crs,*bounds))
        from rasterio.windows import from_bounds as winfrom
        w=winfrom(*coords,transform=src.transform)
        # Bound input window for every pilot and preserve its native transform.
        w=w.round_offsets().round_lengths()
        z=src.read([1,3],window=w,boundless=True,fill_value=0,out_dtype='float32')
        return z,src.window_transform(w),src.crs

def raster_window_teacher13(uav_file,ptfm,pcrs):
    # Warp ONLY one 64x64 target window. WarpedVRT does not materialize the
    # complete mosaics; the destination lattice exactly matches Planet.
    with rasterio.Env(GDAL_CACHEMAX=GTIFF_CACHE13,NUM_THREADS='1'):
        with rasterio.open(uav_file) as src:
            assert src.crs and src.count>=1
            details=dict(dtype=src.dtypes[0],nodata=src.nodata,
                         scales=list(src.scales),offsets=list(src.offsets),
                         mask_flags=str(src.mask_flag_enums),
                         resolution=list(src.res))
            with WarpedVRT(src,crs=pcrs,transform=ptfm,
                           width=PILOT_SIZE13,height=PILOT_SIZE13,
                           resampling=Resampling.average,
                           warp_mem_limit=64) as vrt:
                raw=vrt.read(1,out_dtype='float32')
                mask=vrt.dataset_mask().astype('float32')/255
        return raw,mask,details

def align_cached10_to_planet13(cache10,cache_tfm,cache_crs,dst_tfm,dst_crs):
    result=np.full((PILOT_SIZE13,PILOT_SIZE13),np.nan,dtype='float32')
    cov=np.zeros_like(result)
    # 10m coverage gates teacher validity. Do not evaluate outside observed UAV.
    reproject(cache10[0],result,src_transform=cache_tfm,src_crs=cache_crs,
              src_nodata=None,dst_transform=dst_tfm,dst_crs=dst_crs,
              dst_nodata=np.nan,resampling=Resampling.bilinear)
    reproject(cache10[1],cov,src_transform=cache_tfm,src_crs=cache_crs,
              src_nodata=None,dst_transform=dst_tfm,dst_crs=dst_crs,
              dst_nodata=0,resampling=Resampling.nearest)
    return result,cov

source13=[]
for site,names in MOS13.items():
    p=planet_file(site,'Annual')
    assert p.exists(),p
    for name in names:
        tp=teacher_path13(name)
        cp=cached10_path13(name)
        assert tp.exists() and cp.exists(),(site,name,tp,cp)
        with rasterio.open(p) as ds, rasterio.open(tp) as src:
            assert ds.crs and src.crs
            assert ds.count>=16
            assert .1<abs(ds.res[0])<10,ds.res
            source13.append(dict(site=site,mosaic=name,
                planet_pixel_m=float(abs(ds.res[0])),
                planet_crs=str(ds.crs),teacher_crs=str(src.crs),
                teacher_dtype=src.dtypes[0],teacher_nodata=str(src.nodata),
                teacher_scale=str(src.scales[0]),teacher_offset=str(src.offsets[0]),
                source_size_GB=round(tp.stat().st_size/1e9,2)))
display(pd.DataFrame(source13))
pd.DataFrame(source13).to_csv(AUDIT13/'source_metadata.csv',index=False)

rows13=[]
for site,names in MOS13.items():
    with rasterio.open(planet_file(site,'Annual')) as planet:
        for name in names:
            subset=ANCHOR13[(ANCHOR13.site==site)&(ANCHOR13.mosaic==name)]
            assert len(subset)>=PILOTS_PER_MOSAIC13,(site,name,len(subset))
            # Deterministically select low and high Musanga sites to test scaling.
            subset=subset.sort_values('mean_probability').reset_index(drop=True)
            picks=[subset.iloc[int(i)] for i in np.linspace(
                len(subset)*.25,len(subset)*.85,PILOTS_PER_MOSAIC13)]
            for j,rec in enumerate(picks):
                final=AUDIT13/f'{site}_{Path(name).stem}_{j:02d}.npz'
                if final.exists():
                    with np.load(final,allow_pickle=False) as pack:
                        rows13.append(json.loads(str(pack['metadata'].item())))
                    print('CACHED 13A',site,Path(name).stem,j,flush=True)
                    continue
                if MAX_PILOTS13 is not None and len(rows13)>=MAX_PILOTS13:
                    break
                started=time.monotonic()
                transformer=Transformer.from_crs(rec.crs,planet.crs,always_xy=True)
                xp,yp=transformer.transform(float(rec.x),float(rec.y))
                cc,rr=(~planet.transform)*(xp,yp)
                win=Window(int(np.floor(cc))-PILOT_SIZE13//2,
                           int(np.floor(rr))-PILOT_SIZE13//2,
                           PILOT_SIZE13,PILOT_SIZE13)
                ptfm=planet.window_transform(win)
                rgbn=planet.read([13,14,15,16],window=win,
                                 boundless=True,fill_value=0,out_dtype='float32')
                pv=np.isfinite(rgbn).all(axis=0)&(rgbn>0).all(axis=0)
                raw,rawmask,srcinfo=raster_window_teacher13(
                    teacher_path13(name),ptfm,planet.crs)
                from rasterio.transform import array_bounds
                bounds=array_bounds(PILOT_SIZE13,PILOT_SIZE13,ptfm)
                cache10,cachetfm,cachecrs=read_10m_teacher13(name,bounds,planet.crs)
                cache,cov=align_cached10_to_planet13(
                    cache10,cachetfm,cachecrs,ptfm,planet.crs)
                good=pv&(rawmask>.90)&(cov>=.90)&np.isfinite(raw)&np.isfinite(cache)&(cache>=0)&(cache<=1)
                # Test possible uint16 probability interpretation. A hard failure
                # prevents mistaking unscaled uint16 as continuous probabilities.
                scale_metadata=srcinfo['scales'][0]
                offset_metadata=srcinfo['offsets'][0]
                options={'unit':raw}
                if srcinfo['dtype'].startswith('uint'):
                    info=np.iinfo(np.dtype(srcinfo['dtype']))
                    options[f'integer_full_{info.max}']=raw/float(info.max)
                    options['percent_10000']=raw/10000.
                if (scale_metadata is not None and offset_metadata is not None and
                    (scale_metadata!=1. or offset_metadata!=0.)):
                    options['raster_metadata']=raw*scale_metadata+offset_metadata
                matches=[]
                if good.sum()>=30:
                    for label,decoded in options.items():
                        d=np.abs(decoded[good]-cache[good])
                        eligible=np.isfinite(decoded[good])&(
                            decoded[good]>=-.02)&(decoded[good]<=1.02)
                        if eligible.mean()<.98:continue
                        matches.append((float(np.mean(d)),label))
                matches.sort()
                selected=matches[0][1] if matches else 'UNRESOLVED'
                agreement=matches[0][0] if matches else np.nan
                ok=bool(good.sum()>=30 and np.isfinite(agreement) and
                        agreement<=ENCODING_TOL13)
                decoded=options[selected] if selected!='UNRESOLVED' else np.full_like(raw,np.nan)
                record=dict(site=site,mosaic=name,pilot=j,
                    source_label=float(rec.mean_probability),n_valid=int(good.sum()),
                    valid_pct=round(float(good.mean())*100,2),
                    selected_encoding=selected,mean_abs_agreement_10m=agreement,
                    teacher_mean=float(np.mean(decoded[good])) if ok else np.nan,
                    cache_mean=float(np.mean(cache[good])) if good.any() else np.nan,
                    raw_min=float(np.nanmin(raw)),raw_max=float(np.nanmax(raw)),
                    raw_mask_pct=float(np.mean(rawmask>.90)*100),
                    planet_valid_pct=float(np.mean(pv)*100),
                    status='PASS' if ok else 'REVIEW',
                    seconds=round(time.monotonic()-started,2))
                temp=final.with_suffix('.npz.partial')
                with open(temp,'wb') as stream:
                    np.savez_compressed(stream,planet=rgbn.astype('float16'),
                        raw_teacher=raw.astype('float32'),
                        decoded_teacher=np.asarray(decoded,dtype='float32'),
                        uav_mask=rawmask.astype('float32'),valid=good.astype('uint8'),
                        cached10=cache.astype('float32'),
                        metadata=json.dumps(record))
                os.replace(temp,final)
                rows13.append(record)
                print('13A',site,Path(name).stem,j,
                      'valid',int(good.sum()),'best',selected,
                      '10m difference',round(agreement,5) if np.isfinite(agreement) else 'NA',
                      record['status'],'seconds',record['seconds'],flush=True)
        if MAX_PILOTS13 is not None and len(rows13)>=MAX_PILOTS13:break
summary13=pd.DataFrame(rows13)
summary13.to_csv(AUDIT13/'pilot_alignment_audit.csv',index=False)
display(summary13)
if len(summary13)==sum(len(x) for x in MOS13.values())*PILOTS_PER_MOSAIC13:
    disagreements=summary13.groupby('mosaic').selected_encoding.nunique()
    problem=summary13[(summary13.status!='PASS')]
    if (disagreements>1).any():
        print('ATTENTION: encoding differs between pilots of a mosaic.',flush=True)
    if not problem.empty:
        print('STOP: review failed/uncertain encoding or registration before 13B.',flush=True)
    else:
        print('13A pilot checks PASS; inspect example overlays before 13B.',flush=True)
else:
    print('13A partial diagnostic; rerun to finish mosaic samples.',flush=True)
# One quick side-by-side visualization per mosaic, no full mosaics opened.
fig,axes=plt.subplots(len(MOS13['Baego'])+len(MOS13['MOS'])+
                       len(MOS13['Yangambi'])+len(MOS13['Yoko']),3,
                       figsize=(12,16),layout='constrained')
for i,meta in enumerate([x for x in rows13 if x['pilot']==0]):
    file=AUDIT13/f"{meta['site']}_{Path(meta['mosaic']).stem}_{meta['pilot']:02d}.npz"
    with np.load(file) as data:
        raw=data['planet'].astype('float32')
        good=(raw>0).all(axis=0)
        rgb=raw[[2,1,0]].transpose(1,2,0)
        lo=np.percentile(rgb[good],2,axis=0) if good.any() else np.ones(3)
        hi=np.percentile(rgb[good],98,axis=0) if good.any() else np.ones(3)*2
        rgb=np.clip((rgb-lo)/(hi-lo+1e-6),0,1)
        teacher=np.where(data['valid']>0,data['decoded_teacher'],np.nan)
        cache=np.where(data['valid']>0,data['cached10'],np.nan)
    for ax in axes[i]:ax.set_xticks([]);ax.set_yticks([])
    axes[i,0].imshow(rgb);axes[i,0].set_title(meta['site']+' Planet RGB')
    axes[i,1].imshow(teacher,vmin=0,vmax=1,cmap='viridis')
    axes[i,1].set_title('UAV → 4.77m teacher')
    axes[i,2].imshow(cache,vmin=0,vmax=1,cmap='viridis')
    axes[i,2].set_title('Independent 10m teacher')
fig.savefig(AUDIT13/'planet_uav_teacher_preview.png',dpi=145)
plt.show();plt.close(fig)
print('13A saved:',AUDIT13,flush=True)


### 13A.2 · Reassess native-Planet versus cached-10 m teacher agreement (no source rereading)

The original 13A compared each **4.77 m** target pixel to a **bilinearly interpolated, already aggregated 10 m** teacher pixel. Those are different spatial supports, so their mean *absolute pixelwise difference* measures fine-scale texture as well as any misalignment. The earlier 0.12 threshold is **not a registration validation**; one Yoko pilot marginally failed despite the other nine passing.

This diagnostic opens only the 10 saved 13A pilots. It requires a consistent `uint16 / 10000` encoding, compares signed mean differences and aggregated-scale texture, quantifies correlation, and explores whether a small native-pixel shift would meaningfully improve agreement. **Shift inspection is diagnostic only**: no shift or calibration is applied without separate evidence. The pixel-level source remains the actual warped UAV probability raster and its coverage mask, not the interpolated 10 m cache.

A genuine spatial registration claim will additionally require imagery overlay inspection and/or independent ground control, since low-frequency agreement alone cannot establish subpixel accuracy.

In [ ]:
# 13A.2 — Check existing diagnostic patches without reopening huge UAV files.
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from pathlib import Path
from IPython.display import display
assert 'AUDIT13' in globals(),'Run 13A first.'
audit_files13=sorted(AUDIT13.glob('*.npz'))
assert len(audit_files13)==10,('Expected ten cached 13A mosaic pilots',len(audit_files13))
def local_smooth13(x,valid,sigma=1.0):
    v=valid.astype('float64')
    num=gaussian_filter(np.where(valid,x,0.0).astype('float64'),sigma=sigma)
    den=gaussian_filter(v,sigma=sigma)
    return np.divide(num,den,out=np.full_like(num,np.nan),where=den>.6),den
records13v=[]
for f in audit_files13:
    with np.load(f) as z:
        meta=__import__('json').loads(str(z['metadata'].item()))
        predicted=z['decoded_teacher'].astype('float64')
        reference=z['cached10'].astype('float64')
        valid=(z['valid'].astype(bool)&np.isfinite(predicted)&np.isfinite(reference)&
               (predicted>=0)&(predicted<=1)&(reference>=0)&(reference<=1))
        assert meta['selected_encoding']=='percent_10000',(f,meta['selected_encoding'])
        assert valid.sum()>200,(f,valid.sum())
        difference=predicted[valid]-reference[valid]
        p_s,wt=local_smooth13(predicted,valid,sigma=1.15)
        c_s,_=local_smooth13(reference,valid,sigma=1.15)
        core=valid&(wt>.95)&np.isfinite(p_s)&np.isfinite(c_s)
        assert core.sum()>100
        smdiff=p_s[core]-c_s[core]
        corr=np.corrcoef(p_s[core],c_s[core])[0,1] if (
            np.std(p_s[core])>1e-7 and np.std(c_s[core])>1e-7) else np.nan
        records13v.append(dict(
            site=meta['site'],mosaic=meta['mosaic'],pilot=meta['pilot'],
            n_valid=int(valid.sum()),encoding=meta['selected_encoding'],
            native_vs_10m_MAE_pp=100*np.mean(np.abs(difference)),
            signed_bias_pp=100*np.mean(difference),
            native_mean_pct=100*np.mean(predicted[valid]),
            cached_10m_mean_pct=100*np.mean(reference[valid]),
            smoothed_MAE_pp=100*np.mean(np.abs(smdiff)),
            smoothed_corr=corr,
            within_mean_2pp=bool(abs(np.mean(difference))<=.02)
        ))
report13v=pd.DataFrame(records13v).sort_values(['site','mosaic','pilot'])
report13v.to_csv(AUDIT13/'13A2_scale_agreement.csv',index=False)
display(report13v.round(3))
print('All pilots have consistent 0-10000 encoding:',(report13v.encoding=='percent_10000').all())
print('Mean bias within 2 percentage points:',
      int(report13v.within_mean_2pp.sum()),'/',len(report13v))
print('WARNING: similarity to a 10m reference is not proof of subpixel registration.')
print('13A2 files saved in',AUDIT13)


# Musanga — Block 13B: native PlanetScope / UAV dense training tiles

This is **Block 13B within the canonical NB06 notebook**. It uses the already computed full-resolution UAV U-Net probability maps as **dense supervision**, and annual PlanetScope blue/green/red/NIR reflectance as predictors. It **does not** read original UAV RGB orthomosaics, rerun U-Net inference, or retrain earlier regressions.

### Stage 13B goals

- Create non-overlapping **64×64 Planet pixels** (~305×305 m) on each native Planet raster grid, anchored to the existing verified 50 m ROI samples.
- Warp **only each tile's footprint** from UAV uint16 probability maps by area averaging; decode using confirmed **/10000**. Preserve separate ROI, satellite-valid, and UAV-valid masks; do not use padded/nodata pixels as negative examples.
- Save **georeferenced tile metadata**, source names, valid fractions, geographic footprint and fixed tile IDs. Split assignment belongs to *13C*, where buffers must be based on tile extents and not just label centers; the two MOS mosaics stay in one forest holdout.
- Cached chunks permit safe restart. Default **five-tile smoke test, selecting one representative high-coverage candidate from each mosaic**; inspect output before large-scale extraction.

**Important limitation:** these are teacher-supervised spatial predictions. Pixel-level probability agreement with the UAV model is *not* independent ground truth. Keeping tile footprints will allow rigorous geographic holdouts later. Monthly Planet/S2/embedding channels are *not* mixed into this first dataset; they will be added through independent branches after a Planet-only dense U-Net baseline.

Run the code cell; first run writes only 3 new tiles (`MAX_NEW_TILES13B=3`). Once we inspect its outputs, change that parameter to `None` to finish cached extraction.

**Revision after first pilot:** the original 128-pixel tiles disproportionately sampled Baego boundary fragments (valid fractions 0.08–0.46); 13B now uses 64-pixel patches and a masked-loss-appropriate ≥30% joint valid threshold, with ≥80% Planet validity. The five pilot tiles are chosen by per-mosaic anchor density, rather than file order. Source fingerprint changes, so the earlier cache is preserved. Geographic train/test separation and leakage-safe buffers remain required in 13C.


In [ ]:
# 13B — Restartable native Planet/UAV dense training tiles.
from pathlib import Path
import os,json,hashlib,time,math
import numpy as np,pandas as pd,rasterio,geopandas as gpd
from rasterio.windows import Window
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.transform import array_bounds
from pyproj import Transformer
from shapely.geometry import box
from IPython.display import display

ROOTS13B=[Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
          Path('/content/drive/MyDrive/Musanga/Final')]
FINAL13B=next((p for p in ROOTS13B if p.is_dir()),None)
if FINAL13B is None:
    from google.colab import drive
    drive.mount('/content/drive')
    FINAL13B=next((p for p in ROOTS13B if p.is_dir()),None)
assert FINAL13B is not None,'Musanga Final workspace not found.'
ROOT13B=FINAL13B/'Outputs/V3_1'
NB05_13B=ROOT13B/'05_sat_predictor_exploration'
OUT13B=ROOT13B/'06_spatial_planet_embeddings/13_dense_planet_unet/13B_tiles'
OUT13B.mkdir(parents=True,exist_ok=True)
SRC_UAV13B=ROOT13B/'03_M2_warmstart_c6a33504c37efb2ebb84_full_mosaic/probability_maps'
ANCHOR_FILE13B=ROOT13B/'06_spatial_planet_embeddings/09_uav_targets_50m.parquet'
MANIFEST_FILE13B=ROOT13B/'00_input_audit/00_input_manifest.json'
assert ANCHOR_FILE13B.is_file() and MANIFEST_FILE13B.is_file()
ANCH13B=pd.read_parquet(ANCHOR_FILE13B)
assert {'site','mosaic','x','y','crs','mean_probability'}.issubset(ANCH13B.columns)
manifest=json.loads(MANIFEST_FILE13B.read_text())
roi_file=Path(manifest['roi'])
assert roi_file.exists(),roi_file
ROI13B=gpd.read_file(roi_file)
assert not ROI13B.empty and ROI13B.crs

YRS13B={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2021}
MOSAICS13B={'Baego':['Baego.tif'],
            'MOS':['Mos_1_2_transparent_mosaic_group1_georef.tif',
                   'Mos_5_6_transparent_mosaic_group1_georef.tif'],
            'Yangambi':['Yangambi_georef.tif'],
            'Yoko':['Yoko_kvo_georef.tif']}
SIZE13B=64   # ~305 m support: better suited to irregular narrow ROI footprints
MAX_NEW_TILES13B=5    # One representative tile per mosaic; None extracts all.
MIN_ROI13B=.30         # Masked loss; permit partial but meaningful ROI coverage.
MIN_VALID_PIXELS13B=.30
MIN_PLANET13B=.80
SRC_BANDS13B=[13,14,15,16]  # B,G,R,N
ENCODING13B=10000.0
def planet13b(site):
    d=NB05_13B/'input_layers/Musanga_V31_NB05_UAV_ROI_Annual'
    matches=sorted(d.glob(f'NB05_UAV_ROI_{site}_{YRS13B[site]}_Annual_19band*.tif'))
    assert len(matches)==1,(site,matches)
    return matches[0]
def teacher13b(mosaic):
    f=SRC_UAV13B/f'{Path(mosaic).stem}_M2_warmstart_epoch02_probability_u16_FULL.tif'
    assert f.exists(),f
    return f
sources13b={site:planet13b(site) for site in YRS13B}
config13b=dict(version='dense_planet_teacher_v1',tile_px=SIZE13B,
               encoding=ENCODING13B,bands=SRC_BANDS13B,
               min_roi=MIN_ROI13B,min_valid=MIN_VALID_PIXELS13B,
               min_planet=MIN_PLANET13B,
               planet={k:(str(v),v.stat().st_size,v.stat().st_mtime_ns)
                       for k,v in sources13b.items()},
               teacher={m:(str(teacher13b(m)),teacher13b(m).stat().st_size,
                           teacher13b(m).stat().st_mtime_ns)
                        for vv in MOSAICS13B.values() for m in vv},
               anchors=(str(ANCHOR_FILE13B),ANCHOR_FILE13B.stat().st_size,
                        ANCHOR_FILE13B.stat().st_mtime_ns),
               roi=(str(roi_file),roi_file.stat().st_size,roi_file.stat().st_mtime_ns))
RUNID13B=hashlib.sha256(json.dumps(config13b,sort_keys=True).encode()).hexdigest()[:16]
RUN13B=OUT13B/RUNID13B
RUN13B.mkdir(parents=True,exist_ok=True)
conf_file=RUN13B/'config.json'
if conf_file.exists():
    # JSON turns tuples into lists; compare canonical JSON structures.
    # This remains a strict check of all source metadata and parameters.
    assert json.loads(conf_file.read_text())==json.loads(
        json.dumps(config13b,sort_keys=True)),'Cached source/config mismatch.'
else:
    conf_file.write_text(json.dumps(config13b,indent=2))
print('13B cache:',RUN13B,flush=True)

# Snap canonical 50 m ROI-approved anchors to non-overlapping Planet tiles.
# The Planet raster origin defines the lattice, NOT per-mosaic UAV origins.
records=[]
for site,mosaics in MOSAICS13B.items():
    with rasterio.open(sources13b[site]) as planet:
        assert planet.count>=16 and planet.crs and 4.<abs(planet.res[0])<5.5
        for mosaic in mosaics:
            sub=ANCH13B[(ANCH13B.site==site)&(ANCH13B.mosaic==mosaic)]
            assert len(sub)>0,(site,mosaic)
            assert sub.crs.nunique()==1
            tf=Transformer.from_crs(sub.crs.iloc[0],planet.crs,always_xy=True)
            x,y=tf.transform(sub.x.to_numpy(float),sub.y.to_numpy(float))
            px,py=(~planet.transform)*(x,y)
            # Reproducible globally aligned 128-pixel tile origin.
            col0=(np.floor(px/SIZE13B).astype('int64')*SIZE13B)
            row0=(np.floor(py/SIZE13B).astype('int64')*SIZE13B)
            unique=sorted(set(zip(row0,col0)))
            for row,col in unique:
                if row<0 or col<0 or row+SIZE13B>planet.height or col+SIZE13B>planet.width:
                    continue
                win=Window(int(col),int(row),SIZE13B,SIZE13B)
                transform=planet.window_transform(win)
                west,south,east,north=array_bounds(SIZE13B,SIZE13B,transform)
                records.append(dict(site=site,mosaic=mosaic,
                    row0=int(row),col0=int(col),
                    tile_id=f'{site}_{Path(mosaic).stem}_r{row:06d}_c{col:06d}',
                    crs=str(planet.crs),west=west,south=south,east=east,north=north,
                    centre_x=(west+east)/2,centre_y=(south+north)/2,
                    anchor_count=int(((row0==row)&(col0==col)).sum())))
TILES13B=pd.DataFrame(records).sort_values(['site','mosaic','row0','col0']).reset_index(drop=True)
assert len(TILES13B) and not TILES13B.tile_id.duplicated().any()
TILES13B.to_parquet(RUN13B/'tile_manifest_all_candidates.parquet',index=False)
print('Candidate tiles by mosaic:')
display(TILES13B.groupby(['site','mosaic']).agg(
    candidate_tiles=('tile_id','size'),anchors=('anchor_count','sum')))

# Pilot representative dense interior tiles across ALL mosaics, rather than
# the first (usually edge) tiles of Baego. Full extraction processes every tile.
if MAX_NEW_TILES13B is not None:
    selected13b=(TILES13B.sort_values('anchor_count',ascending=False)
       .groupby(['site','mosaic'],sort=True,group_keys=False).head(1).copy())
    assert len(selected13b)==sum(map(len,MOSAICS13B.values()))
    print('Representative pilot tiles (highest supported 50m anchor count):')
    display(selected13b[['site','mosaic','tile_id','anchor_count']])
else:
    selected13b=TILES13B
# Extract windows per mosaic; no full-resolution UAV RGB/teacher raster arrays.
new=0
for (site,mosaic),grp in selected13b.groupby(['site','mosaic'],sort=True):
    tile_dir=RUN13B/site/Path(mosaic).stem
    tile_dir.mkdir(parents=True,exist_ok=True)
    with rasterio.open(sources13b[site]) as planet, rasterio.open(teacher13b(mosaic)) as teacher:
        assert teacher.crs and teacher.nodata==65535,(mosaic,'unexpected teacher nodata',teacher.nodata)
        # Repair the earlier implicit VRT nodata conversion issue explicitly.
        with WarpedVRT(teacher,crs=planet.crs,transform=planet.transform,
                       width=planet.width,height=planet.height,
                       src_nodata=65535,nodata=65535,
                       dtype='float32',resampling=Resampling.average,
                       warp_mem_limit=64) as aligned:
            roi_local=ROI13B.to_crs(planet.crs)
            roi_geom=[g for g in roi_local.geometry if g is not None and not g.is_empty]
            assert roi_geom
            for rec in grp.itertuples(index=False):
                dest=tile_dir/f'{rec.tile_id}.npz'
                if dest.exists():
                    try:
                        with np.load(dest,allow_pickle=False) as old:
                            assert old['planet'].shape==(4,SIZE13B,SIZE13B)
                            assert old['probability'].shape==(SIZE13B,SIZE13B)
                            assert old['valid'].shape==(SIZE13B,SIZE13B)
                        continue
                    except Exception as err:
                        print('REBUILD partial',dest.name,str(err)[:100],flush=True)
                if MAX_NEW_TILES13B is not None and new>=MAX_NEW_TILES13B:break
                start=time.monotonic()
                win=Window(rec.col0,rec.row0,SIZE13B,SIZE13B)
                ptfm=planet.window_transform(win)
                image=planet.read(SRC_BANDS13B,window=win,out_dtype='float32')
                pv=planet.dataset_mask(window=win)>0
                pv &= np.isfinite(image).all(axis=0)&(image>0).all(axis=0)
                teacher_raw=aligned.read(1,window=win,out_dtype='float32')
                tm=aligned.dataset_mask(window=win)>0
                tv=tm&np.isfinite(teacher_raw)&(teacher_raw>=0)&(teacher_raw<=10000)
                roim=rasterize(((g,1) for g in roi_geom),out_shape=(SIZE13B,SIZE13B),
                               transform=ptfm,dtype='uint8',all_touched=False)>0
                valid=roim&pv&tv
                fraction=dict(roi=float(roim.mean()),planet=float(pv.mean()),
                              teacher=float(tv.mean()),joint=float(valid.mean()))
                accepted=(fraction['roi']>=MIN_ROI13B and
                          fraction['planet']>=MIN_PLANET13B and
                          fraction['joint']>=MIN_VALID_PIXELS13B)
                prob=np.clip(teacher_raw/ENCODING13B,0,1).astype('float32')
                prob[~valid]=0.
                image[:,~pv]=0.
                metadata=dict(site=site,mosaic=mosaic,tile_id=rec.tile_id,
                    crs=rec.crs,row0=int(rec.row0),col0=int(rec.col0),
                    bounds=[rec.west,rec.south,rec.east,rec.north],
                    anchor_count=int(rec.anchor_count),
                    roi_fraction=fraction['roi'],planet_fraction=fraction['planet'],
                    teacher_fraction=fraction['teacher'],joint_fraction=fraction['joint'],
                    accepted=bool(accepted),probability_encoding='uint16_div_10000',
                    source_type='UAV_U-Net_pseudolabel',size=SIZE13B)
                tmp=dest.with_suffix('.npz.partial')
                with open(tmp,'wb') as f:
                    np.savez_compressed(f,planet=np.clip(image,-65000,65000).astype('float16'),
                                        probability=prob.astype('float16'),
                                        valid=valid.astype('uint8'),
                                        roi=roim.astype('uint8'),
                                        metadata=json.dumps(metadata))
                os.replace(tmp,dest)
                new+=1
                print('13B SAVED',rec.tile_id,
                      'accepted',accepted,'valid',round(fraction['joint'],3),
                      'seconds',round(time.monotonic()-start,2),flush=True)
            if MAX_NEW_TILES13B is not None and new>=MAX_NEW_TILES13B:break
    if MAX_NEW_TILES13B is not None and new>=MAX_NEW_TILES13B:break

status=[]
for row in TILES13B.itertuples(index=False):
    path=RUN13B/row.site/Path(row.mosaic).stem/f'{row.tile_id}.npz'
    base=row._asdict()
    base['saved']=path.exists()
    if path.exists():
        with np.load(path,allow_pickle=False) as z:
            meta=json.loads(str(z['metadata'].item()))
        for key in ('accepted','joint_fraction','roi_fraction','planet_fraction','teacher_fraction'):
            base[key]=meta[key]
    status.append(base)
STATE13B=pd.DataFrame(status)
STATE13B.to_csv(RUN13B/'tile_manifest_progress.csv',index=False)
print('13B saved',int(STATE13B.saved.sum()),'of',len(STATE13B),
      'candidate tiles, accepted so far',
      int(STATE13B['accepted'].eq(True).sum()),flush=True)
display(STATE13B[STATE13B.saved].tail(12))
print('13B tile cache:',RUN13B)
print('Inspect five representative mosaic tiles; then set MAX_NEW_TILES13B=None to finish extraction.')


## 13C · Dense 4.77 m PlanetScope U-Net — spatial holdouts and masked learning

**Inputs:** 422 accepted 64 × 64 Planet B/G/R/N patches and aligned UAV U-Net pseudo-label probabilities from 13B; cached files only. The U-Net predicts one probability per **native PlanetScope pixel**, not a 50 m average. The `valid` mask determines exactly which pixels contribute to the masked loss and evaluation.

**Evaluation design:** (a) three deterministic geographically blocked test folds, with training tiles excluded when they lie within a **305 m buffer around the full held-out tile footprints**, and (b) strict leave-one-forest-out (LOFO), grouping *both* MOS mosaics in the MOS forest. Report site-balanced mean absolute error at native resolution, site-specific errors and bias, and spatially aggregated 50 m agreement in a later evaluation block. Pixel counts are **not independent sample counts**. The dense predictions measure fidelity to the saved UAV teacher, not validation against independent field inventories.

**13C.1:** build and inspect split manifest, leakage checks and per-site sample counts **before training**. **13C.2:** train a modest 4-band U-Net with masked MSE, training-only per-band scaling, spatial flips/rotations, forest-balanced tile sampling, saved fold predictions and checkpoints. Start with `MAX_NEW_FOLDS13C=1`; all folds are restartable. Dataset size is small enough for a GPU runtime. No prior notebook cells need to run if the saved 13B cache exists.

**Important:** LOFO has only four forests, and Planet image years differ across sites. Treat transfer accuracy cautiously. We will add S2 temporal/spectral and embedding *spatial patches* only after obtaining this honest Planet-only dense baseline.

**Pilot priority after 13C.1:** start with LOFO Yoko (387 training / 35 test tiles); blocked folds have just 88–108 training tiles after buffering and should be secondary sensitivity analyses.\n

In [ ]:
# 13C.1 — Cached-only geographical split audit (NO GPU training).
from pathlib import Path
import json,hashlib
import numpy as np,pandas as pd
from scipy.spatial import cKDTree
from IPython.display import display
roots=[Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
       Path('/content/drive/MyDrive/Musanga/Final')]
final13c=next((x for x in roots if x.is_dir()),None)
if final13c is None:
    from google.colab import drive
    drive.mount('/content/drive')
    final13c=next((x for x in roots if x.is_dir()),None)
assert final13c is not None
ROOT13C=final13c/'Outputs/V3_1/06_spatial_planet_embeddings/13_dense_planet_unet'
DIR13B=ROOT13C/'13B_tiles/31631cfae43b39ab'
MAN13B=DIR13B/'tile_manifest_progress.csv'
assert MAN13B.is_file(),MAN13B
M13=pd.read_csv(MAN13B)
assert len(M13)==480 and M13.saved.astype(str).str.lower().eq('true').all()
M13=M13[M13.accepted.astype(str).str.lower().eq('true')].copy()
M13=M13.sort_values(['site','mosaic','row0','col0']).reset_index(drop=True)
assert len(M13)==422,('Unexpected valid tile count',len(M13))
assert not M13.tile_id.duplicated().any()
M13['tile_path']=[str(DIR13B/rec.site/Path(rec.mosaic).stem/(rec.tile_id+'.npz'))
                  for rec in M13.itertuples(index=False)]
assert all(Path(x).is_file() for x in M13.tile_path)
assert (M13.crs=='EPSG:32635').all()
BLOCK_M13C=1200.
BUFFER_M13C=305.28  # Full-tile footprint separation (edge to edge) beyond touching.
NFOLDS13C=3
SEED13C=20261025
# Deterministic geographic blocks with per-site reproducible hash.
bx=np.floor(M13.centre_x.to_numpy(float)/BLOCK_M13C).astype('int64')
by=np.floor(M13.centre_y.to_numpy(float)/BLOCK_M13C).astype('int64')
M13['spatial_fold']=(np.mod(bx*73856093+by*19349663,NFOLDS13C)).astype('int8')
# Exact rectangle minimum distance, not center distance.
def rect_distance13c(A,B):
    dx=np.maximum(np.maximum(A[:,None,0]-B[None,:,2],B[None,:,0]-A[:,None,2]),0)
    dy=np.maximum(np.maximum(A[:,None,1]-B[None,:,3],B[None,:,1]-A[:,None,3]),0)
    return np.hypot(dx,dy)
GEOM13C=M13[['west','south','east','north']].to_numpy(float)
SITES13C=['Baego','MOS','Yangambi','Yoko']
SCHEMES13C=[('BLOCK_BUFFERED',str(i)) for i in range(NFOLDS13C)]+[
    ('LEAVE_FOREST_OUT',site) for site in SITES13C]
def split13c(scheme,hold):
    site=M13.site.to_numpy()
    if scheme=='BLOCK_BUFFERED':
        test=(M13.spatial_fold.to_numpy()==int(hold))
    else:
        test=(site==hold)
    train=~test
    if scheme=='BLOCK_BUFFERED':
        for forest in SITES13C:
            ti=np.where(test & (site==forest))[0]
            tr=np.where(train & (site==forest))[0]
            if len(ti)==0 or len(tr)==0:continue
            d=rect_distance13c(GEOM13C[tr],GEOM13C[ti])
            train[tr[d.min(axis=1)<BUFFER_M13C]]=False
    a=np.where(train)[0];b=np.where(test)[0]
    assert len(a)>25 and len(b)>5,(scheme,hold,len(a),len(b))
    assert len(np.intersect1d(a,b))==0
    if scheme=='BLOCK_BUFFERED':
        for forest in SITES13C:
            ti=b[site[b]==forest];tr=a[site[a]==forest]
            if len(ti) and len(tr):
                d=rect_distance13c(GEOM13C[tr],GEOM13C[ti])
                assert np.min(d)>=BUFFER_M13C-1e-6,(scheme,hold,forest,float(np.min(d)))
    else:
        assert not (site[a]==hold).any()
    return a,b
counts=[]
for scheme,hold in SCHEMES13C:
    tr,te=split13c(scheme,hold)
    row=dict(scheme=scheme,heldout=hold,n_train=len(tr),n_test=len(te),
             n_buffer_excluded=len(M13)-len(tr)-len(te))
    for site in SITES13C:
        row[f'train_{site}']=int((M13.site.iloc[tr]==site).sum())
        row[f'test_{site}']=int((M13.site.iloc[te]==site).sum())
    counts.append(row)
SPLITS13C=pd.DataFrame(counts)
print('ACCEPTED per forest:')
display(M13.groupby('site').agg(tiles=('tile_id','size'),valid_share=('joint_fraction','mean')))
print('SPATIAL SPLIT COUNTS; full rectangle exclusion validated:')
display(SPLITS13C)
DIR13C=ROOT13C/'13C_dense_unet'
DIR13C.mkdir(parents=True,exist_ok=True)
M13.to_parquet(DIR13C/'accepted_tile_manifest.parquet',index=False)
SPLITS13C.to_csv(DIR13C/'split_audit.csv',index=False)
print('13C.1 SPLITS VERIFIED. No training performed.',flush=True)


In [ ]:
# 13C.2 — Dense masked U-Net, restartable pilot and fold-level reporting.
# First execute 13C.1 above. GPU is strongly recommended.
import os,random,time,json,hashlib
from pathlib import Path
import numpy as np,pandas as pd,torch
from torch import nn
from torch.utils.data import Dataset,DataLoader,WeightedRandomSampler
from IPython.display import display
assert all(x in globals() for x in ('M13','DIR13C','SCHEMES13C','split13c'))
DEVICE13C=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS13C=30
BATCH13C=16
LR13C=0.0008
MAX_NEW_FOLDS13C=None    # Complete missing LOFO forests; cache avoids repeating Yoko.
torch.set_num_threads(2)
RUNCONF13C=dict(version='dense_unet_13C_v1',tile_run=DIR13B.name,
                 folds=SCHEMES13C,buffer_m=BUFFER_M13C,
                 model='UNet4_24_48_96',epochs=EPOCHS13C,
                 batch=BATCH13C,lr=LR13C,seed=SEED13C,
                 normalization='train_only_robust_planet',
                 target='UAV_teacher_probability_valid_masked',
                 sampling='forest_balanced_replacement')
ID13C=hashlib.sha256(json.dumps(RUNCONF13C,sort_keys=True).encode()).hexdigest()[:16]
TRAIN_DIR13C=DIR13C/ID13C
TRAIN_DIR13C.mkdir(parents=True,exist_ok=True)
(TRAIN_DIR13C/'config.json').write_text(json.dumps(RUNCONF13C,indent=2))
print('13C.2 device:',DEVICE13C,'| fold cache:',TRAIN_DIR13C,flush=True)

def seed13c(value):
    random.seed(value);np.random.seed(value);torch.manual_seed(value)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(value)

def normalize13c(indices):
    # Samples only train tile pixels, never held-out forests or blocks.
    data=[]
    for k in indices[::max(1,len(indices)//100)]:
        with np.load(M13.tile_path.iloc[int(k)],allow_pickle=False) as z:
            img=z['planet'].astype('float32')
            mask=z['valid'].astype(bool)
            data.append(img[:,::4,::4].reshape(4,-1)[:,mask[::4,::4].ravel()])
    for band in range(4):
        assert any(len(a[band]) for a in data)
    allpixels=np.concatenate(data,axis=1)
    med=np.median(allpixels,axis=1)
    q10=np.percentile(allpixels,10,axis=1)
    q90=np.percentile(allpixels,90,axis=1)
    span=np.maximum((q90-q10)/2,1e-3)
    return med.astype('float32'),span.astype('float32')

class DenseTiles13C(Dataset):
    def __init__(self,indices,mu,scale,augment=False):
        self.index=np.asarray(indices,dtype=int)
        self.mu=mu[:,None,None];self.scale=scale[:,None,None]
        self.augment=augment
    def __len__(self):return len(self.index)
    def __getitem__(self,i):
        k=int(self.index[i])
        with np.load(M13.tile_path.iloc[k],allow_pickle=False) as pack:
            x=pack['planet'].astype('float32')
            y=pack['probability'].astype('float32')
            valid=pack['valid'].astype('float32')
        pixel_good=np.isfinite(x).all(axis=0)&(x>0).all(axis=0)
        x=np.clip((x-self.mu)/self.scale,-3,3)
        x[:,~pixel_good]=0
        if self.augment:
            if random.random()<.5:
                x=x[:,:,::-1];y=y[:,::-1];valid=valid[:,::-1]
            if random.random()<.5:
                x=x[:,::-1,:];y=y[::-1,:];valid=valid[::-1,:]
            n=random.randrange(4)
            x=np.rot90(x,n,axes=(1,2));y=np.rot90(y,n);valid=np.rot90(valid,n)
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(y[None])),
                torch.from_numpy(np.ascontiguousarray(valid[None])))

class DoubleConv13(nn.Module):
    def __init__(self,a,b):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(True),
                               nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(True))
    def forward(self,x):return self.net(x)

class DenseUNet13(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1=DoubleConv13(4,24);self.e2=DoubleConv13(24,48)
        self.e3=DoubleConv13(48,96);self.pool=nn.MaxPool2d(2)
        self.bottleneck=DoubleConv13(96,128)
        self.u3=nn.ConvTranspose2d(128,96,2,stride=2)
        self.d3=DoubleConv13(192,96)
        self.u2=nn.ConvTranspose2d(96,48,2,stride=2)
        self.d2=DoubleConv13(96,48)
        self.u1=nn.ConvTranspose2d(48,24,2,stride=2)
        self.d1=DoubleConv13(48,24)
        self.out=nn.Sequential(nn.Conv2d(24,1,1),nn.Sigmoid())
    def forward(self,x):
        a=self.e1(x);b=self.e2(self.pool(a));c=self.e3(self.pool(b))
        bott=self.bottleneck(self.pool(c))
        z=self.d3(torch.cat((self.u3(bott),c),dim=1))
        z=self.d2(torch.cat((self.u2(z),b),dim=1))
        z=self.d1(torch.cat((self.u1(z),a),dim=1))
        return self.out(z)

def train_fold13c(scheme,hold):
    tr,te=split13c(scheme,hold)
    med,span=normalize13c(tr)
    seed13c(SEED13C+sum(map(ord,hold))+(100 if scheme=='LEAVE_FOREST_OUT' else 0))
    site=M13.site.iloc[tr].to_numpy()
    counts={s:int((site==s).sum()) for s in np.unique(site)}
    weights=np.asarray([1/counts[s] for s in site],dtype='float64')
    sampler=WeightedRandomSampler(torch.from_numpy(weights),num_samples=len(tr),
                                   replacement=True)
    trloader=DataLoader(DenseTiles13C(tr,med,span,True),batch_size=BATCH13C,
                        sampler=sampler,num_workers=0,pin_memory=DEVICE13C.type=='cuda')
    teloader=DataLoader(DenseTiles13C(te,med,span,False),batch_size=BATCH13C,
                        shuffle=False,num_workers=0,pin_memory=DEVICE13C.type=='cuda')
    model=DenseUNet13().to(DEVICE13C)
    opt=torch.optim.AdamW(model.parameters(),lr=LR13C,weight_decay=1e-3)
    history=[]
    for ep in range(EPOCHS13C):
        model.train();numer=0.;denom=0.
        for xb,yb,mb in trloader:
            xb=xb.to(DEVICE13C);yb=yb.to(DEVICE13C);mb=mb.to(DEVICE13C)
            opt.zero_grad(set_to_none=True)
            yp=model(xb)
            loss=((yp-yb).square()*mb).sum()/mb.sum().clamp_min(1)
            loss.backward();opt.step()
            numer+=float(loss.item())*float(mb.sum().item())
            denom+=float(mb.sum().item())
        history.append(dict(epoch=ep+1,train_masked_MSE=numer/denom))
        print(f'13C {scheme}/{hold} epoch {ep+1:02d}/{EPOCHS13C} masked MSE={numer/denom:.6f}',flush=True)
    model.eval();outputs=[];mse=[];mae=[];bias=[];nvalid=[]
    with torch.inference_mode():
        for xb,yb,mb in teloader:
            pred=model(xb.to(DEVICE13C)).cpu().numpy()[:,0]
            real=yb.numpy()[:,0];mask=mb.numpy()[:,0]>0
            outputs.extend(pred.astype('float16'))
            for p,y,m in zip(pred,real,mask):
                assert m.sum()>100,('Unexpected tile with few valid pixels',int(m.sum()))
                delta=p[m]-y[m]
                mse.append(float(np.mean(delta**2)))
                mae.append(float(np.mean(np.abs(delta))))
                bias.append(float(np.mean(delta)))
                nvalid.append(int(m.sum()))
    assert len(outputs)==len(te)
    return model,history,te,np.stack(outputs),mae,bias,mse,nvalid,med,span

# Prioritize forest-to-forest generalization; spatial blocks remain secondary.
# First GPU run is LOFO Yoko (387 train / 35 test), not the small buffered fold.
RUN_ORDER13C=[('LEAVE_FOREST_OUT',s) for s in ('Yoko','Baego','Yangambi','MOS')]
assert all(k in SCHEMES13C for k in RUN_ORDER13C)
print('13C VERSION: FOUR-FOREST LOFO | first fold:',RUN_ORDER13C[0],'train/test:',tuple(len(a) for a in split13c(*RUN_ORDER13C[0])),flush=True)
new13c=0
for scheme,hold in RUN_ORDER13C:
    stem=f'{scheme}_{hold}'
    predfile=TRAIN_DIR13C/f'{stem}_tile_metrics.parquet'
    tilefile=TRAIN_DIR13C/f'{stem}_predictions.npz'
    if predfile.exists() and tilefile.exists():
        try:
            saved=pd.read_parquet(predfile)
            _,te=split13c(scheme,hold)
            with np.load(tilefile) as z:
                ok=(len(saved)==len(te) and np.array_equal(
                    z['tile_index'],te.astype('int32')) and
                    z['prediction'].shape==(len(te),64,64))
            if ok:
                print('CACHED dense U-Net:',stem,flush=True)
                continue
        except Exception as ex:
            print('INCOMPLETE CACHE REBUILD:',stem,str(ex)[:120],flush=True)
    if MAX_NEW_FOLDS13C is not None and new13c>=MAX_NEW_FOLDS13C:
        print('Pilot fold limit reached; change MAX_NEW_FOLDS13C=None to continue.',flush=True)
        break
    t0=time.monotonic()
    net,log,te,pred,mae,bias,mse,valid,med,span=train_fold13c(scheme,hold)
    stats=M13.iloc[te][['site','mosaic','tile_id','joint_fraction']].reset_index(drop=True).copy()
    stats['MAE_pp']=np.asarray(mae)*100
    stats['bias_pp']=np.asarray(bias)*100
    stats['RMSE_pp']=np.sqrt(np.asarray(mse))*100
    stats['valid_pixel_count']=valid
    tmp=predfile.with_suffix('.parquet.partial')
    stats.to_parquet(tmp,index=False);os.replace(tmp,predfile)
    tmp=tilefile.with_suffix('.npz.partial')
    with open(tmp,'wb') as f:
        np.savez_compressed(f,tile_index=te.astype('int32'),
                            prediction=pred.astype('float16'))
    os.replace(tmp,tilefile)
    pd.DataFrame(log).to_csv(TRAIN_DIR13C/f'{stem}_train_history.csv',index=False)
    wtfile=TRAIN_DIR13C/f'{stem}_model.pt'
    tmp=wtfile.with_suffix('.pt.partial')
    torch.save(dict(state=net.state_dict(),planet_median=med,
                    planet_scale=span,configuration=RUNCONF13C),tmp)
    os.replace(tmp,wtfile)
    new13c+=1
    print('SAVED:',stem,'tiles=',len(te),'seconds=',round(time.monotonic()-t0,1),flush=True)
    display(stats.groupby('site').agg(tiles=('tile_id','size'),
        MAE_pp=('MAE_pp','mean'),bias_pp=('bias_pp','mean')).round(3))
files=[]
for scheme,hold in SCHEMES13C:
    path=TRAIN_DIR13C/f'{scheme}_{hold}_tile_metrics.parquet'
    if path.exists():
        d=pd.read_parquet(path);d['scheme']=scheme;d['holdout']=hold
        files.append(d)
if files:
    allmetrics=pd.concat(files,ignore_index=True)
    allmetrics.to_parquet(TRAIN_DIR13C/'all_completed_tile_metrics.parquet',index=False)
    print('13C folds completed:',len(files),'/',len(SCHEMES13C))
    display(allmetrics.groupby(['scheme','site']).agg(
        tiles=('tile_id','size'),MAE_pp=('MAE_pp','mean'),
        bias_pp=('bias_pp','mean')).round(3))
print('CHECKPOINTS:',TRAIN_DIR13C,flush=True)


### 13C.3 · Diagnose saved dense U-Net predictions before additional training

**Cached-only (no GPU necessary).** Evaluate 13C's completed spatial fold 0 with pooled **valid-pixel** MAE, RMSE, bias and R² by forest; compare with two honest constant predictors: mean UAV probability across all *training tiles* and mean over training tiles in the *same forest*. The latter is an intentionally strong **within-forest** baseline and is **not** available under leave-one-forest-out. Report distributions and inspect four representative maps. These metrics measure agreement with the UAV teacher, **not** actual field Musanga detection accuracy. The site R² is undefined if the reference has negligible variation.

**Interpretation:** The first fold used 88 training tiles and 145 test tiles, with substantial geographic buffering. A 4.77 m pixel-level MAE is not directly comparable to errors previously reported for 50 m averages. This diagnostic avoids additional training until we understand whether model predictions recover within-forest spatial differences and improve on trivial means.

In [ ]:
# 13C.3 — Diagnostic of saved BLOCK_BUFFERED/0 outputs; zero GPU time.
from pathlib import Path
import json,numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
ROOT13C3=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/06_spatial_planet_embeddings/13_dense_planet_unet')
DIR13B3=ROOT13C3/'13B_tiles/31631cfae43b39ab'
DIR13C3=ROOT13C3/'13C_dense_unet'
MAN13C3=pd.read_parquet(DIR13C3/'accepted_tile_manifest.parquet')
RUN13C3=DIR13C3/'816bb788b43ec113'
SPLIT13C3='BLOCK_BUFFERED_0'
PRED13C3=RUN13C3/f'{SPLIT13C3}_predictions.npz'
assert PRED13C3.exists(),PRED13C3
with np.load(PRED13C3,allow_pickle=False) as z:
    ti13c3=z['tile_index'].astype(int)
    yp13c3=z['prediction'].astype('float32')
assert yp13c3.shape==(len(ti13c3),64,64)
assert not MAN13C3.tile_id.duplicated().any()
# Rebuild the split using the saved fold assignments, where available.
if 'M13' in globals() and 'split13c' in globals():
    tr13c3,te13c3=split13c('BLOCK_BUFFERED','0')
    assert np.array_equal(ti13c3,te13c3)
    train13c3=M13.iloc[tr13c3]
else:
    # Site/block split implementation reproduced from 13C.1, only to identify
    # training pixels for constant baselines; no model fitting.
    m=MAN13C3
    x=m.centre_x.to_numpy(float);y=m.centre_y.to_numpy(float)
    fol=np.mod(np.floor(x/1200).astype('int64')*73856093+
               np.floor(y/1200).astype('int64')*19349663,3)
    assert np.array_equal(ti13c3,np.where(fol==0)[0])
    te13c3=ti13c3
    trainmask=fol!=0
    for forest in sorted(m.site.unique()):
        tst=m.iloc[te13c3][lambda d:d.site==forest]
        if tst.empty:continue
        ids=np.where(trainmask & (m.site.to_numpy()==forest))[0]
        A=m.iloc[ids][['west','south','east','north']].to_numpy(float)
        B=tst[['west','south','east','north']].to_numpy(float)
        dx=np.maximum(np.maximum(A[:,None,0]-B[None,:,2],B[None,:,0]-A[:,None,2]),0)
        dy=np.maximum(np.maximum(A[:,None,1]-B[None,:,3],B[None,:,1]-A[:,None,3]),0)
        trainmask[ids[np.hypot(dx,dy).min(axis=1)<305.28]]=False
    train13c3=m.iloc[np.where(trainmask)[0]]
print('Saved trained fold:',SPLIT13C3,'train tiles',len(train13c3),'test tiles',len(ti13c3),flush=True)

def load_tile13c3(rec):
    file=DIR13B3/rec.site/Path(rec.mosaic).stem/(rec.tile_id+'.npz')
    with np.load(file,allow_pickle=False) as z:
        return z['probability'].astype('float32'),z['valid'].astype(bool)
# Pixel-weighted constant reference baselines computed from training data only.
sumsite13c3={};countsite13c3={}
for rec in train13c3.itertuples(index=False):
    y,m=load_tile13c3(rec)
    sumsite13c3[rec.site]=sumsite13c3.get(rec.site,0.)+float(y[m].sum())
    countsite13c3[rec.site]=countsite13c3.get(rec.site,0)+int(m.sum())
overall_mean13c3=sum(sumsite13c3.values())/sum(countsite13c3.values())
forest_mean13c3={k:sumsite13c3[k]/countsite13c3[k] for k in sumsite13c3}
print('Training-only mean UAV probability (all forests):',round(overall_mean13c3,3))
print('Training-only within-forest means:',{k:round(v,3) for k,v in forest_mean13c3.items()})
results13c3=[]
show13c3={}
for pos,k in enumerate(ti13c3):
    rec=MAN13C3.iloc[int(k)]
    target,mask=load_tile13c3(rec)
    pred=yp13c3[pos]
    assert mask.sum()>100 and np.isfinite(pred[mask]).all()
    if rec.site not in show13c3:
        show13c3[rec.site]=(rec.tile_id,target,mask,pred)
    gt=target[mask];hat=pred[mask]
    results13c3.append(dict(site=rec.site,tile_id=rec.tile_id,
        n=int(mask.sum()),mae=float(np.abs(hat-gt).sum()),
        bias=float((hat-gt).sum()),sse=float(((hat-gt)**2).sum()),
        sumy=float(gt.sum()),sumy2=float((gt*gt).sum()),
        baseline_global_abs=float(np.abs(overall_mean13c3-gt).sum()),
        baseline_site_abs=float(np.abs(forest_mean13c3[rec.site]-gt).sum()),
        pred_mean=float(hat.mean()),teacher_mean=float(gt.mean())))
TABLE13C3=pd.DataFrame(results13c3)
rows=[]
for forest,d in TABLE13C3.groupby('site'):
    N=d.n.sum();mu=d.sumy.sum()/N
    variance=d.sumy2.sum()-N*mu**2
    rows.append(dict(site=forest,test_tiles=len(d),valid_pixels=int(N),
      teacher_mean_pct=100*mu,pred_mean_pct=100*d.pred_mean.mean(),
      MAE_pp=100*d.mae.sum()/N,
      RMSE_pp=100*np.sqrt(d.sse.sum()/N),bias_pp=100*d.bias.sum()/N,
      R2_pixel=1-d.sse.sum()/variance if variance>1e-6 else np.nan,
      global_train_mean_MAE_pp=100*d.baseline_global_abs.sum()/N,
      own_forest_train_mean_MAE_pp=100*d.baseline_site_abs.sum()/N))
OUT13C3=pd.DataFrame(rows).sort_values('site')
OUT13C3.to_csv(DIR13C3/'13C3_fold0_diagnostic.csv',index=False)
display(OUT13C3.round(3))
fig,axs=plt.subplots(len(show13c3),3,figsize=(10,2.8*len(show13c3)),constrained_layout=True)
for axrow,(forest,(tileid,target,mask,pred)) in zip(axs,sorted(show13c3.items())):
    axrow[0].imshow(np.ma.masked_where(~mask,target),vmin=0,vmax=1,cmap='viridis')
    axrow[1].imshow(np.ma.masked_where(~mask,pred),vmin=0,vmax=1,cmap='viridis')
    axrow[2].imshow(np.ma.masked_where(~mask,pred-target),vmin=-.5,vmax=.5,cmap='coolwarm')
    for ax,title in zip(axrow,(f'{forest} UAV teacher','Planet U-Net prediction','Prediction − teacher')):
        ax.set_title(title);ax.axis('off')
fig.suptitle('13C spatial fold 0 — first held-out tile from each site')
fig.savefig(DIR13C3/'13C3_fold0_map_diagnostic.png',dpi=150,bbox_inches='tight')
plt.show()
print('Diagnostics saved:',DIR13C3,flush=True)


### 13C.4 · Squeeze more information from PlanetScope — one cached diagnostic

**No retraining, no extra notebooks.** Uses completed **BLOCK_BUFFERED/0** predictions and the same UAV-teacher targets.

1. **Calibration:** per-forest prediction/reference means, signed bias, and variance ratio (a post-hoc descriptive linear relationship is **not** fitted or used to adjust held-out results).
2. **Discrimination:** pooled **within-forest** Pearson/Spearman correlations and R² at each evaluation support. These describe spatial association; pixels within a tile are not independent.
3. **Spatial support:** area-weighted, *masked* probabilities reprojected to independent **10, 30 and 50 m grid cells**, requiring at least 75% observed UAV valid support **and** 95% tile-footprint support. Report n cells, MAE/RMSE/bias, R² and skill versus a training-only global constant. Tiles are aggregated **individually**, so grid cells crossing tile edges are omitted; those numbers are diagnostics, not the final seamless 50 m evaluation.
4. **PlanetScope input quality:** compare B/G/R/N robust reflectance distributions, train-normalized medians, and the fraction beyond the network's `[-3,+3]` clipping bounds. Also assess invalid pixels. This detects band distribution shifts **but cannot independently certify cloud masking**.

Outputs are saved alongside 13C results. **Decision gate:** proceed to LOFO Yoko only after identifying whether the major problem is calibration, lost small-scale signal, or between-site input shift.

In [ ]:
# 13C.4 — Calibration, discrimination, 4.77/10/30/50m support and band shift.
# Cached files only; CPU-friendly; no model fitting.
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt,torch
from scipy.stats import spearmanr,pearsonr
from rasterio.warp import reproject
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from IPython.display import display
from collections import defaultdict

ROOT134=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/06_spatial_planet_embeddings/13_dense_planet_unet')
CACHE134=ROOT134/'13B_tiles/31631cfae43b39ab'
WORK134=ROOT134/'13C_dense_unet'
FOLD134=WORK134/'816bb788b43ec113'
META134=pd.read_parquet(WORK134/'accepted_tile_manifest.parquet').reset_index(drop=True)
with np.load(FOLD134/'BLOCK_BUFFERED_0_predictions.npz',allow_pickle=False) as d:
    IDX134=d['tile_index'].astype(int)
    PR134=d['prediction'].astype('float32')
assert len(IDX134)==145 and PR134.shape==(145,64,64)
MODEL134=FOLD134/'BLOCK_BUFFERED_0_model.pt'
assert MODEL134.is_file(),MODEL134
# Locally generated 13C.2 checkpoint contains NumPy normalisation arrays,
# which PyTorch >=2.6 rejects under the restricted weights-only unpickler.
# weights_only=False is appropriate ONLY for this checkpoint we produced
# ourselves in this same project; never use it for downloaded/untrusted files.
ck134=torch.load(MODEL134,map_location='cpu',weights_only=False)
MU134=np.asarray(ck134['planet_median'],dtype='float32')
SC134=np.asarray(ck134['planet_scale'],dtype='float32')
assert MU134.shape==(4,) and (SC134>0).all()
BANDS134=('B','G','R','NIR')
SCALES134=(4.77,10.,30.,50.)
MASK_MIN134=.75
EDGE_MIN134=.95
def tile134(rec):
    f=CACHE134/rec.site/Path(rec.mosaic).stem/(rec.tile_id+'.npz')
    with np.load(f,allow_pickle=False) as z:
        x=z['planet'].astype('float32')
        y=z['probability'].astype('float32')
        m=z['valid'].astype(bool)
    assert x.shape==(4,64,64) and y.shape==m.shape==(64,64)
    return x,y,m

# Footprint alignment is determined by each Planet tile's saved UTM coordinates,
# independently of source raster size (4.77m, 64x64).
def aggregate134(y,p,m,rec,scale):
    if scale==4.77:
        return y[m].astype('float32'),p[m].astype('float32'),m.sum()
    west,south,east,north=[float(getattr(rec,k)) for k in ('west','south','east','north')]
    x0=np.floor(west/scale)*scale
    y0=np.ceil(north/scale)*scale
    nx=int(np.ceil((east-x0)/scale))
    ny=int(np.ceil((y0-south)/scale))
    src_transform=from_origin(west,north,(east-west)/64,(north-south)/64)
    dst_transform=from_origin(x0,y0,scale,scale)
    mf=m.astype('float32')
    src=np.stack([y*mf,p*mf,mf],axis=0).astype('float32')
    dst=np.zeros((3,ny,nx),dtype='float32')
    reproject(source=src,destination=dst,src_transform=src_transform,
              src_crs='EPSG:32635',dst_transform=dst_transform,
              dst_crs='EPSG:32635',src_nodata=None,dst_nodata=None,
              resampling=Resampling.average)
    # Compute exact destination cell intersection with tile bounds.
    xx0=x0+np.arange(nx)*scale
    xx1=xx0+scale
    yy1=y0-np.arange(ny)*scale
    yy0=yy1-scale
    xover=np.maximum(0,np.minimum(xx1,east)-np.maximum(xx0,west))/scale
    yover=np.maximum(0,np.minimum(yy1,north)-np.maximum(yy0,south))/scale
    footprint_fraction=yover[:,None]*xover[None,:]
    ok=(dst[2]>=MASK_MIN134)&(footprint_fraction>=EDGE_MIN134)&(dst[2]>1e-6)
    return (dst[0][ok]/dst[2][ok],dst[1][ok]/dst[2][ok],int(ok.sum()))

pairs134=defaultdict(lambda:dict(y=[],p=[],tile=[]))
bandrows134=[]
for position,idx in enumerate(IDX134):
    rec=META134.iloc[int(idx)]
    x,y,m=tile134(rec)
    p=PR134[position]
    assert np.isfinite(x[:,m]).all() and np.isfinite(p[m]).all()
    for scale in SCALES134:
        a,b,n=aggregate134(y,p,m,rec,scale)
        if n:
            key=(rec.site,scale)
            pairs134[key]['y'].append(a)
            pairs134[key]['p'].append(b)
            pairs134[key]['tile'].append(rec.tile_id)
    z=(x-MU134[:,None,None])/SC134[:,None,None]
    for j,name in enumerate(BANDS134):
        v=x[j,m];nv=z[j,m]
        bandrows134.append(dict(site=rec.site,band=name,n=len(v),
           median=float(np.median(v)),p10=float(np.percentile(v,10)),
           p90=float(np.percentile(v,90)),median_train_z=float(np.median(nv)),
           pct_clipped=float(np.mean(np.abs(nv)>3)*100),
           pct_zero_or_negative=float(np.mean(v<=0)*100)))
    if (position+1)%40==0:
        print('13C.4 inspected tiles',position+1,'/',len(IDX134),flush=True)

# Use the same global training-only constant as 13C.3, if present,
# otherwise recover it from saved training-only tiles and split membership.
if 'overall_mean13c3' in globals():
    CONST134=float(overall_mean13c3)
else:
    assert 'split13c' in globals() and 'M13' in globals(), 'Run 13C.1 or 13C.3 first'
    tr134,_=split13c('BLOCK_BUFFERED','0')
    numerator=denominator=0.
    for rec in M13.iloc[tr134].itertuples(index=False):
        _,y,m=tile134(rec)
        numerator+=float(y[m].sum());denominator+=int(m.sum())
    CONST134=numerator/denominator

metrics134=[]
for (site,scale),v in sorted(pairs134.items()):
    yy=np.concatenate(v['y']);pp=np.concatenate(v['p'])
    if len(yy)<3:continue
    diff=pp-yy
    denom=np.sum((yy-yy.mean())**2)
    pr=float(np.corrcoef(yy,pp)[0,1]) if np.std(pp)>1e-8 and np.std(yy)>1e-8 else np.nan
    sp=float(spearmanr(yy,pp).statistic) if len(yy)>4 and np.std(yy)>1e-8 else np.nan
    metrics134.append(dict(site=site,scale_m=scale,n_cells=len(yy),
        n_tiles=len(set(v['tile'])),teacher_mean_pct=100*yy.mean(),
        predicted_mean_pct=100*pp.mean(),bias_pp=100*diff.mean(),
        MAE_pp=100*np.abs(diff).mean(),RMSE_pp=100*np.sqrt(np.mean(diff**2)),
        R2=1-np.sum(diff**2)/denom if denom>1e-8 else np.nan,
        pearson=pr,spearman=sp,
        pred_to_reference_sd_ratio=float(np.std(pp)/np.std(yy)) if np.std(yy)>1e-8 else np.nan,
        train_mean_MAE_pp=100*np.abs(CONST134-yy).mean()))
MET134=pd.DataFrame(metrics134)
MET134.to_csv(WORK134/'13C4_scale_skill_by_site.csv',index=False)
display(MET134.round(3))
print('Training-only global constant probability:',round(CONST134,4),flush=True)

# Band comparability: average per-tile percentiles is descriptive, not pooled quantiles.
B134=pd.DataFrame(bandrows134)
S134=B134.groupby(['site','band'],as_index=False).agg(
    n_tiles=('n','size'),median=('median','median'),
    tile_p10_median=('p10','median'),tile_p90_median=('p90','median'),
    train_z_median=('median_train_z','median'),
    clipped_pct=('pct_clipped','mean'))
S134.to_csv(WORK134/'13C4_Planet_band_shift.csv',index=False)
print('Planet B/G/R/NIR reflectance QA (held-out test tiles, model training normalization):')
display(S134.round(3))

fig,axs=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
for site,g in MET134.groupby('site'):
    axs[0].plot(g.scale_m,g.MAE_pp,marker='o',label=site)
    axs[1].plot(g.scale_m,g.R2,marker='o',label=site)
axs[0].set(xlabel='Evaluation support (m)',ylabel='MAE (percentage points)',title='Spatial aggregation')
axs[1].set(xlabel='Evaluation support (m)',ylabel='Held-out pixel/cell R²',title='Spatial variation explained')
axs[1].axhline(0,color='grey',lw=.7)
for ax in axs:ax.grid(alpha=.25);ax.legend(fontsize=8)
fig.savefig(WORK134/'13C4_scale_diagnostic.png',dpi=170,bbox_inches='tight')
plt.show()
print('13C.4 saved:',WORK134)
print('NOTE: tile-wise grids exclude cross-tile cells; final mapping should reassemble mosaics.')


### 13D.0 · PlanetScope temporal inventory — verify monthly availability before adding channels

The current dense U-Net has **only annual PlanetScope** B/G/R/NIR (bands 13–16 of NB05 annual composites). S2 quarterly features exist in the earlier 50 m experiments, **but are not fed to the 4.77 m U-Net**. This lightweight audit checks whether NB05 / Planet files available on Google Drive include monthly, quarterly or scene-level source products, and lists the actual annual-band descriptions. **It does not download, composite, or read whole rasters.**

Decision: prefer monthly only if each site/month has adequate cloud-free support; otherwise evaluate quarterly/wet–dry composites with clear-pixel counts. Do not treat calendar-month filenames alone as proof of usable data.

In [ ]:
# 13D.0 — Local Planet temporal inventory: metadata only; no imagery reads.
from pathlib import Path
from collections import Counter
import re,pandas as pd,rasterio
from IPython.display import display
final130=next((p for p in [
    Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
    Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
if final130 is None:
    from google.colab import drive
    drive.mount('/content/drive')
    final130=next((p for p in [
        Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
        Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert final130 is not None
nb05_130=final130/'Outputs/V3_1/05_sat_predictor_exploration'
assert nb05_130.exists(),nb05_130
locations130=[nb05_130/'input_layers',nb05_130/'derived_features']
# Directory traversal only, limits printed rows (no pixel/raster IO).
entries130=[]
for base in locations130:
    if not base.is_dir():continue
    for f in base.rglob('*'):
        if not f.is_file():continue
        if f.suffix.lower() not in ('.tif','.tiff','.parquet','.csv','.json','.geojson'):continue
        low=f.name.lower()
        group=('MONTH' if re.search(r'month|monthly|(?:^|[_-])m(?:0?[1-9]|1[012])(?:[_-]|\\.)',low)
               else 'QUARTER' if re.search(r'quarter|season|q[1-4]',low)
               else 'ANNUAL' if 'annual' in low else 'OTHER')
        entries130.append(dict(kind=group,file=f.name,folder=str(f.parent.relative_to(nb05_130)),
                              size_MB=round(f.stat().st_size/1e6,2)))
INV130=pd.DataFrame(entries130)
if INV130.empty:raise RuntimeError('No NB05 input/derived feature files found')
print('NB05 temporal candidate inventory (file counts):')
display(INV130.groupby(['kind','folder']).size().rename('files').reset_index().sort_values(['kind','folder']))
print('Monthly/seasonal candidate files (up to 35):')
display(INV130[INV130.kind.isin(['MONTH','QUARTER'])].head(35))
annual130=INV130[INV130.file.str.contains('Annual_19band',case=False,regex=False)]
print('Annual 19-band Planet metadata:')
details130=[]
for f in annual130.file:
    paths=sorted((nb05_130/'input_layers').rglob(f))
    if len(paths)!=1:continue
    with rasterio.open(paths[0]) as ds:
        details130.append(dict(file=f,band_count=ds.count,
            crs=str(ds.crs),resolution=tuple(round(x,4) for x in ds.res),
            descriptions=[str(x) for x in ds.descriptions],
            band_13_16=[str(x) for x in ds.descriptions[12:16]]))
display(pd.DataFrame(details130).drop(columns='descriptions') if details130 else pd.DataFrame())
out130=final130/'Outputs/V3_1/06_spatial_planet_embeddings/13_dense_planet_unet/13D_temporal_inventory'
out130.mkdir(parents=True,exist_ok=True)
INV130.to_csv(out130/'NB05_temporal_file_inventory.csv',index=False)
pd.DataFrame(details130).to_json(out130/'NB05_annual_band_metadata.json',orient='records',indent=2)
print('Inventory written to:',out130)
print('Only file existence and metadata were checked; cloud quality and monthly completeness remain unverified.')


### 13D.1 · Monthly PlanetScope quality over the actual training footprints

**Focused, cached-input audit; no model training or new satellite extraction.** Find the existing 12 monthly NICFI DN4 images per forest and assess their per-month **valid-data fraction, band values and temporal jumps** on a deterministic sample of accepted native PlanetScope tiles. Compare their metadata (CRS, pixel size, band interpretation) to the annual raster. Do **not** assume 4-band order, units or projection match without checking. Monthly DN imagery is not by itself an independently verified cloud-free product: validity here checks mask/nodata, and the spectral jumps are only **cloud/haze/shadow warning proxies**.

This allows one next, defensible decision: retain 12 months where valid and reasonably stable, or summarize to multi-month/seasonal composites with QA masks. No model changes until the **Yoko strict LOFO** result is reviewed.

In [ ]:
# 13D.1 — Sample existing monthly Planet imagery over 13B forest footprints.
# Read only 32x32 windows per sampled tile, no whole-raster operations.
from pathlib import Path
import re,json,hashlib
import numpy as np,pandas as pd,rasterio
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window
from rasterio.enums import Resampling
from IPython.display import display
base=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1')
n05=base/'05_sat_predictor_exploration/input_layers'
tile_cache=base/'06_spatial_planet_embeddings/13_dense_planet_unet/13B_tiles/31631cfae43b39ab'
accepted=pd.read_csv(tile_cache/'tile_manifest_progress.csv')
accepted=accepted[accepted.accepted.astype(str).str.lower()=='true'].copy()
years={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2021}
annualdir=n05/'Musanga_V31_NB05_UAV_ROI_Annual'
monthlydirs=[n05/'Musanga_V31_NB05_UAV_ROI_Monthly',
             n05/'Musanga_V3_1_05_Planet_NICFI_Monthly']
quality=[]
metadata=[]
SAMPLE_TILES_PER_SITE=8
WIN=32
for site,year in years.items():
    annual=sorted(annualdir.glob(f'NB05_UAV_ROI_{site}_{year}_Annual_19band*.tif'))
    assert len(annual)==1,(site,'annual',annual)
    monthfiles={}
    for folder in monthlydirs:
        if not folder.is_dir():continue
        for f in folder.glob(f'*{site}_{year}_M??_DN4.tif'):
            match=re.search(r"_M([0-9]{2})_DN4[.]tif$", f.name)
            if match:
                month=int(match.group(1))
                if month in monthfiles:
                    raise AssertionError(f'Duplicate {site} month {month}: {monthfiles[month]} vs {f}')
                monthfiles[month]=f
    print(site,'months available:',sorted(monthfiles),flush=True)
    assert monthfiles,(site,'no monthly files found')
    tiles=accepted[accepted.site==site].sort_values('tile_id')
    assert len(tiles)>0
    selected=tiles.iloc[np.linspace(0,len(tiles)-1,min(SAMPLE_TILES_PER_SITE,len(tiles)),dtype=int)]
    with rasterio.open(annual[0]) as a:
        assert a.count==19,(site,a.count)
        metadata.append(dict(site=site,type='annual',file=annual[0].name,
                   resolution=tuple(a.res),crs=str(a.crs),bands=a.count,
                   descriptions=str(a.descriptions[12:16]),
                   nodata=str(a.nodata),dtype=str(a.dtypes[0])))
        for month,f in sorted(monthfiles.items()):
            with rasterio.open(f) as src:
                assert src.count==4,(f,src.count)
                metadata.append(dict(site=site,type=f'M{month:02d}',file=f.name,
                         resolution=tuple(src.res),crs=str(src.crs),bands=src.count,
                         descriptions=str(src.descriptions),
                         nodata=str(src.nodata),dtype=str(src.dtypes[0])))
                # VRT reads only small windows but ensures annual native grid alignment.
                with WarpedVRT(src,crs=a.crs,transform=a.transform,
                          width=a.width,height=a.height,
                          resampling=Resampling.nearest) as vrt:
                    for tile in selected.itertuples(index=False):
                        window=Window(int(tile.col0)+16,int(tile.row0)+16,WIN,WIN)
                        pixel=vrt.read([1,2,3,4],window=window,out_dtype='float32')
                        satmask=vrt.dataset_mask(window=window)>0
                        valid=satmask & np.isfinite(pixel).all(axis=0) & (pixel>0).all(axis=0)
                        # Teacher valid pixels restrict the QA to actually supervised support.
                        tilefile=tile_cache/site/Path(tile.mosaic).stem/(tile.tile_id+'.npz')
                        with np.load(tilefile,allow_pickle=False) as z:
                            teachmask=z['valid'][16:48,16:48].astype(bool)
                        mask=valid&teachmask
                        av=a.read([13,14,15,16],window=window,out_dtype='float32')
                        amask=teachmask&np.isfinite(av).all(axis=0)&(av>0).all(axis=0)
                        med=np.median(pixel[:,mask],axis=1) if mask.any() else np.full(4,np.nan)
                        annualmed=np.median(av[:,amask],axis=1) if amask.any() else np.full(4,np.nan)
                        quality.append(dict(site=site,month=month,tile=tile.tile_id,
                            n_roi=int(teachmask.sum()),
                            valid_fraction=float(mask.sum()/max(1,teachmask.sum())),
                            **{f'median_band_{j+1}':float(med[j]) for j in range(4)},
                            **{f'annual_median_band_{j+1}':float(annualmed[j]) for j in range(4)}))
Q131=pd.DataFrame(quality)
M131=pd.DataFrame(metadata)
assert not Q131.empty
summary=(Q131.groupby(['site','month']).agg(
    sample_tiles=('tile','nunique'),
    valid_fraction=('valid_fraction','mean'),
    **{f'median_band_{b}':(f'median_band_{b}','median') for b in range(1,5)}
).reset_index())
# Per-site month-to-month median change; no multiline lambda.
for b in range(1, 5):
    col = f'median_band_{b}'
    change = summary.groupby('site')[col].pct_change(fill_method=None).abs()
    summary[f'band_{b}_change_fraction'] = change.replace([np.inf, -np.inf], np.nan)
summary['maximum_band_change_fraction']=summary[
    [f'band_{b}_change_fraction' for b in range(1,5)]].max(axis=1)
out=base/'06_spatial_planet_embeddings/13_dense_planet_unet/13D_temporal_inventory'
out.mkdir(parents=True,exist_ok=True)
Q131.to_csv(out/'13D1_monthly_tile_QA.csv',index=False)
summary.to_csv(out/'13D1_monthly_site_QA.csv',index=False)
M131.to_csv(out/'13D1_monthly_raster_metadata.csv',index=False)
print('Monthly availability, validity and median band values (band ordering to verify from metadata):')
display(summary.round(3))
print('Band descriptions, grid and nodata examples:')
display(M131.groupby('site',sort=False).head(2))
print('QA saved to',out,flush=True)
print('IMPORTANT: DN4 image validity and temporal jumps are NOT an authoritative cloud mask.')


### 13D.2 · PlanetScope seasonal predictors on the SAME dense UAV training tiles

Build a **small, controlled temporal feature set**, without new satellite downloads or UAV processing. Monthly four-band Planet DN4 images are sampled on precisely the same native 4.77 m PlanetScope grid and same 64 × 64 accepted tiles as Block 13B. The annual-only model remains the untouched benchmark. Initial seasonal channels: monthly median B/G/R/N, temporal 10–90% spread of each band, median NDVI, NDVI temporal spread, plus clear-observation fraction (11 channels). A first-pass caution mask excludes **September for Yangambi and Yoko**, where 13D.1 suggests major visible-band contamination; remaining potential cloud/haze effects are not independently certified. Baego has only 8 available months; this is represented by the observation-count fraction and must be reported.

**Run first with `MAX_NEW_TILES13D2=5`** (one representative tile per mosaic); inspect results before setting it to `None`. Cache is restartable, and contains **features + masks only**, not duplicated UAV target arrays. Next controlled experiment: train annual 4-channel versus annual+these 11 channels using exactly the same LOFO folds (start with Yoko). After that, **Sentinel-2 seasonal bands** will be the next incremental experiment.

In [ ]:
# 13D.2 — Restartable native-grid seasonal Planet features; 5-tile pilot.
from pathlib import Path
from collections import defaultdict
import json,os,re,hashlib
import numpy as np,pandas as pd,rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.windows import Window
from IPython.display import display
BASE132=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1')
INPUT132=BASE132/'05_sat_predictor_exploration/input_layers'
TILE132=BASE132/'06_spatial_planet_embeddings/13_dense_planet_unet/13B_tiles/31631cfae43b39ab'
assert TILE132.is_dir()
accepted132=pd.read_csv(TILE132/'tile_manifest_progress.csv')
accepted132=accepted132[accepted132.accepted.astype(str).str.lower().eq('true')].copy()
assert len(accepted132)==422,len(accepted132)
YEARS132={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2021}
MONT132=[INPUT132/'Musanga_V31_NB05_UAV_ROI_Monthly',
         INPUT132/'Musanga_V3_1_05_Planet_NICFI_Monthly']
ANN132=INPUT132/'Musanga_V31_NB05_UAV_ROI_Annual'
OUT132=BASE132/'06_spatial_planet_embeddings/13_dense_planet_unet/13D_seasonal_tiles_v1'
OUT132.mkdir(parents=True,exist_ok=True)
MAX_NEW_TILES13D2=5     # Pilot: one tile per mosaic; then None.
# Initial documented QA exclusion, NOT a global atmospheric correction.
EXCLUDE132={'Baego':set(),'MOS':set(),'Yangambi':{9},'Yoko':{9}}
NAMES132=['B_month_median','G_month_median','R_month_median','N_month_median',
          'B_month_p90p10','G_month_p90p10','R_month_p90p10','N_month_p90p10',
          'NDVI_month_median','NDVI_month_p90p10','monthly_valid_fraction']
PRODS132={}
for site,year in YEARS132.items():
    aa=sorted(ANN132.glob(f'NB05_UAV_ROI_{site}_{year}_Annual_19band*.tif'))
    assert len(aa)==1,(site,aa)
    months={}
    for folder in MONT132:
        if not folder.exists():continue
        for f in folder.glob(f'*{site}_{year}_M??_DN4.tif'):
            match=re.search(r'_M([0-9]{2})_DN4[.]tif$',f.name)
            if match:
                m=int(match.group(1))
                assert m not in months,(site,m,months[m],f) if m in months else True
                months[m]=f
    assert len(months)>=8,(site,sorted(months))
    use={m:v for m,v in sorted(months.items()) if m not in EXCLUDE132[site]}
    assert len(use)>=7,(site,len(use))
    PRODS132[site]=(aa[0],use)
    print(site,'usable months',list(use),'excluded',sorted(EXCLUDE132[site]),flush=True)
CONFIG132=dict(version='planet_seasonal_11_v1',features=NAMES132,
               parent='31631cfae43b39ab',excluded={k:sorted(v) for k,v in EXCLUDE132.items()},
               annual_monthly_files={k:dict(annual=str(a),monthly={str(m):str(v) for m,v in d.items()})
                                     for k,(a,d) in PRODS132.items()})
ID132=hashlib.sha256(json.dumps(CONFIG132,sort_keys=True).encode()).hexdigest()[:16]
RUN132=OUT132/ID132
RUN132.mkdir(exist_ok=True)
conf132=RUN132/'config.json'
if conf132.exists():
    assert json.loads(conf132.read_text())==json.loads(json.dumps(CONFIG132))
else:
    conf132.write_text(json.dumps(CONFIG132,indent=2))
# Pilot: one well-supported tile per mosaic, then cache all 422.
pilot132=(accepted132.sort_values('joint_fraction',ascending=False)
          .groupby(['site','mosaic'],sort=True,group_keys=False).head(1))
selected132=pilot132 if MAX_NEW_TILES13D2 is not None else accepted132
print('13D.2 processing',len(selected132),'tiles; available',len(accepted132),flush=True)
counter132=0
for site,g in selected132.groupby('site',sort=True):
    annual,monthly=PRODS132[site]
    with rasterio.open(annual) as a:
        assert a.crs and a.count>=16
        sources={m:rasterio.open(f) for m,f in monthly.items()}
        vrts={}
        try:
            for m,src in sources.items():
                assert src.count==4,src.name
                # WarpedVRT permits slight grid offsets without full-image reprojection.
                vrts[m]=WarpedVRT(src,crs=a.crs,transform=a.transform,
                    width=a.width,height=a.height,resampling=Resampling.nearest)
            for rec in g.itertuples(index=False):
                dest=RUN132/rec.site/Path(rec.mosaic).stem/(rec.tile_id+'.npz')
                dest.parent.mkdir(parents=True,exist_ok=True)
                if dest.exists():
                    with np.load(dest,allow_pickle=False) as z:
                        assert z['features'].shape==(11,64,64)
                        assert z['valid'].shape==(64,64)
                    continue
                if MAX_NEW_TILES13D2 is not None and counter132>=MAX_NEW_TILES13D2:break
                tilepath=TILE132/rec.site/Path(rec.mosaic).stem/(rec.tile_id+'.npz')
                with np.load(tilepath,allow_pickle=False) as z:
                    uv=z['valid'].astype(bool)
                window=Window(int(rec.col0),int(rec.row0),64,64)
                channels=[]
                for m,vrt in sorted(vrts.items()):
                    x=vrt.read([1,2,3,4],window=window,out_dtype='float32')
                    good=(vrt.dataset_mask(window=window)>0)&np.isfinite(x).all(axis=0)&(x>0).all(axis=0)
                    x[:,~good]=np.nan
                    channels.append(x)
                cube=np.stack(channels,axis=0)  # month, band, y, x
                goodmonths=np.isfinite(cube).all(axis=1)
                # Exclude invalid monthly pixels without conflating them with zero Musanga.
                with np.errstate(invalid='ignore',divide='ignore'):
                    n=len(channels)
                    # nanpercentile: only emit features where >=3 monthly observations.
                    n_valid=goodmonths.sum(axis=0)
                    sufficient=(n_valid>=3)&uv
                    safe=np.where(np.isfinite(cube),cube,np.nan)
                    med=np.nanmedian(safe,axis=0)
                    spread=np.nanpercentile(safe,90,axis=0)-np.nanpercentile(safe,10,axis=0)
                    ndvi=(safe[:,3]-safe[:,2])/(safe[:,3]+safe[:,2]+1e-6)
                    ndmed=np.nanmedian(ndvi,axis=0)
                    ndspread=np.nanpercentile(ndvi,90,axis=0)-np.nanpercentile(ndvi,10,axis=0)
                    features=np.concatenate([med,spread,ndmed[None],ndspread[None],
                                             (n_valid/n)[None]],axis=0)
                assert features.shape==(11,64,64)
                finite=np.isfinite(features).all(axis=0)&sufficient
                features[:,~finite]=0.
                feature_stats=dict(site=site,mosaic=rec.mosaic,tile_id=rec.tile_id,
                    months_used=list(monthly),n_months=len(monthly),
                    n_valid=int(finite.sum()),n_uav_valid=int(uv.sum()),
                    usable_fraction=float(finite.sum()/max(1,uv.sum())))
                temp=dest.with_suffix('.npz.partial')
                with open(temp,'wb') as fh:
                    np.savez_compressed(fh,features=features.astype('float16'),
                        valid=finite.astype('uint8'),metadata=json.dumps(feature_stats))
                os.replace(temp,dest)
                counter132+=1
                print('13D2 SAVED',rec.tile_id,'feature-valid',round(feature_stats['usable_fraction'],3),
                      'months',len(monthly),flush=True)
        finally:
            for v in vrts.values():v.close()
            for src in sources.values():src.close()
    if MAX_NEW_TILES13D2 is not None and counter132>=MAX_NEW_TILES13D2:break
status=[]
for rec in accepted132.itertuples(index=False):
    path=RUN132/rec.site/Path(rec.mosaic).stem/(rec.tile_id+'.npz')
    data=dict(site=rec.site,mosaic=rec.mosaic,tile_id=rec.tile_id,saved=path.exists())
    if path.exists():
        with np.load(path,allow_pickle=False) as z:
            data.update(json.loads(str(z['metadata'].item())))
    status.append(data)
STATUS132=pd.DataFrame(status)
STATUS132.to_csv(RUN132/'seasonal_progress.csv',index=False)
print('Seasonal tiles saved:',int(STATUS132.saved.sum()),'/',len(STATUS132))
display(STATUS132[STATUS132.saved].groupby('site').agg(
    tiles=('tile_id','size'),mean_feature_valid_fraction=('usable_fraction','mean')).round(3))
print('Run cache:',RUN132)


### 13D.3 · Controlled annual versus seasonal PlanetScope U-Net — Yoko forest holdout

All 422 seasonal 11-feature tiles are now saved. This step uses **exactly the existing 13C.1 forest holdout**: train on Baego (18), MOS (332), Yangambi (37); test on **Yoko (35)**. All 4 annual and 11 seasonal bands are normalized from **training sites only**. Same compact U-Net architecture, 30 epochs, masked MSE and forest-balanced training as 13C.2. The yearly and monthly source products are not refitted or altered. Results are saved with a distinct config fingerprint.

This cell runs only the seasonal model and saves predictions, held-out pixel metrics, and tile-level held-out pixel metrics. The 30/50 m aggregation will use the same cached prediction arrays in the following comparison block. It prints the annual-only Yoko baseline **if it exists**; otherwise clearly notes that a matched annual-only Yoko run is still required. Differences between models are meaningful only when the test cohort, split, training budget, and randomization are matched. As before, UAV probability is a pseudo-label, not a field-observed true probability.

**Run 13C.1 (split definitions) first**, then run this cell on GPU. You do **not** need to run 13C.2 or 13D.2 again.

In [ ]:
# 13D.3 — Annual+seasonal (15-channel) Planet dense U-Net, strict LOFO Yoko.
from pathlib import Path
import json,os,hashlib,random,time
import numpy as np,pandas as pd,torch
from torch import nn
from torch.utils.data import Dataset,DataLoader,WeightedRandomSampler
from IPython.display import display
assert 'M13' in globals() and 'split13c' in globals(), 'Run 13C.1 first (only reads manifest).'
BASE133=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1')
ROOT133=BASE133/'06_spatial_planet_embeddings/13_dense_planet_unet'
TILES133=ROOT133/'13B_tiles/31631cfae43b39ab'
SEASONROOT133=ROOT133/'13D_seasonal_tiles_v1/002933e18a2acd4f'
assert (SEASONROOT133/'seasonal_progress.csv').exists()
progress133=pd.read_csv(SEASONROOT133/'seasonal_progress.csv')
assert len(progress133)==422 and progress133.saved.astype(str).str.lower().eq('true').all()
assert len(M13)==422
TRA133,TEST133=split13c('LEAVE_FOREST_OUT','Yoko')
assert len(TRA133)==387 and len(TEST133)==35
assert (M13.iloc[TEST133].site=='Yoko').all()
EPOCHS133=30
BATCH133=16
LR133=8e-4
SEED133=20261025+sum(map(ord,'Yoko'))+100
DEVICE133=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE133.type=='cuda','Please select a Colab GPU runtime before training.'
CONF133=dict(version='planet_annual_plus_seasonal_unet_15channels_v1',
    annual_tile_cache=TILES133.name,seasonal_tile_cache=SEASONROOT133.name,
    holdout='Yoko',epochs=EPOCHS133,batch=BATCH133,lr=LR133,seed=SEED133,
    channels=15,architecture='UNet4_24_48_96',
    sampling='balanced_forest_replacement',loss='masked_MSE')
ID133=hashlib.sha256(json.dumps(CONF133,sort_keys=True).encode()).hexdigest()[:16]
OUT133=ROOT133/'13D3_seasonal_unet_Yoko'/ID133
OUT133.mkdir(parents=True,exist_ok=True)
(OUT133/'config.json').write_text(json.dumps(CONF133,indent=2))
print('13D.3:',DEVICE133,'output:',OUT133,flush=True)

def paths133(rec):
    rel=Path(rec.site)/Path(rec.mosaic).stem/(rec.tile_id+'.npz')
    return TILES133/rel,SEASONROOT133/rel
def load133(i):
    rec=M13.iloc[int(i)]
    ap,sp=paths133(rec)
    with np.load(ap,allow_pickle=False) as a,np.load(sp,allow_pickle=False) as b:
        annual=a['planet'].astype('float32')
        seasonal=b['features'].astype('float32')
        target=a['probability'].astype('float32')
        mask=a['valid'].astype(bool)&b['valid'].astype(bool)
    assert annual.shape==(4,64,64) and seasonal.shape==(11,64,64)
    assert mask.sum()>100,(rec.tile_id,mask.sum())
    return np.concatenate((annual,seasonal),axis=0),target,mask

def train_normalizer133(indices):
    # Pixel samples restricted to the 3 training forests; no Yoko leakage.
    samples=[]
    for idx in indices:
        x,y,m=load133(idx)
        small=m[::4,::4]
        if small.any():samples.append(x[:,::4,::4][:,small])
    a=np.concatenate(samples,axis=1)
    med=np.median(a,axis=1).astype('float32')
    scale=np.maximum((np.percentile(a,90,axis=1)-np.percentile(a,10,axis=1))/2,
                     0.01).astype('float32')
    assert med.shape==(15,) and np.isfinite(med).all() and (scale>0).all()
    return med,scale

class Tile133(Dataset):
    def __init__(self,indices,median,scale,augment=False):
        self.ids=np.array(indices,dtype=int)
        self.mean=median[:,None,None]
        self.scale=scale[:,None,None]
        self.augment=augment
    def __len__(self):return len(self.ids)
    def __getitem__(self,k):
        x,y,m=load133(self.ids[k])
        x=np.clip((x-self.mean)/self.scale,-3,3)
        x[:,~m]=0
        if self.augment:
            if random.random()<.5:
                x=x[:,:,::-1];y=y[:,::-1];m=m[:,::-1]
            if random.random()<.5:
                x=x[:,::-1,:];y=y[::-1,:];m=m[::-1,:]
            turns=random.randrange(4)
            x=np.rot90(x,turns,axes=(1,2))
            y=np.rot90(y,turns)
            m=np.rot90(m,turns)
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(y[None])),
                torch.from_numpy(np.ascontiguousarray(m[None].astype('float32'))))

class DoubleConv13(nn.Module):
    def __init__(self,a,b):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(True),
                               nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(True))
    def forward(self,x):return self.net(x)

class SeasonalUNet13(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1=DoubleConv13(15,24);self.e2=DoubleConv13(24,48)
        self.e3=DoubleConv13(48,96);self.pool=nn.MaxPool2d(2)
        self.bottleneck=DoubleConv13(96,128)
        self.u3=nn.ConvTranspose2d(128,96,2,stride=2)
        self.d3=DoubleConv13(192,96)
        self.u2=nn.ConvTranspose2d(96,48,2,stride=2)
        self.d2=DoubleConv13(96,48)
        self.u1=nn.ConvTranspose2d(48,24,2,stride=2)
        self.d1=DoubleConv13(48,24)
        self.out=nn.Sequential(nn.Conv2d(24,1,1),nn.Sigmoid())
    def forward(self,x):
        a=self.e1(x);b=self.e2(self.pool(a));c=self.e3(self.pool(b))
        bott=self.bottleneck(self.pool(c))
        z=self.d3(torch.cat((self.u3(bott),c),dim=1))
        z=self.d2(torch.cat((self.u2(z),b),dim=1))
        z=self.d1(torch.cat((self.u1(z),a),dim=1))
        return self.out(z)


def evaluate133(model,loader):
    model.eval()
    preds=[]
    with torch.inference_mode():
        for x,y,m in loader:
            preds.append(model(x.to(DEVICE133)).cpu().numpy()[:,0])
    return np.concatenate(preds,axis=0).astype('float16')

predfile133=OUT133/'Yoko_predictions.npz'
metricfile133=OUT133/'Yoko_tile_metrics.csv'
weightfile133=OUT133/'Yoko_model.pt'
if predfile133.exists() and metricfile133.exists() and weightfile133.exists():
    print('13D.3 cached Yoko seasonal model; no retraining.',flush=True)
    with np.load(predfile133,allow_pickle=False) as z:
        assert np.array_equal(z['tile_index'],TEST133.astype('int32'))
        predictions133=z['prediction'].astype('float32')
else:
    random.seed(SEED133);np.random.seed(SEED133);torch.manual_seed(SEED133)
    torch.cuda.manual_seed_all(SEED133)
    median133,scale133=train_normalizer133(TRA133)
    train_site=M13.site.iloc[TRA133].to_numpy()
    n_site={site:(train_site==site).sum() for site in set(train_site)}
    weights=np.array([1/n_site[site] for site in train_site],dtype='float64')
    sampler=WeightedRandomSampler(torch.from_numpy(weights),num_samples=len(TRA133),replacement=True)
    trloader=DataLoader(Tile133(TRA133,median133,scale133,True),
                        batch_size=BATCH133,sampler=sampler,num_workers=0)
    teloader=DataLoader(Tile133(TEST133,median133,scale133,False),
                        batch_size=BATCH133,shuffle=False,num_workers=0)
    model133=SeasonalUNet13().to(DEVICE133)
    opt=torch.optim.AdamW(model133.parameters(),lr=LR133,weight_decay=0.001)
    log=[]
    for epoch in range(EPOCHS133):
        model133.train()
        numerator=denominator=0.
        for x,y,m in trloader:
            x=x.to(DEVICE133);y=y.to(DEVICE133);m=m.to(DEVICE133)
            opt.zero_grad(set_to_none=True)
            pred=model133(x)
            loss=((pred-y)**2*m).sum()/m.sum().clamp_min(1)
            loss.backward();opt.step()
            numerator+=float(((pred.detach()-y)**2*m).sum().item())
            denominator+=float(m.sum().item())
        mse=numerator/denominator
        log.append(dict(epoch=epoch+1,train_masked_MSE=mse))
        print(f'13D.3 LOFO Yoko epoch {epoch+1:02d}/{EPOCHS133} masked MSE={mse:.6f}',flush=True)
    predictions133=evaluate133(model133,teloader)
    tmp=predfile133.with_suffix('.npz.partial')
    with open(tmp,'wb') as f:
        np.savez_compressed(f,tile_index=TEST133.astype('int32'),
                            prediction=predictions133)
    os.replace(tmp,predfile133)
    pd.DataFrame(log).to_csv(OUT133/'training_history.csv',index=False)
    tmp=weightfile133.with_suffix('.pt.partial')
    # Tensors only: compatible with PyTorch weights_only=True.
    torch.save(dict(state=model133.state_dict(),
                    planet_season_median=torch.from_numpy(median133),
                    planet_season_scale=torch.from_numpy(scale133),
                    config=CONF133),tmp)
    os.replace(tmp,weightfile133)
assert predictions133.shape==(35,64,64)

# Weighted across truly valid pixels, not averaging tiles equally.
stats=[];ys=[];ps=[]
for j,k in enumerate(TEST133):
    x,y,m=load133(k)
    pred=predictions133[j].astype('float32')
    delta=pred[m]-y[m]
    ys.append(y[m]);ps.append(pred[m])
    stats.append(dict(site='Yoko',tile_id=M13.iloc[k].tile_id,n_valid=int(m.sum()),
        MAE_pp=100*np.abs(delta).mean(),bias_pp=100*delta.mean(),
        RMSE_pp=100*np.sqrt(np.mean(delta**2)),
        teacher_mean_pct=100*y[m].mean(),prediction_mean_pct=100*pred[m].mean()))
pd.DataFrame(stats).to_csv(metricfile133,index=False)
truth=np.concatenate(ys);prediction=np.concatenate(ps)
err=prediction-truth
r2=1-np.square(err).sum()/np.square(truth-truth.mean()).sum()
result133=pd.DataFrame([dict(model='Planet annual+seasonal',test_site='Yoko',
     n_tiles=35,n_valid_pixels=len(truth),
     MAE_pp=100*np.abs(err).mean(),RMSE_pp=100*np.sqrt(np.square(err).mean()),
     bias_pp=100*err.mean(),R2_pixel=r2,
     pearson=float(np.corrcoef(truth,prediction)[0,1]))])
result133.to_csv(OUT133/'Yoko_pixel_summary.csv',index=False)
display(result133.round(3))
ANNUAL133=ROOT133/'13C_dense_unet/816bb788b43ec113/LEAVE_FOREST_OUT_Yoko_predictions.npz'
if ANNUAL133.exists():
    with np.load(ANNUAL133,allow_pickle=False) as z:
        baseline_index=z['tile_index'].astype(int)
        baseline_pred=z['prediction'].astype('float32')
    assert np.array_equal(baseline_index,TEST133)
    annual_error=np.concatenate([(baseline_pred[j][load133(k)[2]]-load133(k)[1][load133(k)[2]])
                                  for j,k in enumerate(TEST133)])
    print('Annual-only Yoko MAE (same targets):',round(100*np.abs(annual_error).mean(),3),
          'pp | seasonal:',round(float(result133.MAE_pp.iloc[0]),3),'pp')
else:
    print('Annual-only LOFO Yoko checkpoint not found. Run 13C.2 Yoko before declaring seasonal improvement.')
print('Seasonal Yoko run saved to:',OUT133)


### 13D.4 · Matched annual vs seasonal PlanetScope — Yoko LOFO comparison

**Run after 13C.2 completes the `LEAVE_FOREST_OUT/Yoko` annual-only model.** The 13D.3 seasonal predictions are already saved. This CPU-only block evaluates the **same 35 Yoko tiles**, same valid UAV pixels, and common 10/30/50 m grid cells. It prints MAE, RMSE, bias, R² and Pearson for each model. The aggregation deliberately omits cross-tile edge cells and never treats missing teacher data as Musanga absence. The **4.77 m metric is the primary dense prediction metric**, while 30/50 m assessment tests scale-dependent usefulness.

**Interpretation gate:** compare paired metrics first; then decide whether monthly information materially improves Yoko transfer. A single forest is not sufficient to claim general transferability, and a new embedding/S2 experiment should still be validated with the same forest splits.

In [ ]:
# 13D.4 — Compare SAVED annual-only and seasonal Planet U-Nets on identical Yoko pixels.
# Requires 13C.2 Yoko checkpoint and completed 13D.3; CPU only.
from pathlib import Path
import numpy as np, pandas as pd, rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from IPython.display import display

root1344=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/06_spatial_planet_embeddings/13_dense_planet_unet')
tiles1344=root1344/'13B_tiles/31631cfae43b39ab'
manifest1344=pd.read_parquet(root1344/'13C_dense_unet/accepted_tile_manifest.parquet').reset_index(drop=True)
# Discover completed 13C.2 annual-only Yoko outputs across config-hash caches.
candidates1344=sorted((root1344/'13C_dense_unet').glob(
    '*/LEAVE_FOREST_OUT_Yoko_predictions.npz'))
season1344=root1344/'13D3_seasonal_unet_Yoko/ecafd2222e4edda9/Yoko_predictions.npz'
if not candidates1344:
    print('No annual-only Yoko prediction file found in any 13C configuration cache.')
    for folder in sorted((root1344/'13C_dense_unet').glob('*')):
        if folder.is_dir():
            print('Existing cache:',folder.name,
                  'Yoko metrics:',(folder/'LEAVE_FOREST_OUT_Yoko_tile_metrics.parquet').exists(),
                  'Yoko predictions:',(folder/'LEAVE_FOREST_OUT_Yoko_predictions.npz').exists())
    raise FileNotFoundError('13C.2 has not saved LOFO Yoko predictions here. Check its final SAVED/CACHED line and CHECKPOINTS directory; do not rerun unless needed.')
assert len(candidates1344)==1,('Multiple Yoko annual candidates; select verified cache explicitly',candidates1344)
annual1344=candidates1344[0]
print('Found annual Yoko prediction:',annual1344)
assert season1344.exists(),season1344
with np.load(annual1344,allow_pickle=False) as z:
    aidx1344=z['tile_index'].astype(int);apred1344=z['prediction'].astype('float32')
with np.load(season1344,allow_pickle=False) as z:
    sidx1344=z['tile_index'].astype(int);spred1344=z['prediction'].astype('float32')
assert np.array_equal(aidx1344,sidx1344) and len(aidx1344)==35
assert apred1344.shape==spred1344.shape==(35,64,64)
assert (manifest1344.iloc[aidx1344].site=='Yoko').all()

def aggregate1344(reference,modelpred,valid,rec,meters):
    if meters==4.77:
        return reference[valid],modelpred[valid]
    w,s,e,n=[float(getattr(rec,k)) for k in ('west','south','east','north')]
    left=np.floor(w/meters)*meters
    top=np.ceil(n/meters)*meters
    nx=int(np.ceil((e-left)/meters))
    ny=int(np.ceil((top-s)/meters))
    source_transform=from_origin(w,n,(e-w)/64,(n-s)/64)
    dest_transform=from_origin(left,top,meters,meters)
    m=valid.astype('float32')
    # To avoid GDAL nodata assumptions, use weighted source layers,
    # then divide by the explicitly warped observation-valid fraction.
    source=np.stack((reference*m,modelpred*m,m),axis=0).astype('float32')
    output=np.zeros((3,ny,nx),dtype='float32')
    reproject(source,output,src_transform=source_transform,dst_transform=dest_transform,
              src_crs='EPSG:32635',dst_crs='EPSG:32635',
              resampling=Resampling.average,src_nodata=None,dst_nodata=None)
    xlo=left+np.arange(nx)*meters
    xhi=xlo+meters
    yhi=top-np.arange(ny)*meters
    ylo=yhi-meters
    fx=np.maximum(0,np.minimum(xhi,e)-np.maximum(xlo,w))/meters
    fy=np.maximum(0,np.minimum(yhi,n)-np.maximum(ylo,s))/meters
    complete=fy[:,None]*fx[None,:]>=.95
    ok=complete&(output[2]>=.75)
    return output[0][ok]/output[2][ok].clip(min=1e-6),output[1][ok]/output[2][ok].clip(min=1e-6)

scales1344=(4.77,10.,30.,50.)
pairs1344={(model,scale):[[],[]] for model in ('Annual','Annual + seasonal') for scale in scales1344}
for j,idx in enumerate(aidx1344):
    rec=manifest1344.iloc[int(idx)]
    file=tiles1344/rec.site/Path(rec.mosaic).stem/(rec.tile_id+'.npz')
    with np.load(file,allow_pickle=False) as z:
        ref=z['probability'].astype('float32')
        valid=z['valid'].astype(bool)
    for name,pred in [('Annual',apred1344[j]),('Annual + seasonal',spred1344[j])]:
        for scale in scales1344:
            yy,pp=aggregate1344(ref,pred,valid,rec,scale)
            assert np.isfinite(yy).all() and np.isfinite(pp).all()
            pairs1344[(name,scale)][0].append(yy)
            pairs1344[(name,scale)][1].append(pp)

table1344=[]
for (model,scale),(ys,ps) in pairs1344.items():
    y=np.concatenate(ys);pred=np.concatenate(ps)
    err=pred-y
    ss=np.square(y-y.mean()).sum()
    table1344.append(dict(model=model,scale_m=scale,n_cells=len(y),
        MAE_pp=100*np.abs(err).mean(),RMSE_pp=100*np.sqrt(np.square(err).mean()),
        bias_pp=100*err.mean(),R2=1-np.square(err).sum()/ss if ss>0 else np.nan,
        pearson=float(np.corrcoef(y,pred)[0,1]) if np.std(pred)>1e-8 else np.nan,
        teacher_mean_pct=100*y.mean(),pred_mean_pct=100*pred.mean()))
RES1344=pd.DataFrame(table1344).sort_values(['scale_m','model'])
display(RES1344.round(3))
print('SEASONAL minus ANNUAL: negative Delta MAE means seasonal improves:')
comp1344=RES1344.pivot(index='scale_m',columns='model',values='MAE_pp')
delta1344=comp1344['Annual + seasonal']-comp1344['Annual']
display(pd.DataFrame(dict(scale_m=delta1344.index,delta_MAE_pp=delta1344.values)).round(3))
save1344=root1344/'13D3_seasonal_unet_Yoko/ecafd2222e4edda9'
RES1344.to_csv(save1344/'Yoko_annual_vs_seasonal_matched_scale.csv',index=False)
print('Matched 4.77–50m comparison saved:',save1344)


### 13E.1 · Sentinel-2 seasonal inputs for PlanetScope spatial U-Net — alignment and QA

The next controlled experiment is **annual PlanetScope spatial U-Net (4 native bands)** + **Sentinel-2 quarterly spectroscopy** (4 quarters × 10 feature bands, plus 4 clear-observation count bands). The existing three-year quarterly S2 composites are cached in NB04 and already used for the 50 m HGB comparison; no new GEE task or re-extraction is needed.

**First stage (this block):** validate source raster dimensions/band descriptions and extract a **small 5-tile pilot**, one tile per mosaic, using GDAL's read-only warped view to sample S2 onto the exact same 64×64 / 4.77 m Planet tile geometry. Reproject *continuous reflectances bilinearly* and *count/QA bands with nearest neighbour*. Store per-tile **44 S2 channels** as a separate float16 NPZ with valid-observation masks; do not modify cached Planet/UAV data. A 44-channel upsampled tensor is a convenient grid representation, **not** evidence of 4.77 m spectral resolution.

**Interpretation/QA gate:** inspect per-site completeness, quarter observation counts and source band descriptions before training. The quarterly composites are based on **site-specific historical year windows**, not necessarily the same UAV acquisition date; we will report that caveat and ensure the held-out Yoko forest is never used for normalization.

Once the pilot passes, set `MAX_NEW_TILES13E1=None` to complete all 422 tiles (restartable), then proceed to 13E.2 fusion training with identical Yoko LOFO splits.


In [ ]:
# 13E.1 — Cached S2 quarterly (44 bands) on exact native Planet tile grid.
# 5-tile QA pilot; after inspection switch MAX_NEW_TILES13E1=None.
from pathlib import Path
import os,json,hashlib,math
import numpy as np,pandas as pd,rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.windows import Window
from IPython.display import display

BASE13E=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1')
ROOT13E=BASE13E/'06_spatial_planet_embeddings/13_dense_planet_unet'
TILE13E=ROOT13E/'13B_tiles/31631cfae43b39ab'
MAN13E=pd.read_csv(TILE13E/'tile_manifest_progress.csv')
MAN13E=MAN13E[MAN13E.accepted.astype(str).str.lower().eq('true')].copy()
assert len(MAN13E)==422
S2DIR13E=BASE13E/'04_satellite_predictors/input_layers/S2_QUARTERLY_3YR'
ANNDIR13E=BASE13E/'05_sat_predictor_exploration/input_layers/Musanga_V31_NB05_UAV_ROI_Annual'
YRS13E={'Baego':(2021,2023),'MOS':(2022,2024),
        'Yangambi':(2022,2024),'Yoko':(2018,2020)}
PYEARS13E={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2021}
MAX_NEW_TILES13E1=5  # pilot; None completes all 422
COUNT_BANDS13E=[11,22,33,44]
SOURCES13E={}
source_metadata13e=[]
for site,(first,last) in YRS13E.items():
    src=S2DIR13E/f'{site}_S2_QUARTERLY_3YR_{first}_{last}_10m.tif'
    ann=ANNDIR13E/f'NB05_UAV_ROI_{site}_{PYEARS13E[site]}_Annual_19band.tif'
    assert src.is_file(),('Missing S2 quarterly raster',site,str(src))
    assert ann.is_file(),('Missing Planet grid reference',site,str(ann))
    with rasterio.open(src) as ds,rasterio.open(ann) as ref:
        assert ds.count==44,(site,'Expected 44 S2 quarterly bands',ds.count)
        assert ref.count==19,(site,'Expected 19 annual Planet bands',ref.count)
        assert ref.crs==ds.crs,('CRS mismatch',site,str(ds.crs),str(ref.crs))
        assert 9.5<=abs(ds.res[0])<=10.5
        source_metadata13e.append(dict(site=site,years=f'{first}-{last}',
            source=str(src),crs=str(ds.crs),shape=str((ds.height,ds.width)),
            s2_resolution=str(ds.res),planet_resolution=str(ref.res),
            s2_nodata=ds.nodata,descriptions=str(ds.descriptions),
            s2_size_MB=round(src.stat().st_size/1048576,1)))
    SOURCES13E[site]=(src,ann)
display(pd.DataFrame(source_metadata13e)[['site','years','s2_resolution',
    'planet_resolution','s2_nodata','s2_size_MB','descriptions']])

CONFIG13E=dict(version='S2_44bands_on_13B_planet_grid_v1',
    original_tile_run='31631cfae43b39ab',
    source={site:{'s2':str(s),'planet_grid':str(a)}
            for site,(s,a) in SOURCES13E.items()},
    counts=COUNT_BANDS13E,interpolation='bilinear_40_spectral_nearest_4_counts',
    target_grid='native_Planet_4_77m_64x64',
    valid_s2='at_least_one_nonzero_quarter_count')
KEY13E=hashlib.sha256(json.dumps(CONFIG13E,sort_keys=True).encode()).hexdigest()[:16]
OUT13E=ROOT13E/'13E1_s2_quarterly_tiles'/KEY13E
OUT13E.mkdir(parents=True,exist_ok=True)
configfile13e=OUT13E/'config.json'
if configfile13e.exists():
    assert json.loads(configfile13e.read_text())==CONFIG13E
else:
    configfile13e.write_text(json.dumps(CONFIG13E,indent=2))
print('13E.1 cache:',OUT13E,flush=True)

# Each forest's first tile in each mosaic, but prefer well-covered tiles.
PILOT13E=(MAN13E.sort_values('joint_fraction',ascending=False)
          .groupby(['site','mosaic'],sort=True,group_keys=False).head(1))
SELECT13E=PILOT13E if MAX_NEW_TILES13E1 is not None else MAN13E
n_saved13e=0
for site,group in SELECT13E.groupby('site',sort=True):
    s2file,planetfile=SOURCES13E[site]
    with rasterio.open(s2file) as src,rasterio.open(planetfile) as ref:
        with WarpedVRT(src,crs=ref.crs,transform=ref.transform,
                       width=ref.width,height=ref.height,
                       resampling=Resampling.bilinear) as interp:
            with WarpedVRT(src,crs=ref.crs,transform=ref.transform,
                           width=ref.width,height=ref.height,
                           resampling=Resampling.nearest) as nearest:
                for tile in group.itertuples(index=False):
                    relative=Path(tile.site)/Path(tile.mosaic).stem/(tile.tile_id+'.npz')
                    dest=OUT13E/relative
                    dest.parent.mkdir(parents=True,exist_ok=True)
                    if dest.exists():
                        with np.load(dest,allow_pickle=False) as z:
                            assert z['features'].shape==(44,64,64)
                            assert z['valid'].shape==(64,64)
                        continue
                    if MAX_NEW_TILES13E1 is not None and n_saved13e>=MAX_NEW_TILES13E1:break
                    win=Window(int(tile.col0),int(tile.row0),64,64)
                    continuous=[i for i in range(1,45) if i not in COUNT_BANDS13E]
                    xx=interp.read(continuous,window=win,out_dtype='float32')
                    counts=nearest.read(COUNT_BANDS13E,window=win,out_dtype='float32')
                    full=np.zeros((44,64,64),dtype='float32')
                    full[np.asarray(continuous)-1]=xx
                    full[np.asarray(COUNT_BANDS13E)-1]=counts
                    src_good=(np.isfinite(full).all(axis=0)&
                              (counts>0).any(axis=0)&
                              (counts>=0).all(axis=0))
                    with np.load(TILE13E/relative,allow_pickle=False) as teacher:
                        teacher_good=teacher['valid'].astype(bool)
                    good=src_good&teacher_good
                    finite=np.isfinite(full).all(axis=0)
                    full[:,~finite]=0
                    stats=dict(site=site,mosaic=tile.mosaic,tile_id=tile.tile_id,
                        n_teacher_valid=int(teacher_good.sum()),
                        n_s2valid=int(good.sum()),
                        s2valid_over_teacher=float(good.sum()/max(1,teacher_good.sum())),
                        quarterly_clear_median=[float(np.median(counts[q][teacher_good]))
                                                for q in range(4)])
                    partial=dest.with_suffix('.npz.partial')
                    with open(partial,'wb') as output:
                        np.savez_compressed(output,features=full.astype('float16'),
                            valid=good.astype('uint8'),metadata=json.dumps(stats))
                    os.replace(partial,dest)
                    n_saved13e+=1
                    print('13E1 SAVED',tile.tile_id,
                          'coverage',round(stats['s2valid_over_teacher'],3),
                          'quarter counts',stats['quarterly_clear_median'],flush=True)
    if MAX_NEW_TILES13E1 is not None and n_saved13e>=MAX_NEW_TILES13E1:break

rows13e=[]
for tile in MAN13E.itertuples(index=False):
    relative=Path(tile.site)/Path(tile.mosaic).stem/(tile.tile_id+'.npz')
    dest=OUT13E/relative
    record=dict(site=tile.site,mosaic=tile.mosaic,tile_id=tile.tile_id,saved=dest.exists())
    if dest.exists():
        with np.load(dest,allow_pickle=False) as z:
            record.update(json.loads(str(z['metadata'].item())))
    rows13e.append(record)
PROGRESS13E=pd.DataFrame(rows13e)
PROGRESS13E.to_csv(OUT13E/'tile_progress.csv',index=False)
print('13E.1 S2 tiles saved:',int(PROGRESS13E.saved.sum()),'/',len(PROGRESS13E))
display(PROGRESS13E[PROGRESS13E.saved].groupby('site').agg(
    tiles=('tile_id','size'),S2_coverage=('s2valid_over_teacher','mean')).round(3))
print('13E.1 QA file:',OUT13E/'tile_progress.csv')



### 13E.2 · PlanetScope spatial + quarterly Sentinel-2 fusion U-Net — strict Yoko holdout

All **422 S2 tiles** passed 13E.1 coverage checks. This restartable experiment trains on Baego, MOS and Yangambi (387 tiles) and tests on Yoko (35 tiles). **PlanetScope annual B/G/R/N (4.77 m)** feeds the spatial branch; **44 Sentinel-2 quarterly bands (10 m native)** feed a separate small 1×1-convolution feature branch. Its learned 16-channel representation is combined with Planet before the same 24/48/96-channel U-Net encoder-decoder. This preserves their distinction as separate input streams, although the S2 data are sampled to the Planet pixel lattice; **no extra native S2 spatial detail is implied**.

Use existing 13B teacher/Planet tiles and 13E.1 S2 caches only; normalize each branch from **training forests only**, maintain the exact original 13C LOFO Yoko split, 30 epochs, masked MSE and forest-balanced sampler. Save a distinct checkpoint and prediction cache. Report held-out Yoko native-resolution metrics and the **paired annual-only model** on common valid pixels. The 10/30/50 m exact geographic aggregation can be calculated from the two cached predictions in a following CPU-only analysis.

Run **13C.1 first only if `M13` and `split13c` are not defined**. No previous models need retraining. GPU required for a new fusion fit; cached predictions load without retraining.

**Caution:** the S2 quarterly record is site-specific (Yoko 2018–2020 vs 2021 Planet, other forests later). Apparent benefits could reflect acquisition-window confounding; forest-level generalization must ultimately be checked beyond Yoko.


In [ ]:
# 13E.2 — Two-branch Planet annual + S2 quarterly fusion, Yoko LOFO.
import os,json,hashlib,random,time
from pathlib import Path
import numpy as np,pandas as pd,torch
from torch import nn
from torch.utils.data import Dataset,DataLoader,WeightedRandomSampler
from IPython.display import display
assert 'M13' in globals() and 'split13c' in globals(), 'First run 13C.1 (split definitions).'
ROOT132E=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/06_spatial_planet_embeddings/13_dense_planet_unet')
PLAN132E=ROOT132E/'13B_tiles/31631cfae43b39ab'
S2_132E=ROOT132E/'13E1_s2_quarterly_tiles/a7d502802feb93ea'
status132e=pd.read_csv(S2_132E/'tile_progress.csv')
assert len(status132e)==422 and status132e.saved.astype(str).str.lower().eq('true').all()
tr132e,te132e=split13c('LEAVE_FOREST_OUT','Yoko')
assert len(tr132e)==387 and len(te132e)==35
assert (M13.iloc[te132e].site=='Yoko').all()
SEED132E=20261025+sum(map(ord,'Yoko'))+100
EPOCHS132E=30
BATCH132E=16
LR132E=0.0008
CONFIG132E=dict(version='13E2_planet4_S2quarterly44_fusion_v1',
    target_tiles=PLAN132E.name,s2_cache=S2_132E.name,
    holdout='Yoko',train_tiles=387,test_tiles=35,
    architecture='Planet4_plus_S2_44_to_16_separate_branch_UNet24_48_96',
    epochs=EPOCHS132E,batch=BATCH132E,lr=LR132E,seed=SEED132E,
    loss='valid_masked_MSE',sampling='forest_balanced_replacement',
    normalization='per_input_band_train_forests_only')
key132e=hashlib.sha256(json.dumps(CONFIG132E,sort_keys=True).encode()).hexdigest()[:16]
OUT132E=ROOT132E/'13E2_planet_s2_fusion_Yoko'/key132e
OUT132E.mkdir(parents=True,exist_ok=True)
(OUT132E/'config.json').write_text(json.dumps(CONFIG132E,indent=2))
print('13E.2 output:',OUT132E,flush=True)

def load132e(index):
    rec=M13.iloc[int(index)]
    rel=Path(rec.site)/Path(rec.mosaic).stem/(rec.tile_id+'.npz')
    with np.load(PLAN132E/rel,allow_pickle=False) as a:
        p=a['planet'].astype('float32')
        y=a['probability'].astype('float32')
        am=a['valid'].astype(bool)
    with np.load(S2_132E/rel,allow_pickle=False) as z:
        s=z['features'].astype('float32')
        sm=z['valid'].astype(bool)
    assert p.shape==(4,64,64) and s.shape==(44,64,64)
    mask=am&sm&np.isfinite(p).all(axis=0)&np.isfinite(s).all(axis=0)
    assert mask.sum()>100,(rec.tile_id,int(mask.sum()))
    return p,s,y,mask

def normalizer132e(train):
    pieces_p=[];pieces_s=[]
    for ix in train:
        p,s,y,mask=load132e(ix)
        valid=mask[::4,::4]
        if valid.any():
            pieces_p.append(p[:,::4,::4][:,valid])
            pieces_s.append(s[:,::4,::4][:,valid])
    out=[]
    for chunks,nb in [(pieces_p,4),(pieces_s,44)]:
        z=np.concatenate(chunks,axis=1)
        median=np.median(z,axis=1).astype('float32')
        scale=np.maximum((np.percentile(z,90,axis=1)-np.percentile(z,10,axis=1))/2,0.01).astype('float32')
        assert median.shape==(nb,) and np.isfinite(median).all() and np.isfinite(scale).all()
        out.extend([median,scale])
    return out

class FusionTiles132E(Dataset):
    def __init__(self,indices,norm,augment=False):
        self.ids=np.asarray(indices,dtype=int)
        self.pm,self.ps,self.sm,self.ss=[v[:,None,None] for v in norm]
        self.augment=augment
    def __len__(self):return len(self.ids)
    def __getitem__(self,i):
        p,s,y,m=load132e(self.ids[i])
        p=np.clip((p-self.pm)/self.ps,-3,3)
        s=np.clip((s-self.sm)/self.ss,-3,3)
        p[:,~m]=0;s[:,~m]=0
        if self.augment:
            if random.random()<0.5:
                p=p[:,:,::-1];s=s[:,:,::-1];y=y[:,::-1];m=m[:,::-1]
            if random.random()<0.5:
                p=p[:,::-1,:];s=s[:,::-1,:];y=y[::-1,:];m=m[::-1,:]
            turns=random.randrange(4)
            p=np.rot90(p,turns,axes=(1,2))
            s=np.rot90(s,turns,axes=(1,2))
            y=np.rot90(y,turns);m=np.rot90(m,turns)
        return tuple(torch.from_numpy(np.ascontiguousarray(v)) for v in (
            p,s,y[None],m[None].astype('float32')))

class Conv132E(nn.Module):
    def __init__(self,a,b):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(a,b,3,padding=1),
            nn.BatchNorm2d(b),nn.ReLU(inplace=True),
            nn.Conv2d(b,b,3,padding=1),
            nn.BatchNorm2d(b),nn.ReLU(inplace=True))
    def forward(self,x):return self.net(x)

class PlanetS2UNet132E(nn.Module):
    def __init__(self):
        super().__init__()
        self.s2branch=nn.Sequential(nn.Conv2d(44,16,1),
                                    nn.BatchNorm2d(16),nn.ReLU(inplace=True))
        self.e1=Conv132E(20,24);self.e2=Conv132E(24,48)
        self.e3=Conv132E(48,96);self.pool=nn.MaxPool2d(2)
        self.bottleneck=Conv132E(96,128)
        self.u3=nn.ConvTranspose2d(128,96,2,stride=2)
        self.d3=Conv132E(192,96)
        self.u2=nn.ConvTranspose2d(96,48,2,stride=2)
        self.d2=Conv132E(96,48)
        self.u1=nn.ConvTranspose2d(48,24,2,stride=2)
        self.d1=Conv132E(48,24)
        self.out=nn.Sequential(nn.Conv2d(24,1,1),nn.Sigmoid())
    def forward(self,p,s):
        x=torch.cat((p,self.s2branch(s)),dim=1)
        a=self.e1(x);b=self.e2(self.pool(a));c=self.e3(self.pool(b))
        z=self.d3(torch.cat((self.u3(self.bottleneck(self.pool(c))),c),dim=1))
        z=self.d2(torch.cat((self.u2(z),b),dim=1))
        z=self.d1(torch.cat((self.u1(z),a),dim=1))
        return self.out(z)

predfile132e=OUT132E/'Yoko_predictions.npz'
weights132e=OUT132E/'Yoko_model.pt'
if predfile132e.exists() and weights132e.exists():
    with np.load(predfile132e,allow_pickle=False) as f:
        assert np.array_equal(f['tile_index'],te132e.astype('int32'))
        pred132e=f['prediction'].astype('float32')
    print('CACHED 13E.2 predictions; skipping GPU fitting.',flush=True)
else:
    assert torch.cuda.is_available(),'Use a GPU Colab runtime for first training.'
    random.seed(SEED132E);np.random.seed(SEED132E);torch.manual_seed(SEED132E)
    torch.cuda.manual_seed_all(SEED132E)
    torch.set_num_threads(2)
    norm132e=normalizer132e(tr132e)
    forest=M13.site.iloc[tr132e].to_numpy()
    freq={s:int((forest==s).sum()) for s in set(forest)}
    weights=np.array([1/freq[s] for s in forest],dtype='float64')
    sampler=WeightedRandomSampler(torch.from_numpy(weights),num_samples=len(tr132e),replacement=True)
    training=DataLoader(FusionTiles132E(tr132e,norm132e,True),batch_size=BATCH132E,
                        sampler=sampler,num_workers=0)
    testing=DataLoader(FusionTiles132E(te132e,norm132e,False),batch_size=BATCH132E,
                       shuffle=False,num_workers=0)
    model=PlanetS2UNet132E().cuda()
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR132E,weight_decay=1e-3)
    history=[]
    for epoch in range(EPOCHS132E):
        model.train();numer=denom=0.
        for p,s,y,m in training:
            p=p.cuda(non_blocking=True);s=s.cuda(non_blocking=True)
            y=y.cuda(non_blocking=True);m=m.cuda(non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            yp=model(p,s)
            loss=(((yp-y)**2)*m).sum()/m.sum().clamp_min(1.)
            loss.backward();optimizer.step()
            numer+=float(loss.item())*float(m.sum().item())
            denom+=float(m.sum().item())
        current=numer/denom
        history.append(dict(epoch=epoch+1,train_masked_MSE=current))
        print(f'13E.2 LOFO Yoko epoch {epoch+1:02d}/{EPOCHS132E} masked MSE={current:.6f}',flush=True)
    model.eval();out=[]
    with torch.inference_mode():
        for p,s,y,m in testing:
            out.append(model(p.cuda(),s.cuda()).cpu().numpy()[:,0].astype('float16'))
    pred132e=np.concatenate(out,axis=0).astype('float32')
    temp=predfile132e.with_suffix('.npz.partial')
    with open(temp,'wb') as f:
        np.savez_compressed(f,tile_index=te132e.astype('int32'),
                            prediction=pred132e.astype('float16'))
    os.replace(temp,predfile132e)
    tmp=weights132e.with_suffix('.pt.partial')
    torch.save(dict(state=model.state_dict(),
        planet_median=torch.from_numpy(norm132e[0]),
        planet_scale=torch.from_numpy(norm132e[1]),
        s2_median=torch.from_numpy(norm132e[2]),
        s2_scale=torch.from_numpy(norm132e[3]),configuration=CONFIG132E),tmp)
    os.replace(tmp,weights132e)
    pd.DataFrame(history).to_csv(OUT132E/'training_history.csv',index=False)
assert pred132e.shape==(35,64,64)

# Compare only identically valid UAV/S2 pixels in the held-out Yoko tiles.
annual_candidates=sorted((ROOT132E/'13C_dense_unet').glob(
    '*/LEAVE_FOREST_OUT_Yoko_predictions.npz'))
assert len(annual_candidates)==1,('Expected one annual baseline cache',annual_candidates)
with np.load(annual_candidates[0],allow_pickle=False) as b:
    assert np.array_equal(b['tile_index'],te132e.astype('int32'))
    annualpred=b['prediction'].astype('float32')
paired={'Annual Planet':([],[]),'Planet + quarterly S2':([],[])}
for j,idx in enumerate(te132e):
    p,s,target,valid=load132e(idx)
    for name,pp in [('Annual Planet',annualpred[j]),
                    ('Planet + quarterly S2',pred132e[j])]:
        paired[name][0].append(target[valid])
        paired[name][1].append(pp[valid])
report=[]
for name,(ys,ps) in paired.items():
    y=np.concatenate(ys);pred=np.concatenate(ps);err=pred-y
    report.append(dict(model=name,scale_m=4.77,n_pixels=len(y),
        MAE_pp=100*np.abs(err).mean(),RMSE_pp=100*np.sqrt(np.mean(err**2)),
        bias_pp=100*err.mean(),R2=1-np.sum(err**2)/np.sum((y-y.mean())**2),
        pearson=np.corrcoef(y,pred)[0,1]))
report132e=pd.DataFrame(report)
report132e.to_csv(OUT132E/'Yoko_paired_native_metrics.csv',index=False)
display(report132e.round(3))
print('13E.2 predictions and checkpoint:',OUT132E)



### 13E.3 · Does quarterly S2 fusion improve canopy fractions at 10, 30 or 50 m?

CPU-only **paired geographical scale assessment** using the already-saved annual PlanetScope and 13E.2 Planet+S2 Yoko predictions. Exact same 35 held-out tiles and valid UAV reference pixels, with the same aggregation method as 13D.4. Reports both **MAE/R²** and **bias** at 4.77, 10, 30 and 50 m. Saves the matched table within the 13E.2 output folder. No retraining or earlier blocks needed; just mount Google Drive.


In [ ]:
# 13E.3 — Compare SAVED annual-only and fusion Planet U-Nets on identical Yoko pixels.
# Requires 13C.2 Yoko checkpoint and completed 13D.3; CPU only.
from pathlib import Path
import numpy as np, pandas as pd, rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from IPython.display import display

root133E=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/06_spatial_planet_embeddings/13_dense_planet_unet')
tiles133E=root133E/'13B_tiles/31631cfae43b39ab'
manifest133E=pd.read_parquet(root133E/'13C_dense_unet/accepted_tile_manifest.parquet').reset_index(drop=True)
# Discover completed 13C.2 annual-only Yoko outputs across config-hash caches.
candidates133E=sorted((root133E/'13C_dense_unet').glob(
    '*/LEAVE_FOREST_OUT_Yoko_predictions.npz'))
fusion133E=root133E/'13E2_planet_s2_fusion_Yoko/fe22da7d40c05546/Yoko_predictions.npz'
if not candidates133E:
    print('No annual-only Yoko prediction file found in any 13C configuration cache.')
    for folder in sorted((root133E/'13C_dense_unet').glob('*')):
        if folder.is_dir():
            print('Existing cache:',folder.name,
                  'Yoko metrics:',(folder/'LEAVE_FOREST_OUT_Yoko_tile_metrics.parquet').exists(),
                  'Yoko predictions:',(folder/'LEAVE_FOREST_OUT_Yoko_predictions.npz').exists())
    raise FileNotFoundError('13C.2 has not saved LOFO Yoko predictions here. Check its final SAVED/CACHED line and CHECKPOINTS directory; do not rerun unless needed.')
assert len(candidates133E)==1,('Multiple Yoko annual candidates; select verified cache explicitly',candidates133E)
annual133E=candidates133E[0]
print('Found annual Yoko prediction:',annual133E)
assert fusion133E.exists(),fusion133E
with np.load(annual133E,allow_pickle=False) as z:
    aidx133E=z['tile_index'].astype(int);apred133E=z['prediction'].astype('float32')
with np.load(fusion133E,allow_pickle=False) as z:
    sidx133E=z['tile_index'].astype(int);spred133E=z['prediction'].astype('float32')
assert np.array_equal(aidx133E,sidx133E) and len(aidx133E)==35
assert apred133E.shape==spred133E.shape==(35,64,64)
assert (manifest133E.iloc[aidx133E].site=='Yoko').all()

def aggregate133E(reference,modelpred,valid,rec,meters):
    if meters==4.77:
        return reference[valid],modelpred[valid]
    w,s,e,n=[float(getattr(rec,k)) for k in ('west','south','east','north')]
    left=np.floor(w/meters)*meters
    top=np.ceil(n/meters)*meters
    nx=int(np.ceil((e-left)/meters))
    ny=int(np.ceil((top-s)/meters))
    source_transform=from_origin(w,n,(e-w)/64,(n-s)/64)
    dest_transform=from_origin(left,top,meters,meters)
    m=valid.astype('float32')
    # To avoid GDAL nodata assumptions, use weighted source layers,
    # then divide by the explicitly warped observation-valid fraction.
    source=np.stack((reference*m,modelpred*m,m),axis=0).astype('float32')
    output=np.zeros((3,ny,nx),dtype='float32')
    reproject(source,output,src_transform=source_transform,dst_transform=dest_transform,
              src_crs='EPSG:32635',dst_crs='EPSG:32635',
              resampling=Resampling.average,src_nodata=None,dst_nodata=None)
    xlo=left+np.arange(nx)*meters
    xhi=xlo+meters
    yhi=top-np.arange(ny)*meters
    ylo=yhi-meters
    fx=np.maximum(0,np.minimum(xhi,e)-np.maximum(xlo,w))/meters
    fy=np.maximum(0,np.minimum(yhi,n)-np.maximum(ylo,s))/meters
    complete=fy[:,None]*fx[None,:]>=.95
    ok=complete&(output[2]>=.75)
    return output[0][ok]/output[2][ok].clip(min=1e-6),output[1][ok]/output[2][ok].clip(min=1e-6)

scales133E=(4.77,10.,30.,50.)
pairs133E={(model,scale):[[],[]] for model in ('Annual Planet','Planet + quarterly S2') for scale in scales133E}
for j,idx in enumerate(aidx133E):
    rec=manifest133E.iloc[int(idx)]
    file=tiles133E/rec.site/Path(rec.mosaic).stem/(rec.tile_id+'.npz')
    with np.load(file,allow_pickle=False) as z:
        ref=z['probability'].astype('float32')
        valid=z['valid'].astype(bool)
    for name,pred in [('Annual Planet',apred133E[j]),('Planet + quarterly S2',spred133E[j])]:
        for scale in scales133E:
            yy,pp=aggregate133E(ref,pred,valid,rec,scale)
            assert np.isfinite(yy).all() and np.isfinite(pp).all()
            pairs133E[(name,scale)][0].append(yy)
            pairs133E[(name,scale)][1].append(pp)

table133E=[]
for (model,scale),(ys,ps) in pairs133E.items():
    y=np.concatenate(ys);pred=np.concatenate(ps)
    err=pred-y
    ss=np.square(y-y.mean()).sum()
    table133E.append(dict(model=model,scale_m=scale,n_cells=len(y),
        MAE_pp=100*np.abs(err).mean(),RMSE_pp=100*np.sqrt(np.square(err).mean()),
        bias_pp=100*err.mean(),R2=1-np.square(err).sum()/ss if ss>0 else np.nan,
        pearson=float(np.corrcoef(y,pred)[0,1]) if np.std(pred)>1e-8 else np.nan,
        teacher_mean_pct=100*y.mean(),pred_mean_pct=100*pred.mean()))
RES133E=pd.DataFrame(table133E).sort_values(['scale_m','model'])
display(RES133E.round(3))
print('SEASONAL minus ANNUAL: negative Delta MAE means S2 fusion improves:')
comp133E=RES133E.pivot(index='scale_m',columns='model',values='MAE_pp')
delta133E=comp133E['Planet + quarterly S2']-comp133E['Annual Planet']
display(pd.DataFrame(dict(scale_m=delta133E.index,delta_MAE_pp=delta133E.values)).round(3))
save133E=root133E/'13E2_planet_s2_fusion_Yoko/fe22da7d40c05546'
RES133E.to_csv(save133E/'Yoko_annual_vs_S2_fusion_matched_scale.csv',index=False)
print('Matched 4.77–50m comparison saved:',save133E)



### 13F.1 · Convergence audit — single sources versus combinations

**Purpose:** decide which satellite information genuinely adds predictive skill, without launching another model grid. This CPU-only block audits previously saved, strict `LEAVE_FOREST_OUT` **50 m** predictions from Sections 11B (HGB), 12B (Planet spatial CNN) and 12C (hybrid). It joins their predictions by **identical site/mosaic/row/col**, requires identical UAV teacher targets, and reports **per-forest MAE, R², bias, pooled and equal-forest metrics**, including the MOS abundance ≥40% subset. If a run is absent or yields a different reference cohort, it will be flagged rather than silently compared.

**Interpretation:** the existing HGB `B0` combines S2 + embeddings; `B1` adds **Planet spectral summaries**, while `PlanetCNN` isolates **Planet native spatial/spectral** and `HybridCNN` fuses Planet spatial patches with S2 + embeddings. They are **not** independent S2-only and embedding-only baselines. Report their absence clearly. Also list the completed dense 13C–13E Yoko predictions **separately**, because they use a different reference support and must not enter the matched HGB/CNN table without a spatial intersection.

**Next decision:** if the archived matched comparison is complete, add only the missing S2-only and embedding-only HGB baselines using the already cached 11A features and frozen 11B folds (no new downloads). Before any dense-model comparison, harmonize Yoko test cells explicitly; the 13C dense 50 m aggregates differ from the 11B 50 m lattice and currently cannot be compared by summary statistics alone.

No GPU, no earlier cells required: mount Google Drive and run **13F.1**.


In [ ]:
# 13F.1 — Strict LOFO matched 50m archive audit; no fitting or re-extraction.
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

BASE13F=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1')
B11F=BASE13F/'06_spatial_planet_embeddings/11_spatial_ablation_50m'
N12F=BASE13F/'06_spatial_planet_embeddings/12_native_planet_cnn'
ROOT13F=BASE13F/'06_spatial_planet_embeddings/13_dense_planet_unet'
SAVE13F=BASE13F/'06_spatial_planet_embeddings/13F_source_ablation'
SAVE13F.mkdir(parents=True,exist_ok=True)
FORESTS13F=['Baego','MOS','Yangambi','Yoko']
KEYS13F=['site','mosaic','row','col']
# One unique completed run is required for each architecture; do not guess.
families13f={
    'S2+EMB (HGB B0)':(B11F/'fold_predictions','B0_S2_EMB'),
    'S2+EMB+Planet spectral (HGB B1)':(B11F/'fold_predictions','B1_B0_PlanetSpectral'),
    'Planet spatial CNN':(N12F/'cnn_fold_predictions','PlanetCNN'),
    'Planet spatial+S2+EMB hybrid':(N12F/'hybrid_fold_predictions','HybridCNN')
}
def candidates13f(root,hold,ending):
    return sorted(root.glob(f'*/LEAVE_FOREST_OUT_{hold}_{ending}.parquet'))
missing13f=[]
records13f=[]
selection13f={}
for model,(folder,suffix) in families13f.items():
    perforest=[]
    for forest in FORESTS13F:
        choices=candidates13f(folder,forest,suffix)
        if len(choices)!=1:
            missing13f.append(dict(model=model,forest=forest,found=len(choices),
                                   candidates='; '.join(str(p) for p in choices[:3])))
            continue
        data=pd.read_parquet(choices[0])
        expected=KEYS13F+['observed','predicted']
        assert all(c in data.columns for c in expected),(choices[0],data.columns.tolist())
        data=data[expected].copy()
        assert len(data)>0 and not data.duplicated(KEYS13F).any(),choices[0]
        assert (data.site==forest).all(),(forest,choices[0])
        assert np.isfinite(data[['observed','predicted']].to_numpy('float64')).all(),choices[0]
        data['model']=model
        perforest.append(data)
        selection13f[(model,forest)]=str(choices[0])
    if len(perforest)==4:
        records13f.append(pd.concat(perforest,ignore_index=True))
if missing13f:
    print('ARCHIVED MODEL COVERAGE — missing or ambiguous per-forest files:')
    display(pd.DataFrame(missing13f))
assert records13f,'No complete model family found; inspect saved run directories.'
big13f=pd.concat(records13f,ignore_index=True)
modelnames13f=sorted(big13f.model.unique())
print('Complete archived model families:',modelnames13f)
# Require absolutely identical sites, mosaics and reference targets in all included models.
reference13f=big13f[big13f.model==modelnames13f[0]][KEYS13F+['observed']]
reference13f=reference13f.sort_values(KEYS13F).reset_index(drop=True)
paired13f=[]
for model in modelnames13f:
    subset=big13f[big13f.model==model].sort_values(KEYS13F).reset_index(drop=True)
    if len(subset)!=len(reference13f) or not subset[KEYS13F].equals(reference13f[KEYS13F]):
        raise AssertionError(f'13F cannot compare non-identical footprints for {model}')
    if not np.allclose(subset.observed.to_numpy(),reference13f.observed.to_numpy(),
                       rtol=0,atol=1e-6):
        raise AssertionError(f'13F teacher values differ for {model}')
    paired13f.append(subset)
MATCHED13F=pd.concat(paired13f,ignore_index=True)
print('Identical 50m test cells per model:',len(reference13f),'by forest:')
display(reference13f.groupby('site').size().rename('n_50m_cells').to_frame())

def eval13f(frame):
    y=frame.observed.to_numpy(float)
    p=frame.predicted.to_numpy(float)
    e=p-y
    ss=float(np.square(y-y.mean()).sum())
    high=y>=0.4
    return dict(n=len(y),MAE_pp=100*np.abs(e).mean(),
       RMSE_pp=100*np.sqrt(np.square(e).mean()),bias_pp=100*e.mean(),
       R2=(1-float(np.square(e).sum())/ss) if ss>0 else np.nan,
       pearson=np.corrcoef(y,p)[0,1] if np.std(p)>0 and np.std(y)>0 else np.nan,
       n_high=int(high.sum()),high_bias_pp=(100*e[high].mean()) if high.any() else np.nan,
       high_recall=(float((p[high]>=0.4).mean())) if high.any() else np.nan)
site_rows13f=[]
for (model,forest),group in MATCHED13F.groupby(['model','site']):
    site_rows13f.append(dict(model=model,site=forest,**eval13f(group)))
SITE13F=pd.DataFrame(site_rows13f).sort_values(['site','MAE_pp'])
display(SITE13F.round(3))
global_rows13f=[]
for model,group in MATCHED13F.groupby('model'):
    sites=SITE13F[SITE13F.model==model].set_index('site').loc[FORESTS13F]
    # Equal-forest R2 pools 1/n_site weighted squared residuals; not mean site R2.
    weights=1/group.site.map(group.site.value_counts()).to_numpy(float)
    weights/=weights.sum()
    y=group.observed.to_numpy(float);p=group.predicted.to_numpy(float)
    mean=np.dot(weights,y)
    denominator=np.dot(weights,(y-mean)**2)
    mos=group[group.site=='MOS']
    moss=eval13f(mos)
    global_rows13f.append(dict(model=model,
        equal_forest_MAE_pp=sites.MAE_pp.mean(),
        equal_forest_R2=1-np.dot(weights,(y-p)**2)/denominator,
        pooled_MAE_pp=100*np.abs(y-p).mean(),
        pooled_R2=eval13f(group)['R2'],
        MOS_MAE_pp=moss['MAE_pp'],
        MOS_high_n=moss['n_high'],
        MOS_high_bias_pp=moss['high_bias_pp'],
        MOS_high_recall=moss['high_recall']))
GLOBAL13F=pd.DataFrame(global_rows13f).sort_values('equal_forest_MAE_pp')
print('EXISTING MATCHED 50m FOREST-TRANSFER BENCHMARK:')
display(GLOBAL13F.round(3))
SITE13F.to_csv(SAVE13F/'13F1_per_forest_existing_models.csv',index=False)
GLOBAL13F.to_csv(SAVE13F/'13F1_equal_forest_existing_models.csv',index=False)
pd.DataFrame([dict(model=k[0],site=k[1],path=v) for k,v in selection13f.items()]
             ).to_csv(SAVE13F/'13F1_sources.csv',index=False)
# Independently audit the NEW dense Yoko model availability, not matched to older 50m cells.
dense13f={
    'Planet annual U-Net':ROOT13F/'13C_dense_unet/816bb788b43ec113/LEAVE_FOREST_OUT_Yoko_predictions.npz',
    'Planet seasonal U-Net':ROOT13F/'13D3_seasonal_unet_Yoko/ecafd2222e4edda9/Yoko_predictions.npz',
    'Planet + S2 U-Net':ROOT13F/'13E2_planet_s2_fusion_Yoko/fe22da7d40c05546/Yoko_predictions.npz'}
print('Separate dense-Yoko evaluation cache; DIFFERENT footprint from 11/12:')
display(pd.DataFrame([dict(model=k,exists=v.is_file(),path=str(v))
                      for k,v in dense13f.items()]))
print('MISSING ABALATIONS: S2 only; Embeddings only. Section 11B B0 is S2+embeddings, not S2 only.')
print('Note: this 13F.1 comparison is strictly within the archived 50m cohort.')
print('13F.1 audit saved to:',SAVE13F)



### 13F.2 · Final missing single-source ablations (S2-only, embeddings-only, embedding + context)

This block fills **only three missing baselines** with the **same HGB hyperparameters, exact LOFO forest holdouts, identical original 50 m reference cells, and 1,500 training cells maximum per forest** as 11B. All S2 and embedding predictors are already cached in the NB06 11A input table. No new Earth Engine extraction, UAV processing or CNN training.

1. **S2 only:** cached annual S2 + 3-year quarterly S2 (spectral + season).
2. **Embeddings only:** 64 annual embedding channels.
3. **Embeddings + embedding spatial context:** annual 64 channels plus 128 precomputed neighborhood summary features.

The existing 11B **S2+embeddings** and **S2+embeddings+Planet spectral** baselines are loaded, not retrained. Compare five models on identical rows across all four strict LOFO forests; report **equal-forest R²/MAE**, per-site performance, and MOS high-cover bias/recall. A check ensures the cached 11A feature table matches the archived 11B held-out coordinates and targets before fitting. Separate per-forest parquet files make runs restartable. **Run directly after mounting Google Drive; no earlier notebook cells or GPU required.**

> Scope: this isolates the value of S2 seasonal features and embedding spectral/spatial context within HGB at the archived 50 m training support. It does **not** yet compare the dense 13C Planet U-Net because its validation footprints differ.


In [ ]:
# 13F.2 — Three missing standalone satellite baselines on frozen 11B 50m LOFO cells.
from pathlib import Path
import os, json, hashlib, time
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from IPython.display import display
from threadpoolctl import threadpool_limits

B13F=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/06_spatial_planet_embeddings')
B11F2=B13F/'11_spatial_ablation_50m'
OUT13F2=B13F/'13F_source_ablation/13F2_single_sources_v1'
OUT13F2.mkdir(parents=True,exist_ok=True)
FORESTS13F2=['Baego','MOS','Yangambi','Yoko']
KEYS13F2=['site','mosaic','row','col']
MODELS13F2={
    'S2 only':('RAW_ANNUAL__','S2_QUARTERLY_3YR__'),
    'Embeddings only':('EMB_ANNUAL__',),
    'Embeddings + spatial context':('EMB_ANNUAL__','EMB_mean_','EMB_sd_')
}
REFDIR13F2=B11F2/'fold_predictions'
ref_files13f2={}
for forest in FORESTS13F2:
    matches=sorted(REFDIR13F2.glob(f'*/LEAVE_FOREST_OUT_{forest}_B0_S2_EMB.parquet'))
    assert len(matches)==1,('Missing/ambiguous frozen B0 split',forest,matches)
    ref_files13f2[forest]=matches[0]
assert len({p.parent for p in ref_files13f2.values()})==1, 'B0 forests in different fit runs'
REFERENCE_RUN13F2=next(iter(ref_files13f2.values())).parent
print('Frozen 11B baseline run:',REFERENCE_RUN13F2)

# Discover the exact original 11A input cache by checking held-out keys/targets.
caches13f2=sorted((B11F2/'inputs').glob('combined_50m_*.parquet'))
assert caches13f2, 'Missing cached 11A combined input'
ref13f2={site:pd.read_parquet(path,columns=KEYS13F2+['observed','predicted'])
         for site,path in ref_files13f2.items()}
for site,frame in ref13f2.items():
    assert (frame.site==site).all() and not frame.duplicated(KEYS13F2).any()
def matching_cache13f2(path):
    df=pd.read_parquet(path)
    if df.duplicated(KEYS13F2).any() or 'target_fraction' not in df.columns:
        return None
    # Mirror 11A's common-case input mask. This is crucial when original
    # cache contains rows excluded during the original 11A preparation.
    spatial=[f'PLANET_{q}_{b}' for q in ('mean','std') for b in ('B','G','R','N')]
    spatial += [f'PLANET_texture_{b}' for b in
                ('NIR_CV_r1','NIR_CV_r3','NIR_CV_r7','NIR_highpass')]
    spatial += [f'EMB_{q}_{j:02d}' for q in ('mean','sd') for j in range(1,65)]
    upstream=[col for col in df.columns
              if col.startswith(('RAW_ANNUAL__','S2_QUARTERLY_3YR__','EMB_ANNUAL__'))]
    required=upstream+spatial
    if not set(required).issubset(df.columns):
        return None
    clear=[f'S2_QUARTERLY_3YR__{j:02d}' for j in (11,22,33,44)]
    good=np.isfinite(df[required].to_numpy(dtype='float32')).all(axis=1)
    good &= (df[clear].to_numpy(dtype='float32')>=1).all(axis=1)
    good &= df.target_fraction.between(0,1).to_numpy()
    df=df.loc[good].sort_values(KEYS13F2,kind='stable').reset_index(drop=True)
    for site in FORESTS13F2:
        a=df.loc[df.site==site,KEYS13F2+['target_fraction']].reset_index(drop=True)
        b=ref13f2[site].sort_values(KEYS13F2).reset_index(drop=True)
        if len(a)!=len(b) or not a[KEYS13F2].equals(b[KEYS13F2]):
            return None
        if not np.allclose(a.target_fraction,b.observed,atol=1e-6,rtol=0):
            return None
    return df

eligible13f2=[]
for file in caches13f2:
    maybe=matching_cache13f2(file)
    if maybe is not None: eligible13f2.append((file,maybe))
assert len(eligible13f2)==1,(
    'Need exactly one matching 11A input cache; inspect candidate files',
    [str(f) for f in caches13f2],len(eligible13f2))
INPUTFILE13F2,D13F2=eligible13f2[0]
print('Verified exact input table:',INPUTFILE13F2,'rows',len(D13F2))
assert len(D13F2)==14454
sourcecols13f2={}
for model,prefixes in MODELS13F2.items():
    sourcecols13f2[model]=[k for k in D13F2.columns
                         if any(k.startswith(prefix) for prefix in prefixes)]
    assert len(sourcecols13f2[model])==(
        64 if model=='Embeddings only' else
        192 if model=='Embeddings + spatial context' else
        len([k for k in D13F2 if k.startswith(('RAW_ANNUAL__','S2_QUARTERLY_3YR__'))])
    ),(model,len(sourcecols13f2[model]))
print('Source predictor counts:',{k:len(v) for k,v in sourcecols13f2.items()})

PARAMS13F2=dict(learning_rate=.06,max_iter=180,max_leaf_nodes=20,
                min_samples_leaf=35,l2_regularization=2,
                early_stopping=False,random_state=20261024)
CONF13F2=dict(version='13F2_single_sources_v1',frozen_11B=str(REFERENCE_RUN13F2),
  source_file=str(INPUTFILE13F2),sources=sourcecols13f2,
  forest_order=FORESTS13F2,cap=1500,seed=20261024,
  params=PARAMS13F2)
CONFIGPATH13F2=OUT13F2/'config.json'
if CONFIGPATH13F2.exists():
    assert json.loads(CONFIGPATH13F2.read_text())==CONF13F2
else:
    CONFIGPATH13F2.write_text(json.dumps(CONF13F2,indent=2))
MAX_NEW_FITS13F2=None  # Set 1 for a quick pilot; None = all 12 HGB fits.
newfits13f2=0
with threadpool_limits(limits=1):
    for hold in FORESTS13F2:
        test=D13F2.site.eq(hold).to_numpy()
        train=~test
        # Reproduce 11B's forest-wise capped sampling and RNG lifecycle.
        rng=np.random.default_rng(20261024)
        ids=[]
        for forest in FORESTS13F2:
            eligible=np.where(train & D13F2.site.eq(forest).to_numpy())[0]
            if len(eligible)>1500:
                eligible=rng.choice(eligible,1500,replace=False)
            ids.extend(eligible.tolist())
        tr=np.asarray(sorted(ids),dtype=int)
        te=np.flatnonzero(test)
        holdframe=D13F2.iloc[te][KEYS13F2+['target_fraction']].reset_index(drop=True)
        expected=ref13f2[hold].sort_values(KEYS13F2).reset_index(drop=True)
        assert holdframe[KEYS13F2].equals(expected[KEYS13F2])
        assert np.allclose(holdframe.target_fraction,expected.observed,atol=1e-6,rtol=0)
        for model,cols in sourcecols13f2.items():
            dest=OUT13F2/f'LEAVE_FOREST_OUT_{hold}_{model.replace(" ","_")}.parquet'
            if dest.exists():
                saved=pd.read_parquet(dest)
                if (len(saved)==len(te) and
                    saved[KEYS13F2].equals(expected[KEYS13F2]) and
                    np.allclose(saved.observed,expected.observed,atol=1e-6,rtol=0)
                    and np.isfinite(saved.predicted).all()):
                    print('CACHED',hold,model,flush=True)
                    continue
            if MAX_NEW_FITS13F2 is not None and newfits13f2>=MAX_NEW_FITS13F2:
                print('Pilot cap reached; rerun this cell to resume.',flush=True)
                break
            started=time.monotonic()
            model_fit=HistGradientBoostingRegressor(**PARAMS13F2)
            model_fit.fit(D13F2.iloc[tr][cols].to_numpy('float32'),
                          D13F2.iloc[tr].target_fraction.to_numpy('float32'))
            predictions=np.clip(model_fit.predict(D13F2.iloc[te][cols].to_numpy('float32')),0,1)
            result=expected.copy()
            result['predicted']=predictions
            tmp=dest.with_suffix('.parquet.partial')
            result.to_parquet(tmp,index=False)
            os.replace(tmp,dest)
            newfits13f2+=1
            print('SAVED',hold,model,'elapsed_s',round(time.monotonic()-started,1),flush=True)
        if MAX_NEW_FITS13F2 is not None and newfits13f2>=MAX_NEW_FITS13F2:
            break

# Summarize ONLY complete four-forest models, including existing B0/B1.
all_models13f2={}
for model in MODELS13F2:
    stem=model.replace(' ','_')
    paths=[OUT13F2/f'LEAVE_FOREST_OUT_{f}_{stem}.parquet' for f in FORESTS13F2]
    if all(p.exists() for p in paths):
        all_models13f2[model]=dict(zip(FORESTS13F2,paths))
for label,suffix in [
    ('S2 + embeddings','B0_S2_EMB'),
    ('S2 + embeddings + Planet spectral','B1_B0_PlanetSpectral')]:
    all_models13f2[label]={
        site:REFERENCE_RUN13F2/f'LEAVE_FOREST_OUT_{site}_{suffix}.parquet'
        for site in FORESTS13F2}
rows13f2=[]
for model,files in all_models13f2.items():
    for site,file in files.items():
        d=pd.read_parquet(file)
        yy=d.observed.to_numpy(float);pp=d.predicted.to_numpy(float);ee=pp-yy
        high=yy>=.40
        denom=np.square(yy-yy.mean()).sum()
        rows13f2.append(dict(model=model,site=site,n=len(d),
            MAE_pp=100*np.abs(ee).mean(),RMSE_pp=100*np.sqrt(np.square(ee).mean()),
            R2=1-np.square(ee).sum()/denom if denom>0 else np.nan,
            bias_pp=100*ee.mean(),n_high=int(high.sum()),
            high_bias_pp=100*ee[high].mean() if high.any() else np.nan,
            high_recall=float((pp[high]>=.40).mean()) if high.any() else np.nan,
            SSE=float(np.square(ee).sum()),sum_y=float(yy.sum()),
            sum_y2=float(np.square(yy).sum())))
SITE13F2=pd.DataFrame(rows13f2)
summary13f2=[]
# Equal-forest R2 uses an equal-forest weighted pooled target variance.
total_y=sum(ref13f2[s].observed.mean() for s in FORESTS13F2)/4
total_variance=sum(
    (np.square(ref13f2[s].observed-total_y)).mean() for s in FORESTS13F2)/4
for model,group in SITE13F2.groupby('model'):
    if len(group)!=4:continue
    mse_equal=float(np.mean(group.RMSE_pp.to_numpy()**2))/10000
    mos=group[group.site=='MOS'].iloc[0]
    summary13f2.append(dict(model=model,
       equal_forest_MAE_pp=float(group.MAE_pp.mean()),
       equal_forest_R2=1-mse_equal/total_variance,
       MOS_MAE_pp=float(mos.MAE_pp),
       MOS_high_n=int(mos.n_high),
       MOS_high_bias_pp=float(mos.high_bias_pp),
       MOS_high_recall=float(mos.high_recall)))
SUMMARY13F2=pd.DataFrame(summary13f2).sort_values('equal_forest_MAE_pp')
display(SUMMARY13F2.round(3))
print('Detailed per-forest results:')
display(SITE13F2.drop(columns=['SSE','sum_y','sum_y2']).sort_values(['site','MAE_pp']).round(3))
SITE13F2.to_csv(OUT13F2/'13F2_per_forest.csv',index=False)
SUMMARY13F2.to_csv(OUT13F2/'13F2_equal_forest.csv',index=False)
print('13F.2 saved:',OUT13F2)



### 13F.3 · Final controlled three-component ablation (7 combinations × 4 forest holdouts)

**Fixed 50 m target on exactly the 14,454 matched 11B/13F.2 cells.** Inputs are (P) native 32×32 PlanetScope 4.77 m **spatial CNN patch**; (S) Sentinel-2 annual + 3-year quarterly **spectral/temporal bands** (59 tabular features); and (E) 64 annual satellite embeddings + **128 spatial neighborhood mean/SD features** (192 embedding/context features). The embedding branch is a learned MLP applied to precomputed spatial summaries, **not a CNN over raw embedding rasters**. All three are predictive representations; they are not mutually independent measurements.

Train one compact multi-branch CNN/MLP with fixed hyperparameters and exactly **seven component subsets: P, S, E, P+S, P+E, S+E, P+S+E**. P uses the *same architecture* in every combination; S and E also retain fixed encoders. Forest-balanced sample cap 1,500 per training forest, 30 epochs, train-only robust scaling, same held-out cells and a fixed run seed. All models are trained in this single architecture family for valid contrasts. This is separate from HGB and earlier 12C estimates.

**Start with `MAX_NEW_FITS13F3=1`**, which fits *one* model on *one* forest holdout and saves it. The first pair is **P-only / Yoko**, relatively fast. Inspect loss/held-out results before `None` (all remaining fits, restartable). Only complete seven-model / four-holdout sets are used for final global comparisons; incomplete runs print progress. No Earth Engine processing or previous notebook blocks needed, only mounted Drive and GPU.

Limitations: forest holdouts supply just four independent environments. Fine-scale Planet patch data are aligned to the same 50 m labels but have neighboring imagery; the embedding spatial context uses summary features rather than raw 2D embedding neighborhoods.


In [ ]:
# 13F.3 — Seven-source combinations on the EXACT 11B 50m sample support.
import os,json,hashlib,random,time
from pathlib import Path
import numpy as np,pandas as pd,torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from IPython.display import display

B133=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/06_spatial_planet_embeddings')
D133=pd.read_parquet(B133/'11_spatial_ablation_50m/inputs/combined_50m_8d5e4a9227917a86.parquet')
KEY133=['site','mosaic','row','col']
SITES133=['Baego','MOS','Yangambi','Yoko']
# Apply precisely the frozen 11A valid-cell selection, before opening patches.
spatial133=[f'PLANET_{q}_{b}' for q in ('mean','std') for b in ('B','G','R','N')]
spatial133 += [f'PLANET_texture_{b}' for b in
               ('NIR_CV_r1','NIR_CV_r3','NIR_CV_r7','NIR_highpass')]
spatial133 += [f'EMB_{q}_{j:02d}' for q in ('mean','sd') for j in range(1,65)]
sat133=[c for c in D133.columns if c.startswith(('RAW_ANNUAL__','S2_QUARTERLY_3YR__','EMB_ANNUAL__'))]
allcols133=sat133+spatial133
assert set(allcols133).issubset(D133.columns)
qacols133=[f'S2_QUARTERLY_3YR__{b:02d}' for b in (11,22,33,44)]
good133=np.isfinite(D133[allcols133].to_numpy('float32')).all(axis=1)
good133 &= (D133[qacols133].to_numpy('float32')>=1).all(axis=1)
good133 &= D133.target_fraction.between(0,1).to_numpy()
D133=D133.loc[good133].sort_values(KEY133,kind='stable').reset_index(drop=True)
assert len(D133)==14454 and D133.site.nunique()==4
REF133=B133/'11_spatial_ablation_50m/fold_predictions/23c75e9ccb6c536b'
for forest in SITES133:
    ref=pd.read_parquet(REF133/f'LEAVE_FOREST_OUT_{forest}_B0_S2_EMB.parquet',
                        columns=KEY133+['observed']).sort_values(KEY133).reset_index(drop=True)
    part=D133.loc[D133.site==forest,KEY133+['target_fraction']].reset_index(drop=True)
    assert part[KEY133].equals(ref[KEY133]) and np.allclose(
        part.target_fraction,ref.observed,atol=1e-6,rtol=0)
PARENT133=B133/'12_native_planet_cnn/native_patches'
runs133=sorted(PARENT133.glob('*/planet_native_32px_float16.npy'))
goodruns133=[]
for f in runs133:
    qa=f.parent/'patch_validity.parquet'
    if not qa.exists():continue
    metadata=pd.read_parquet(qa)
    if (len(metadata)==len(D133) and
        metadata[KEY133].reset_index(drop=True).equals(D133[KEY133]) and
        (metadata.planet_native_valid>=.80).all()):
        goodruns133.append(f)
assert len(goodruns133)==1,('Expected exactly one matching native patch cache',goodruns133)
PATCH133=np.load(goodruns133[0],mmap_mode='r')
assert PATCH133.shape==(14454,4,32,32)
COLS133={
    'S':[c for c in D133 if c.startswith(('RAW_ANNUAL__','S2_QUARTERLY_3YR__'))],
    'E':[c for c in D133 if c.startswith(('EMB_ANNUAL__','EMB_mean_','EMB_sd_'))]}
assert len(COLS133['S'])==59 and len(COLS133['E'])==192
TAB133={k:D133[v].to_numpy('float32') for k,v in COLS133.items()}
TARGET133=D133.target_fraction.to_numpy('float32')
COMBOS133=['P','S','E','PS','PE','SE','PSE']
EPOCHS133F=30
BATCH133F=96
LR133F=.001
SEED133F=20261026
MAX_NEW_FITS13F3=1  # Pilot one fold/model; set None to complete all 28.
DEVICE133F=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CONF133F=dict(version='13F3_three_component_ablation_v1',
  sample_support='11B_14454_50m',planet_patch_run=goodruns133[0].parent.name,
  input_file='combined_50m_8d5e4a9227917a86.parquet',
  branch_columns=COLS133,combos=COMBOS133,forests=SITES133,
  epochs=EPOCHS133F,batch=BATCH133F,lr=LR133F,seed=SEED133F,
  train_cap_per_forest=1500,loss='MSE',normalization='training_only',
  model='P_CNN_24_48_64_96__S_MLP_64_32__E_MLP_64_32__fusion_MLP_48')
HASH133F=hashlib.sha256(json.dumps(CONF133F,sort_keys=True).encode()).hexdigest()[:16]
OUT133F=B133/'13F_source_ablation/13F3_three_components'/HASH133F
OUT133F.mkdir(parents=True,exist_ok=True)
cp133f=OUT133F/'config.json'
if cp133f.exists():assert json.loads(cp133f.read_text())==CONF133F
else:cp133f.write_text(json.dumps(CONF133F,indent=2))
print('13F.3 cache:',OUT133F,'device:',DEVICE133F,
      'patches:',goodruns133[0],flush=True)

def split133f(hold):
    rng=np.random.default_rng(20261024)
    site=D133.site.to_numpy()
    tr=[]
    for forest in SITES133:
        if forest==hold:continue
        eligible=np.flatnonzero(site==forest)
        if len(eligible)>1500:eligible=rng.choice(eligible,1500,replace=False)
        tr.extend(eligible.tolist())
    return np.asarray(sorted(tr),dtype=int),np.flatnonzero(site==hold)

def normalization133f(train,components):
    stats={}
    if 'P' in components:
        a=np.asarray(PATCH133[train[::max(1,len(train)//1000)]],dtype='float32')
        band=[]
        for b in range(4):
            v=a[:,b].ravel();v=v[np.isfinite(v)&(v>0)]
            assert len(v)>1000
            q=np.percentile(v,[10,50,90])
            band.append((q[1],max((q[2]-q[0])/2,1e-3)))
        stats['P']=np.asarray(band,dtype='float32')
    for k in ('S','E'):
        if k in components:
            x=TAB133[k][train]
            q10,q50,q90=np.percentile(x,[10,50,90],axis=0)
            stats[k]=np.stack([q50,np.maximum((q90-q10)/2,1.)],axis=1).astype('float32')
    return stats

class AblaData133F(Dataset):
    def __init__(self,ids,components,norm,augment=False):
        self.ids=np.asarray(ids,dtype=int);self.parts=components
        self.norm=norm;self.augment=augment
    def __len__(self):return len(self.ids)
    def __getitem__(self,j):
        idx=int(self.ids[j]);data={}
        if 'P' in self.parts:
            a=np.asarray(PATCH133[idx],dtype='float32')
            ok=np.isfinite(a).all(axis=0)&(a>0).all(axis=0)
            p=np.clip((a-self.norm['P'][:,0,None,None])/
                      self.norm['P'][:,1,None,None],-3,3)
            p[:,~ok]=0
            if self.augment:
                if random.random()<.5:p=p[:,:,::-1]
                if random.random()<.5:p=p[:,::-1,:]
                p=np.rot90(p,random.randrange(4),axes=(1,2))
            data['P']=torch.from_numpy(np.ascontiguousarray(p))
        for key in ('S','E'):
            if key in self.parts:
                x=np.clip((TAB133[key][idx]-self.norm[key][:,0])/
                          self.norm[key][:,1],-5,5)
                data[key]=torch.from_numpy(np.ascontiguousarray(x))
        return data,torch.tensor(TARGET133[idx],dtype=torch.float32)

class PlanetNet133F(nn.Module):
    def __init__(self):
        super().__init__()
        def stack(a,b):
            return nn.Sequential(nn.Conv2d(a,b,3,padding=1),
                nn.BatchNorm2d(b),nn.ReLU(inplace=True),
                nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),
                nn.ReLU(inplace=True),nn.MaxPool2d(2))
        self.net=nn.Sequential(stack(4,24),stack(24,48),stack(48,64),
            stack(64,96),nn.AdaptiveAvgPool2d(1),nn.Flatten(),
            nn.Linear(96,32),nn.ReLU())
    def forward(self,x):return self.net(x)

class SourceModel133F(nn.Module):
    def __init__(self,parts):
        super().__init__()
        self.parts=parts
        if 'P' in parts:self.p=PlanetNet133F()
        if 'S' in parts:self.s=nn.Sequential(nn.Linear(59,64),nn.ReLU(),
            nn.Dropout(.1),nn.Linear(64,32),nn.ReLU())
        if 'E' in parts:self.e=nn.Sequential(nn.Linear(192,64),nn.ReLU(),
            nn.Dropout(.1),nn.Linear(64,32),nn.ReLU())
        self.head=nn.Sequential(nn.Linear(32*len(parts),48),nn.ReLU(),
            nn.Dropout(.15),nn.Linear(48,1),nn.Sigmoid())
    def forward(self,inputs):
        features=[]
        if 'P' in self.parts:features.append(self.p(inputs['P']))
        if 'S' in self.parts:features.append(self.s(inputs['S']))
        if 'E' in self.parts:features.append(self.e(inputs['E']))
        return self.head(torch.cat(features,dim=1)).flatten()

def fit133f(hold,combo):
    tr,te=split133f(hold)
    norm=normalization133f(tr,combo)
    seed=SEED133F+sum(map(ord,hold))+sum(map(ord,combo))
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    train_loader=DataLoader(AblaData133F(tr,combo,norm,True),
        batch_size=BATCH133F,shuffle=True,num_workers=0)
    test_loader=DataLoader(AblaData133F(te,combo,norm,False),
        batch_size=BATCH133F,shuffle=False,num_workers=0)
    net=SourceModel133F(combo).to(DEVICE133F)
    opt=torch.optim.AdamW(net.parameters(),lr=LR133F,weight_decay=.001)
    history=[]
    for epoch in range(EPOCHS133F):
        net.train();total=0.;count=0
        for batch,target in train_loader:
            inp={k:v.to(DEVICE133F) for k,v in batch.items()}
            y=target.to(DEVICE133F)
            opt.zero_grad(set_to_none=True)
            pred=net(inp);loss=((pred-y)**2).mean()
            loss.backward();opt.step()
            total+=float(loss.item())*len(y);count+=len(y)
        history.append(dict(epoch=epoch+1,train_MSE=total/count))
        print(f'13F.3 {hold}/{combo} {epoch+1:02d}/{EPOCHS133F} MSE={total/count:.5f}',
              flush=True)
    net.eval();preds=[]
    with torch.inference_mode():
        for inp,y in test_loader:
            preds.append(net({k:v.to(DEVICE133F) for k,v in inp.items()})
                         .cpu().numpy())
    return te,np.clip(np.concatenate(preds),0,1),history

new133f=0
for hold in ['Yoko','Baego','Yangambi','MOS']:
    for combo in COMBOS133:
        te=split133f(hold)[1]
        destination=OUT133F/f'LOFO_{hold}_{combo}.parquet'
        if destination.exists():
            old=pd.read_parquet(destination)
            expected=D133.iloc[te][KEY133].reset_index(drop=True)
            if (len(old)==len(te) and old[KEY133].equals(expected) and
                np.isfinite(old.predicted).all()):
                print('CACHED',hold,combo,flush=True);continue
        if MAX_NEW_FITS13F3 is not None and new133f>=MAX_NEW_FITS13F3:
            break
        assert DEVICE133F.type=='cuda','GPU required for new CNN fits.'
        tick=time.monotonic()
        idx,pred,history=fit133f(hold,combo)
        data=D133.iloc[idx][KEY133+['target_fraction']].reset_index(drop=True)
        data=data.rename(columns={'target_fraction':'observed'})
        data['predicted']=pred
        tmp=destination.with_suffix('.parquet.partial')
        data.to_parquet(tmp,index=False);os.replace(tmp,destination)
        pd.DataFrame(history).to_csv(
            OUT133F/f'LOFO_{hold}_{combo}_history.csv',index=False)
        new133f+=1
        print('SAVED',hold,combo,'seconds',round(time.monotonic()-tick,1),flush=True)
    if MAX_NEW_FITS13F3 is not None and new133f>=MAX_NEW_FITS13F3:break

existing133f=[(h,c) for h in SITES133 for c in COMBOS133
             if (OUT133F/f'LOFO_{h}_{c}.parquet').exists()]
print('13F.3 COMPLETE FITS:',len(existing133f),'/28')
if len(existing133f)==28:
    records=[];sites=[]
    for combo in COMBOS133:
        subparts=[]
        for hold in SITES133:
            data=pd.read_parquet(OUT133F/f'LOFO_{hold}_{combo}.parquet')
            y=data.observed.to_numpy(float);p=data.predicted.to_numpy(float)
            error=p-y;hi=y>=.4
            sites.append(dict(model=combo,site=hold,n=len(y),
              MAE_pp=100*np.abs(error).mean(),bias_pp=100*error.mean(),
              high_n=int(hi.sum()),
              high_bias_pp=100*error[hi].mean() if hi.any() else np.nan,
              high_recall=(p[hi]>=.4).mean() if hi.any() else np.nan))
            data['site']=hold;subparts.append(data)
        whole=pd.concat(subparts,ignore_index=True)
        weights=1/whole.site.map(whole.site.value_counts()).to_numpy(float)
        weights/=weights.sum()
        yy=whole.observed.to_numpy(float);pp=whole.predicted.to_numpy(float)
        avg=np.dot(weights,yy);den=np.dot(weights,(yy-avg)**2)
        records.append(dict(model=combo,equal_forest_R2=1-np.dot(weights,(yy-pp)**2)/den,
                            equal_forest_MAE_pp=np.mean(
                              [x['MAE_pp'] for x in sites if x['model']==combo])))
    GLOBAL133F=pd.DataFrame(records).sort_values('equal_forest_MAE_pp')
    SITE133F=pd.DataFrame(sites)
    display(GLOBAL133F.round(3))
    display(SITE133F.sort_values(['site','MAE_pp']).round(3))
    GLOBAL133F.to_csv(OUT133F/'13F3_global.csv',index=False)
    SITE133F.to_csv(OUT133F/'13F3_sites.csv',index=False)
else:
    print('Pilot/partial run. Set MAX_NEW_FITS13F3=None to finish remaining fits.')
print('Results directory:',OUT133F)



### 13G · Decisive PlanetScope dense U-Net cross-forest validation (no fusion)

**Finish the existing annual-only 13C U-Net on all four held-out forests**, without changing the original architecture, seed, training data, tile normalization, or 30-epoch training schedule. 13C.2 now runs only **Yoko, Baego, Yangambi, MOS**; previously completed Yoko and any other fold are reused. Earlier buffered within-forest folds are not re-trained.

After 13C.2 finishes, run **13G.1 (CPU only)**: reaggregate the saved 4.77 m dense predictions to 10/30/**50 m** exactly as in 13D.4, per forest; show equal-forest summary, R², MAE, and high Musanga cover bias/recall. The 50 m support is *tile-local* and may not coincide with the 11B/13F reference cells. Thus **do not directly rank these 13G numbers against the 13F 50 m models**; a strictly matched 50 m spatial comparison follows if dense U-Net transfer is promising.

**In Colab:** Mount Drive; run 13C.1, 13C.2, then 13G.1. No 13B re-extraction or other retraining.


In [ ]:
# 13G.1 — Four-forest Planet ANNUAL dense U-Net, cached-only evaluation.
# Prerequisites: Drive mounted; 13C.1 (manifest), 13C.2 (LOFO predictions).
from pathlib import Path
import numpy as np,pandas as pd,rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from IPython.display import display
assert 'M13' in globals() and 'DIR13C' in globals() and 'split13c' in globals()
base13g=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/06_spatial_planet_embeddings/13_dense_planet_unet')
runs13g=sorted((base13g/'13C_dense_unet').glob('*/LEAVE_FOREST_OUT_Yoko_predictions.npz'))
assert len(runs13g)==1,('Ambiguous original annual dense model run',runs13g)
run13g=runs13g[0].parent
forests13g=('Baego','MOS','Yangambi','Yoko')
scales13g=(4.77,10.,30.,50.)
print('Using original 13C dense annual U-Net:',run13g)
for site in forests13g:
    f=run13g/f'LEAVE_FOREST_OUT_{site}_predictions.npz'
    assert f.exists(),('Missing holdout; run 13C.2 (missing LOFO only)',f)
def aggregate13g(reference,modelpred,valid,rec,meters):
    if meters==4.77:
        return reference[valid],modelpred[valid]
    w,s,e,n=[float(getattr(rec,k)) for k in ('west','south','east','north')]
    left=np.floor(w/meters)*meters
    top=np.ceil(n/meters)*meters
    nx=int(np.ceil((e-left)/meters))
    ny=int(np.ceil((top-s)/meters))
    source_transform=from_origin(w,n,(e-w)/64,(n-s)/64)
    dest_transform=from_origin(left,top,meters,meters)
    m=valid.astype('float32')
    # To avoid GDAL nodata assumptions, use weighted source layers,
    # then divide by the explicitly warped observation-valid fraction.
    source=np.stack((reference*m,modelpred*m,m),axis=0).astype('float32')
    output=np.zeros((3,ny,nx),dtype='float32')
    reproject(source,output,src_transform=source_transform,dst_transform=dest_transform,
              src_crs='EPSG:32635',dst_crs='EPSG:32635',
              resampling=Resampling.average,src_nodata=None,dst_nodata=None)
    xlo=left+np.arange(nx)*meters
    xhi=xlo+meters
    yhi=top-np.arange(ny)*meters
    ylo=yhi-meters
    fx=np.maximum(0,np.minimum(xhi,e)-np.maximum(xlo,w))/meters
    fy=np.maximum(0,np.minimum(yhi,n)-np.maximum(ylo,s))/meters
    complete=fy[:,None]*fx[None,:]>=.95
    ok=complete&(output[2]>=.75)
    return output[0][ok]/output[2][ok].clip(min=1e-6),output[1][ok]/output[2][ok].clip(min=1e-6)


rows13g=[]
for site in forests13g:
    filepath=run13g/f'LEAVE_FOREST_OUT_{site}_predictions.npz'
    with np.load(filepath,allow_pickle=False) as z:
        ids=z['tile_index'].astype(int)
        prediction=z['prediction'].astype('float32')
    _,expected=split13c('LEAVE_FOREST_OUT',site)
    assert np.array_equal(ids,expected),(site,'holdout tile mismatch')
    assert len(ids)>0 and prediction.shape==(len(ids),64,64)
    accum={scale:[[],[]] for scale in scales13g}
    for j,k in enumerate(ids):
        rec=M13.iloc[int(k)]
        assert rec.site==site
        with np.load(rec.tile_path,allow_pickle=False) as z:
            y=z['probability'].astype('float32')
            mask=z['valid'].astype(bool)
        for scale in scales13g:
            yy,pp=aggregate13g(y,prediction[j],mask,rec,scale)
            assert np.isfinite(yy).all() and np.isfinite(pp).all()
            accum[scale][0].append(yy)
            accum[scale][1].append(pp)
    for scale,(ya,pa) in accum.items():
        yy=np.concatenate(ya);pp=np.concatenate(pa)
        err=pp-yy;ss=np.square(yy-yy.mean()).sum();high=yy>=.4
        rows13g.append(dict(site=site,scale_m=scale,n_cells=len(yy),
          target_mean_pct=100*yy.mean(),prediction_mean_pct=100*pp.mean(),
          MAE_pp=100*np.abs(err).mean(),
          RMSE_pp=100*np.sqrt(np.square(err).mean()),
          bias_pp=100*err.mean(),R2=1-np.square(err).sum()/ss if ss>0 else np.nan,
          pearson=np.corrcoef(yy,pp)[0,1] if np.std(pp)>0 and np.std(yy)>0 else np.nan,
          n_high=int(high.sum()),
          high_bias_pp=100*err[high].mean() if high.any() else np.nan,
          high_recall=float((pp[high]>=.4).mean()) if high.any() else np.nan,
          target_variance=float(np.mean(np.square(yy-yy.mean()))),
          mse=float(np.mean(err**2))))
SITE13G=pd.DataFrame(rows13g).sort_values(['scale_m','site'])
print('Four held-out forests, annual dense Planet U-Net:')
display(SITE13G.drop(columns=['target_variance','mse']).round(3))
global13g=[]
for scale,sub in SITE13G.groupby('scale_m'):
    # Equal weight to four forests, pooled R2 (rather than average forest R2).
    mu=float(sub.target_mean_pct.mean()/100)
    variance=float(np.mean(sub.target_variance.to_numpy()+
            np.square(sub.target_mean_pct.to_numpy()/100-mu)))
    global13g.append(dict(scale_m=scale,
       equal_forest_MAE_pp=float(sub.MAE_pp.mean()),
       equal_forest_R2=1-float(sub.mse.mean())/variance,
       mean_forest_R2=float(sub.R2.mean()),
       MOS_MAE_pp=float(sub.loc[sub.site=='MOS','MAE_pp'].iloc[0]),
       MOS_high_n=int(sub.loc[sub.site=='MOS','n_high'].iloc[0]),
       MOS_high_bias_pp=float(sub.loc[sub.site=='MOS','high_bias_pp'].iloc[0]),
       MOS_high_recall=float(sub.loc[sub.site=='MOS','high_recall'].iloc[0])))
GLOBAL13G=pd.DataFrame(global13g).sort_values('scale_m')
print('EQUAL-FOREST summary (tile-local spatial support):')
display(GLOBAL13G.round(3))
SITE13G.to_csv(run13g/'13G1_annual_dense_LOFO_sites.csv',index=False)
GLOBAL13G.to_csv(run13g/'13G1_annual_dense_LOFO_equal_forest.csv',index=False)
print('13G.1 saved:',run13g)
print('CAUTION: 13G tile-local 50m cells differ from 13F 50m grid; do not directly rank them.')

